# MEDICAL RAG ASSISSTANT POJECT

By: Maria Krasnova

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 --no-cache-dir -q

In [ ]:
!pip install huggingface_hub tiktoken pymupdf langchain langchain-community chromadb sentence-transformers langchain-huggingface -U langchain-chroma -q

In [3]:
#Libraries for processing dataframes,text
import re
import os
import json,os
import tiktoken
import pandas as pd
import langchain
print(langchain.__version__)

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_chroma import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

1.3.11


/tmp/ipykernel_1435/477544845.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [4]:
!pip install langchain-text-splitters -q

In [5]:
# Formatting model's output to inteded format
from IPython.display import display, Markdown

## Question Answering using LLM

#### Downloading and Loading the model

In [6]:
# Using secret token from hugging face to import model
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_Token')

In [7]:
#Loading Llama 3.1 8B Instruct GGUF model from Hugging Face
model_path = hf_hub_download(
    repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
    filename="Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf"
)

llm = Llama(
    model_path=model_path,
    n_threads= 2,  # uses 2 CPU cores for any non-GPU-offloaded work
    n_batch= 256,  # processes up to 256 tokens per batch during prompt eval
    n_gpu_layers= -1,  # offloads all 32 layers to the T4 GPU
    n_ctx= 4096,  # supports up to 4096 tokens of combined prompt + response
)

Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf:   0%|          | 0.00/5.73G [00:00<?, ?B/s]

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from /root/.cache/huggingface/hub/models--bartowski--Meta-Llama-3.1-8B-Instruct-GGUF/snapshots/bf5b95e96dac0462e2a09145ec66cae9a3f12067/Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename 

#### Response

In [8]:
# Defining Model response parameters

def response(query, params=None):
    # Set standard defaults including all requested parameters
    config = {
        "max_tokens": 1536, # generates up to 1536 tokens before cutting off
        "temperature": 0, # always picks the single highest-probability token (deterministic)
        "top_p": 0.95, # Consider tokens that together make up the top 95% of probability mass
        "top_k": 50, # Maximum number of most-likely next tokens to consider
        "repeat_penalty": 1.1, # mildly discourages repeating the same phrases/words
        "seed": 42, # fixes the random state so reruns are reproducible
        "stop": ["<|eot_id|>", "Q:", "\n\n\n"] # stops generation early if any of these appear
    }

    # Update config if specific params are passed during the call
    if params:
        config.update(params)

    # Calling the model instance with the unpacked configuration
    model_output = llm.create_chat_completion(
        messages=[{"role": "user", "content": query}],
        **config
    )

    return model_output['choices'][0]['message']['content'].strip()

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [9]:
# Base model response
display(Markdown(response("What is the protocol for managing sepsis in a critical care unit?")))

ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     652.29 ms /    50 tokens (   13.05 ms per token,    76.65 tokens per second)
llama_perf_context_print:        eval time =   18015.85 ms /   631 runs   (   28.55 ms per token,    35.02 tokens per second)
llama_perf_context_print:       total time =   20490.36 ms /   681 tokens
llama_perf_context_print:    graphs reused =        628


Managing sepsis in a critical care unit involves a multi-step approach that includes early recognition, prompt treatment, and ongoing monitoring. Here's an overview of the protocol:

**Early Recognition (Sepsis 1)**

1. **Identify high-risk patients**: Patients with suspected infection, such as pneumonia, urinary tract infection, or abdominal infection.
2. **Monitor vital signs**: Temperature, heart rate, respiratory rate, blood pressure, and oxygen saturation.
3. **Use the Sepsis-3 definition**: Sepsis is defined as a Sequential Organ Failure Assessment (SOFA) score ≥ 2 points.

**Prompt Treatment (Sepsis 1)**

1. **Administer broad-spectrum antibiotics**: Within 1 hour of suspected sepsis, administer antibiotics that cover likely pathogens.
2. **Fluid resuscitation**: Administer crystalloid fluids (e.g., normal saline or lactated Ringer's solution) to maintain a mean arterial pressure ≥ 65 mmHg and urine output ≥ 0.5 mL/kg/hour.
3. **Oxygen therapy**: Provide supplemental oxygen as needed to maintain SpO2 ≥ 94%.
4. **Monitor for organ dysfunction**: Assess liver, kidney, cardiovascular, respiratory, coagulation, and neurological function.

**Sepsis Bundle (Sepsis 2)**

1. **Administer antibiotics within 3 hours**: Continue broad-spectrum antibiotics.
2. **Fluid resuscitation**: Continue fluid administration to maintain a mean arterial pressure ≥ 65 mmHg and urine output ≥ 0.5 mL/kg/hour.
3. **Oxygen therapy**: Continue supplemental oxygen as needed.
4. **Lactate measurement**: Measure lactate levels within 6 hours of suspected sepsis.

**Sepsis-Induced Organ Dysfunction (SOFA) Score**

1. **Assess organ function**: Evaluate liver, kidney, cardiovascular, respiratory, coagulation, and neurological function using the SOFA score.
2. **Score ≥ 2 points**: Indicates sepsis-induced organ dysfunction.

**Ongoing Monitoring**

1. **Vital signs**: Continuously monitor temperature, heart rate, respiratory rate, blood pressure, and oxygen saturation.
2. **Lactate levels**: Monitor lactate levels every 6 hours.
3. **Organ function**: Assess liver, kidney, cardiovascular, respiratory, coagulation, and neurological function regularly.

**Escalation of Care**

1. **Consult intensivist or infectious disease specialist**: If sepsis is suspected or confirmed.
2. **Transfer to ICU**: If patient requires close monitoring and intervention.

**De-escalation of Care**

1. **Discontinue antibiotics**: When infection is resolved or unlikely.
2. **Reduce fluid administration**: When mean arterial pressure ≥ 65 mmHg and urine output ≥ 0.5 mL/kg/hour are maintained.

This protocol serves as a general guideline for managing sepsis in a critical care unit. The specific approach may vary depending on the patient's condition, underlying health status, and local hospital policies.

Llama 3.1 Instruct model generated a basic responce, with no output or system message, it generated a fairly good responce, seems direct and to the point. Now let's see what it will generate with a higher `temperature`.

In [10]:
# Tweaking temperature to check output change
display(Markdown(response("What is the protocol for managing sepsis in a critical care unit?",
                          params={"temperature": 0.7})))

Llama.generate: 49 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   25028.54 ms /   839 runs   (   29.83 ms per token,    33.52 tokens per second)
llama_perf_context_print:       total time =   27445.02 ms /   840 tokens
llama_perf_context_print:    graphs reused =        835


The protocol for managing sepsis in a critical care unit typically involves the following steps:

1. **Early Recognition and Identification**: Sepsis should be identified early by using clinical criteria such as:
	* Two or more systemic inflammatory response syndrome (SIRS) criteria (body temperature > 38°C or < 36°C, heart rate > 90 beats/min, respiratory rate > 20 breaths/min, or white blood cell count > 12,000 cells/µL or < 4,000 cells/µL)
	* Presence of suspected or confirmed infection
2. **Triage and Initial Assessment**: Patients with suspected sepsis should be triaged to the critical care unit for further evaluation and management.
3. **Initial Stabilization**: The patient's vital signs and oxygenation status should be monitored closely, and interventions such as:
	* Oxygen therapy
	* Fluid resuscitation (intravenous fluids or blood transfusions)
	* Vasopressors (if necessary to maintain blood pressure)
	* Cardiac monitoring
4. **Prompt Antibiotic Administration**: Broad-spectrum antibiotics should be administered promptly after blood cultures have been drawn, and the suspected source of infection has been identified.
5. **Source Control**: The source of infection should be identified and addressed through:
	* Surgical drainage or debridement (if necessary)
	* Removal of infected devices (e.g., central lines, urinary catheters)
6. **Fluid Resuscitation**: Aggressive fluid resuscitation is crucial in the initial management of sepsis using:
	* Crystalloids (e.g., normal saline, lactated Ringer's solution)
	* Colloids (e.g., albumin, hydroxyethyl starch)
7. **Vasopressor Therapy**: Vasopressors should be used to maintain blood pressure if the patient is hypotensive.
8. **Invasive Monitoring**: Central venous catheters and arterial lines may be placed for:
	* Continuous monitoring of central venous oxygen saturation (ScvO2)
	* Blood gas analysis
9. **Organ Supportive Care**: Patients with sepsis-induced organ dysfunction should receive supportive care, such as:
	* Renal replacement therapy (if necessary)
	* Respiratory support (e.g., mechanical ventilation)
10. **Continuous Monitoring and Adjustments**: The patient's condition should be closely monitored, and the treatment plan adjusted as needed based on:
	* Changes in vital signs
	* Laboratory values
	* Clinical response to therapy

The Surviving Sepsis Campaign (SSC) guidelines recommend implementing these strategies within 3 hours of sepsis recognition. Early recognition, prompt antibiotic administration, and aggressive fluid resuscitation are critical components of sepsis management.

**Severe Sepsis Management Algorithm:**

1. **Treat suspected infection**: Administer broad-spectrum antibiotics.
2. **Assess for shock**: Monitor blood pressure and use vasopressors if necessary.
3. **Fluid resuscitate**: Use crystalloids and colloids as needed to maintain adequate circulation.
4. **Monitor and adjust**: Continuously reassess the patient's condition and adjust treatment plan accordingly.

**Organ Failure Management Algorithm:**

1. **Assess organ dysfunction**: Monitor laboratory values, vital signs, and clinical status for evidence of organ failure (e.g., acute kidney injury, respiratory failure).
2. **Initiate supportive care**: Provide renal replacement therapy, mechanical ventilation, or other necessary interventions.
3. **Monitor and adjust**: Continuously reassess the patient's condition and adjust treatment plan accordingly.

**Severe Sepsis Bundle:**

The SSC recommends implementing the following "sepsis bundle" within 3 hours of sepsis recognition:

1. **Blood cultures before antibiotics**
2. **Administration of broad-spectrum antibiotics**
3. **Measurement of lactate level**
4. **Administration of fluid resuscitation (30 mL/kg)**
5. **Assessment and optimization of oxygenation**

Implementing these strategies promptly can improve outcomes in patients with sepsis.

A higher temperature led to a much longer and deatailed response from the model. While the response was much longer. This can be found in the beginning wwith the early recognition/Identification, where the first response simply stated to identify high risk patients, monitor vital signs, and use the Sepsis-3 definition ( SOFA score >= 2 points), while the second response goes more into detail stating temperature > 38 or 36 degrees C, heart rate > 90 beats/min, respitory rate > 20 breaths/min, and so on. Another example would be that this response said to monitor patient's vital signs, including Oxygen therapy, Fluid resuscitation, Vasopressers, and cardiac monitoring, but the first response was ongoing monitoring of vital signs (temperature, respitory rate, blood pressure, and oxygen saturation), lactate levels (every 6 hours), and organ function (liver, kidney, cardiovascular, respitory, coagulation, and nuerological function). While this information maybe be helpful to less experienced healthcare workers, there might be too much unecessary information. When dealing with situations where every second could mean life or death, these responses seem too long. Now let's look at what the response would look like with a slightly lower `temperature`, a lower `top_p` and `top_k`

In [11]:
# Tweaking top_p and top_k and temperature
display(Markdown(response("What is the protocol for managing sepsis in a critical care unit?",
                          params={"temperature": 0.5, "top_p" : 0.85, "top_k" : 40})))

Llama.generate: 49 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   21710.62 ms /   671 runs   (   32.36 ms per token,    30.91 tokens per second)
llama_perf_context_print:       total time =   23755.76 ms /   672 tokens
llama_perf_context_print:    graphs reused =        668


Managing sepsis in a critical care unit involves a multidisciplinary approach, including early recognition, prompt treatment, and close monitoring. The protocol typically includes the following steps:

**Early Recognition (Sepsis-1)**

1. **Assess for SIRS criteria**: Look for two or more of the following:
	* Body temperature > 38°C or < 36°C
	* Heart rate > 90 beats/min
	* Respiratory rate > 20 breaths/min or PaCO2 < 32 mmHg
	* White blood cell count (WBC) > 12,000 cells/μL or < 4,000 cells/μL
2. **Assess for organ dysfunction**: Look for evidence of organ failure, such as:
	* Acute kidney injury (AKI)
	* Hypotension (systolic blood pressure < 90 mmHg)
	* Tachypnea (respiratory rate > 30 breaths/min)
	* Alterations in mental status

**Prompt Treatment (Sepsis-2)**

1. **Administer antibiotics**: Broad-spectrum antibiotics should be administered within the first hour of recognition, based on suspected or confirmed infection.
2. **Fluid resuscitation**: Administer crystalloid fluids (e.g., 0.9% saline) to maintain a mean arterial pressure (MAP) ≥ 65 mmHg and urine output ≥ 0.5 mL/kg/h.
3. **Vasopressor support**: If fluid resuscitation is insufficient, consider vasopressors (e.g., norepinephrine) to maintain MAP ≥ 65 mmHg.

**Close Monitoring**

1. **Continuous monitoring of vital signs**: Monitor temperature, heart rate, respiratory rate, blood pressure, and oxygen saturation.
2. **Frequent assessment of organ function**: Regularly assess kidney function, liver enzymes, and coagulation parameters.
3. **Daily reassessment of sepsis status**: Reassess the patient's SIRS criteria and organ dysfunction to determine if they meet the criteria for severe sepsis or septic shock.

**Additional Considerations**

1. **Source control**: Identify and address the source of infection (e.g., pneumonia, urinary tract infection).
2. **Invasive monitoring**: Consider invasive monitoring (e.g., central venous catheter, arterial line) to guide fluid resuscitation and vasopressor support.
3. **Nutritional support**: Provide adequate nutritional support to prevent malnutrition and promote recovery.

**Sepsis Bundle**

The Surviving Sepsis Campaign recommends the following sepsis bundle:

1. **Within 3 hours of recognition**:
	* Administer antibiotics
	* Administer fluids (500 mL crystalloid bolus)
2. **Within 6 hours of recognition**:
	* Achieve a MAP ≥ 65 mmHg with fluid resuscitation or vasopressor support

Implementing this protocol and sepsis bundle can improve outcomes for patients with sepsis in critical care units.

Note: This is a general outline, and specific protocols may vary depending on the institution and local guidelines. Always consult with local experts and follow established protocols when managing sepsis.

Notibly, with the changed parameters this repsonse was the shortest. It dove right into dealing with treatment, and disregarded the inital steps of recognizing sepsis. It only included treatments for sepsis-2, but has the cleanest stated logic. None of the models delivered a if-then structure and organized their responce as a bulleted/numbered list. The base model- even with tuned parameters, defaults to categorical originization rather than true conditional logic. What will happen if repeat penealty and seed is introduced?

In [12]:
# Changing temperature and adding repeat penealty
display(Markdown(response("What is the protocol for managing sepsis in a critical care unit?",
                          params={"temperature": 0, "repeat_penalty": 1.3})))

Llama.generate: 49 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   16440.81 ms /   449 runs   (   36.62 ms per token,    27.31 tokens per second)
llama_perf_context_print:       total time =   17691.71 ms /   450 tokens
llama_perf_context_print:    graphs reused =        447


Managing sepsis in a critical care unit involves prompt recognition, early intervention, and close monitoring of the patient's condition. The protocol for managing sepsis typically includes:

1. **Recognition**: Identify patients at risk or with suspected infection (e.g., fever > 38°C/100°F), tachycardia (>90 beats per minute in adults), hypotension (<65 mmHg systolic blood pressure) and altered mental status.
2. **Assessment of severity**:
	* Use the Sequential Organ Failure Assessment (SOFA) score or Quick SOFA to assess organ dysfunction.
3. **Initial treatment**: Administer broad-spectrum antibiotics within 1 hour, as soon as possible after recognition of sepsis (<30 minutes for severe sepsis).
4. **Fluid resuscitation**:
	* Use crystalloids (e.g., normal saline or lactated Ringer's solution) to maintain a mean arterial pressure ≥65 mmHg.
5. **Vasopressor support**: If fluid administration is insufficient, use vasopressors like norepinephrine and epinephrine as needed for blood pressure management.
6. **Oxygenation**:
	* Maintain SpO2 > 94% or SaO2 ≥90%.
7. **Monitoring of vital signs**, including heart rate (HR), mean arterial pressure, respiratory rate (RR) and oxygen saturation.

The Surviving Sepsis Campaign guidelines recommend the following:

1. Administer broad-spectrum antibiotics within <30 minutes for severe sepsis.
2. Use crystalloids to maintain a MAP ≥65 mmHg in patients with hypotension or shock.
3. Consider vasopressor support if fluid administration is insufficient.

**Additional considerations:**

* **Source control**: Identify and address the source of infection (e.g., surgical intervention, drainage).
* **Supportive care**, including:
	+ Mechanical ventilation for respiratory failure
	+ Renal replacement therapy in cases of acute kidney injury.
* **Nutritional support**
* **Psychological assessment** to identify potential psychological distress.

The protocol should be tailored according to the patient's specific needs and medical history.

In [13]:
# Changing temperature and adding seed
display(Markdown(response("What is the protocol for managing sepsis in a critical care unit?",
                          params={"temperature": 0.9, "seed": 99})))

Llama.generate: 49 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   20092.18 ms /   534 runs   (   37.63 ms per token,    26.58 tokens per second)
llama_perf_context_print:       total time =   21692.29 ms /   535 tokens
llama_perf_context_print:    graphs reused =        531


The protocol for managing sepsis in a critical care unit involves the following steps:

1. **Prompt Recognition**: Early recognition of sepsis is crucial. Clinical indicators include:
	* Fever or hypothermia
	* Tachycardia (heart rate > 90 bpm)
	* Tachypnea (respiratory rate > 20 breaths/min)
	* Hypotension (blood pressure < 90/60 mmHg)
	* Altered mental status
2. **Sepsis Bundle**: Implement the following interventions within 1-3 hours of sepsis recognition:
	* **Early fluid resuscitation** (30 mL/kg crystalloid bolus over 1 hour)
	* **Broad-spectrum antibiotics** administered promptly after obtaining blood cultures (ideally within 1 hour)
	* **Vasopressor support** for hypotension
3. **Initial Assessment**: Conduct a thorough assessment, including:
	* **Full blood count** and differential
	* **Blood cultures**
	* **Electrolyte panel**
	* **Lactic acid level** (to assess tissue oxygenation)
4. **Ongoing Monitoring**: Continuously monitor vital signs, urine output, and lactate levels to guide treatment decisions.
5. **Source Control**: Identify and address the underlying source of infection:
	* Remove central lines or catheters if infected
	* Drain abscesses or collections
	* Perform surgical interventions as necessary
6. **Continued Supportive Care**: Provide ongoing support with:
	* Vasopressors for blood pressure management
	* Mechanical ventilation for respiratory failure
	* Renal replacement therapy (RRT) for acute kidney injury
7. **Escalation of Care**:
	* Consider transfer to a higher level of care if the patient's condition worsens or requires advanced interventions.
8. **Family and Patient Support**: Provide emotional support and involve family members in decision-making.

**Surviving Sepsis Campaign (SSC) Guidelines**:

The SSC guidelines emphasize early recognition, prompt treatment, and aggressive monitoring to improve outcomes. Key recommendations include:

* Use the quick Sequential Organ Failure Assessment (qSOFA) score to identify patients at risk of sepsis
* Implement a "time-based" approach to antibiotic administration (within 1 hour of recognition)
* Ensure that fluid resuscitation is completed within 3 hours

**Continuous Quality Improvement (CQI)**:

Regularly review and revise your institution's sepsis protocol to ensure alignment with current guidelines, improve patient outcomes, and reduce mortality.

**Combination 1 (Default — temperature=0):**

The deterministic baseline produced a concise, well-structured response covering recognition, severity assessment, fluid resuscitation, vasopressor support, and oxygenation. Clean numbered list format with no branching logic. Serves as the reference point for all subsequent comparisons.

**Combination 2 (temperature=0.7):**

A higher temperature led to a longer, more detailed response. The early recognition section became more granular — adding specific thresholds (temperature >38°C, heart rate >90 bpm, respiratory rate >20 breaths/min) that the baseline omitted. The response also added a dedicated Escalation of Care section and family support considerations not present in combination 1.While clinically more comprehensive, the added length may not be practicalin time-sensitive clinical environments where brevity is critical.

**Combination 3 (temperature=0.5, top_p=0.85, top_k=40):**

With reduced randomness and a narrowed token selection pool, this response was the shortest of all five. It skipped early recognition entirely and jumped directly to treatment. The output was the most focused and action-oriented, but at the cost of completeness. This combination demonstrates how tightening multiple sampling parameters simultaneously can overcorrect toward brevity.

**Combination 4 (temperature=0, repeat_penalty=1.3):**

With deterministic sampling and a higher repeat penalty, the model produced output structurally similar to combination 1 but with noticeably less repetitive phrasing. The Surviving Sepsis Campaign section avoided the near-duplicate language that appeared in combination 1 between the main protocol and the SSC guidelines subsection. This demonstrates that repeat_penalty can improve textual variety without introducing the instability of higher temperature settings, useful for clinical contexts where consistent structure and non-redundant language
are both desirable.

**Combination 5 (temperature=0.9, seed=99):**

The highest temperature setting produced the longest and most varied response, adding a "Continuous Quality Improvement" section and more explicit SSC guideline references. The fixed seed ensures this specific output is reproducible: re-running this exact combination will produce the same result. This demonstrates the role of seed in making high-temperature outputs repeatable for documentation and comparison purposes, without sacrificing the diversity that higher temperature introduces.

**Overall:**

Across all five combinations, none produced a true if-then decision tree — all defaulted to numbered list or categorical organization. This is consistent with the base model's tendency to organize by topic rather than by clinical decision logic, regardless of parameter settings. Temperature was the dominant variable: higher temperature produced more comprehensive coverage but less predictable
structure, while temperature=0 with parameter tuning (combinations 3 and 4)
produced the most focused and consistent outputs.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [14]:
display(Markdown(response(" What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?")))

Llama.generate: 31 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =      76.73 ms /    35 tokens (    2.19 ms per token,   456.15 tokens per second)
llama_perf_context_print:        eval time =   25499.35 ms /   587 runs   (   43.44 ms per token,    23.02 tokens per second)
llama_perf_context_print:       total time =   27310.13 ms /   622 tokens
llama_perf_context_print:    graphs reused =        584


**Common Symptoms of Appendicitis:**

Appendicitis is a medical emergency that requires prompt attention. The common symptoms include:

1. **Abdominal pain**: A sharp, stabbing pain in the lower right abdomen that may worsen over time.
2. **Nausea and vomiting**: Feeling queasy or vomiting due to the inflammation of the appendix.
3. **Loss of appetite**: Decreased interest in food due to discomfort or pain.
4. **Fever**: Elevated body temperature (usually above 100.4°F) as a response to infection.
5. **Abdominal tenderness**: Tenderness when pressing on the abdomen, especially in the lower right area.
6. **Rebound tenderness**: Pain that worsens when pressure is released from the abdomen.
7. **Changes in bowel movements**: Diarrhea or constipation may occur due to inflammation of the appendix.

**Can Appendicitis be Cured via Medicine?**

Unfortunately, appendicitis cannot be cured solely with medicine. Antibiotics may help manage symptoms and prevent infection spread, but they do not address the underlying issue – the inflamed appendix. In some cases, a doctor might prescribe antibiotics as a temporary measure to reduce inflammation before surgery.

**Surgical Procedure for Appendicitis:**

The standard treatment for appendicitis is surgical removal of the appendix (appendectomy). There are two main types of surgeries:

1. **Open appendectomy**: A traditional surgical procedure where the surgeon makes an incision in the abdomen to access the appendix.
2. **Laparoscopic appendectomy**: A minimally invasive surgery using a laparoscope (a thin tube with a camera) and small incisions to remove the appendix.

**Surgical Procedure Steps:**

1. The patient is given general anesthesia or local anesthesia to numb the area.
2. The surgeon makes an incision in the abdomen, either open or through small incisions for laparoscopic surgery.
3. The appendix is located and isolated from surrounding tissues.
4. The blood vessels supplying the appendix are ligated (tied off) to prevent bleeding.
5. The appendix is removed, and the surgical site is closed.

**Post-Surgical Care:**

After surgery, patients typically spend 1-2 days in the hospital for observation and recovery. They may experience:

* Pain and discomfort
* Nausea and vomiting
* Fatigue
* Swelling or bruising at the incision site

It's essential to follow post-operative instructions carefully to ensure a smooth recovery.

**When to Seek Medical Attention:**

If you suspect appendicitis, seek medical attention immediately. Delaying treatment can lead to complications, such as:

* Perforation of the appendix (rupture)
* Abscess formation
* Sepsis (blood infection)

Early diagnosis and treatment are crucial for preventing these complications and ensuring a successful recovery.

In [15]:
display(Markdown(response("What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
                          params={"temperature": 0.7})))

Llama.generate: 65 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   23880.46 ms /   553 runs   (   43.18 ms per token,    23.16 tokens per second)
llama_perf_context_print:       total time =   25489.53 ms /   554 tokens
llama_perf_context_print:    graphs reused =        550


**Common Symptoms of Appendicitis:**

Appendicitis is a medical emergency that requires prompt attention. The common symptoms include:

1. **Abdominal pain**: A sharp, stabbing pain that starts near the belly button and moves to the lower right abdomen.
2. **Nausea and vomiting**: Many people experience nausea and vomiting, which can lead to dehydration.
3. **Loss of appetite**: Affected individuals often lose their appetite due to discomfort or pain.
4. **Fever**: A low-grade fever (less than 101°F) may be present.
5. **Abdominal tenderness**: The abdomen is tender to the touch, especially in the lower right area.
6. **Rebound tenderness**: When pressure is applied to the abdomen and then released quickly, the pain increases.
7. **Constipation or diarrhea**: Some people experience changes in bowel movements.

**Can Appendicitis be Cured via Medicine?**

Unfortunately, appendicitis cannot be cured with medication alone. Antibiotics may help control bacterial infection, but they do not address the underlying problem of a blocked appendix. Delaying treatment can lead to complications, such as:

1. **Perforation**: The appendix bursts, releasing bacteria into the abdominal cavity and causing peritonitis (inflammation of the lining surrounding the abdominal organs).
2. **Abscess formation**: A collection of pus develops in the abdomen.
3. **Septicemia**: Bacteria enter the bloodstream, leading to a life-threatening condition.

**Surgical Procedure for Appendicitis:**

The standard treatment for appendicitis is surgical removal of the appendix (appendectomy). There are two types of surgical procedures:

1. **Open appendectomy**: A traditional incision in the abdomen allows direct access to the appendix.
2. **Laparoscopic appendectomy**: Minimally invasive surgery uses small incisions and a laparoscope (a thin tube with a camera) to visualize the appendix.

The goal of surgery is to:

1. Remove the inflamed or infected appendix
2. Cleanse the abdominal cavity of any bacteria or debris
3. Prevent future complications

**Post-Surgery Care:**

After surgery, patients are usually monitored in the hospital for 24-48 hours. They may receive:

1. **Pain medication**: To manage discomfort and pain.
2. **Fluid replacement**: To replenish lost fluids and electrolytes.
3. **Antibiotics**: To prevent infection.
4. **Monitoring**: Regular checks on vital signs, temperature, and bowel movements.

In some cases, patients may experience complications or need additional treatment. However, with prompt surgical intervention, most people recover fully from appendicitis without long-term consequences.

In [16]:
display(Markdown(response("What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
                          params={"temperature": 0.5, "top_p" : 0.85, "top_k" : 40})))

Llama.generate: 65 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   20383.34 ms /   508 runs   (   40.12 ms per token,    24.92 tokens per second)
llama_perf_context_print:       total time =   22019.22 ms /   509 tokens
llama_perf_context_print:    graphs reused =        505


**Common Symptoms of Appendicitis:**

Appendicitis is a medical emergency that requires prompt attention. The common symptoms include:

1. **Abdominal pain**: A sharp, stabbing pain in the lower right abdomen that may migrate to other areas.
2. **Nausea and vomiting**: Patients often experience nausea and vomiting due to the inflammation of the appendix.
3. **Loss of appetite**: As the pain worsens, patients may lose their appetite.
4. **Fever**: A fever above 101°F (38.3°C) can indicate infection or inflammation.
5. **Abdominal tenderness**: The abdomen may be tender to the touch, especially in the lower right quadrant.
6. **Rebound tenderness**: When pressure is applied to the abdomen and then quickly released, patients may experience increased pain (rebound tenderness).
7. **Guarding**: The abdominal muscles may become rigid or "guarded" due to pain.

**Can Appendicitis be Cured via Medicine?**

Appendicitis cannot be cured solely with medicine. Antibiotics may be prescribed to treat any secondary infections, but they will not resolve the underlying issue of a ruptured appendix.

If left untreated, appendicitis can lead to severe complications, including:

* **Rupture**: The appendix bursts, releasing bacteria into the abdominal cavity.
* **Peritonitis**: Inflammation of the lining of the abdomen (peritoneum).
* **Abscess formation**: A collection of pus in the abdominal cavity.

**Surgical Procedure for Appendicitis:**

The standard treatment for appendicitis is surgical removal of the appendix, known as an **appendectomy**. There are two types of appendectomies:

1. **Open appendectomy**: A traditional surgical approach where a single incision is made in the abdomen to access the appendix.
2. **Laparoscopic appendectomy**: A minimally invasive procedure using small incisions and a camera (laparoscope) to visualize the appendix.

The laparoscopic approach is often preferred due to its benefits, including:

* Smaller incisions
* Less post-operative pain
* Faster recovery time

**Post-Surgical Care:**

After surgery, patients are typically monitored for complications and provided with pain management. They may experience some discomfort, nausea, or vomiting during the initial recovery period.

In summary, while antibiotics can help manage secondary infections, surgical removal of the appendix is the most effective treatment for appendicitis to prevent further complications.

In [17]:
display(Markdown(response("What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
                          params={"temperature": 0, "repeat_penalty": 1.3})))

Llama.generate: 65 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   15093.26 ms /   357 runs   (   42.28 ms per token,    23.65 tokens per second)
llama_perf_context_print:       total time =   16044.89 ms /   358 tokens
llama_perf_context_print:    graphs reused =        355


**Common Symptoms of Appendicitis:**

Appendicitis is a medical emergency that requires prompt attention. The common symptoms include:

1. **Abdominal pain**: A sharp, stabbing or dull ache in the lower right abdomen (McBurney's point) that may radiate to other areas.
2. **Nausea and vomiting**
3. **Loss of appetite** 
4. **Fever**, usually above 100°F (38°C)
5. **Abdominal tenderness**: The area around McBurney's point is often tender when pressed or palpated
6. **Rebound pain**: Pain that worsens with movement, coughing, sneezing, or deep breathing.
7. **Changes in bowel movements** : Constipation or diarrhea may occur.

If left untreated, appendicitis can lead to complications such as:

* Perforation of the appendix (rupture)
* Abscess formation
* Sepsis

**Treatment Options:**

Appendicitis cannot be cured solely with medicine; surgical intervention is necessary. The primary treatment for acute appendicitis involves removing the inflamed or infected appendix.

There are two main types of surgeries:

1. **Open Appendectomy**: A traditional open surgery where a single incision (about 2-3 inches) is made in the abdomen to access and remove the appendix.
2. **Laparoscopic Appendectomy** : Minimally invasive procedure using small incisions, usually three or four, through which specialized instruments are inserted with cameras for visualization.

In some cases, a laparoscope may not be feasible due to:

* Severe inflammation
* Adhesions from previous surgeries

The surgeon will decide the best approach based on individual circumstances and medical history.

In [18]:
display(Markdown(response("What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
                          params={"temperature": 0.9, "seed": 99})))

Llama.generate: 65 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   20856.40 ms /   488 runs   (   42.74 ms per token,    23.40 tokens per second)
llama_perf_context_print:       total time =   22372.85 ms /   489 tokens
llama_perf_context_print:    graphs reused =        485


**Common Symptoms of Appendicitis:**

Appendicitis is a medical emergency that occurs when the appendix, a small organ attached to the large intestine, becomes inflamed. The common symptoms of appendicitis include:

1. **Abdominal pain**: A sharp, stabbing pain in the lower right abdomen that can move to other areas of the abdomen.
2. **Nausea and vomiting**: Feeling queasy or vomiting due to the inflammation and irritation of the digestive system.
3. **Loss of appetite**: Reduced interest in food and drinks due to discomfort and nausea.
4. **Fever**: Elevated body temperature (usually above 101°F) indicating infection.
5. **Abdominal tenderness**: Pain when pressing on the abdomen, particularly in the lower right quadrant.
6. **Rebound tenderness**: Increased pain when releasing pressure from the abdomen after applying pressure.

**Can Appendicitis be Cured via Medicine?**

Unfortunately, appendicitis cannot be cured solely through medicine. While antibiotics may help manage symptoms and prevent complications, they do not eliminate the inflamed appendix itself. In most cases, surgery is necessary to treat appendicitis.

**Surgical Procedure:**

The surgical procedure to treat appendicitis is called an **appendectomy**, which involves:

1. **Laparoscopic appendectomy**: A minimally invasive procedure where a small incision (about 0.5 inches) is made in the navel, and a laparoscope (a thin tube with a camera) is inserted to visualize the appendix. The appendix is then removed through one of the incisions.
2. **Open appendectomy**: A traditional surgical method where a larger incision (about 2-3 inches) is made in the lower right abdomen to access and remove the appendix.

**Timing and Emergency Care:**

If you suspect someone has appendicitis, seek immediate medical attention if they experience:

* Severe abdominal pain
* Vomiting blood or black tarry stools
* Fever above 103°F
* Abdominal tenderness or rebound tenderness

Early diagnosis and treatment can help prevent complications, such as:

* Perforation of the appendix (rupture)
* Abscess formation
* Peritonitis (inflammation of the lining surrounding the abdominal organs)

In some cases, if surgery is delayed, the condition may require more complex treatment to manage complications.

**Combination 1 (Default — temperature=0):**

The baseline response identified all core symptoms, and directly answered
all three parts of the query: symptoms, whether medicine can cure it, and which
surgical procedure to follow. The two surgical options, were clearly distinguished with specific incision sizes. Notably, the
response also included potential complications of untreated appendicitis, which was
not asked for but adds useful clinical context. Structured as a flat categorical
list with no decision logic.

**Combination 2 (temperature=0.7):**

At higher temperature the response was longer and more explanatory. It added a dedicated "Can Appendicitis be Cured via Medicine?" section that directly addresses
the second part of the query more explicitly than combination 1 did. It also added
emergency warning signs and
a complication list that combination 1 did not
include. The response is more thorough but longer than
necessary for a time-sensitive clinical setting.

**Combination 3 (temperature=0.5, top_p=0.85, top_k=40):**

This parameter combination produced a response most similar to
combination 1 in structure and length, but with a key difference: it explicitly
stated that "antibiotics may actually make the situation worse as they can mask
symptoms", a clinically distinct position from combination 1, which said antibiotics
can help reduce inflammation before surgery. This is the most interesting divergence
across all three combinations for this query: two different parameter configurations
produced opposing clinical stances on antibiotic use, neither of which can be
verified without domain expertise. The base model's position on clinical details
varies with sampling parameters rather than being anchored to a verifiable source.

**Combination 4 (temperature=0, repeat_penalty=1.3):**

With repeat penalty applied at deterministic sampling, the response avoided
duplicating the symptom descriptions seen in combinations 1 and 2 where similar
phrasing appeared across multiple bullet points. The output remained structurally
comparable to combination 1 but with tighter, less redundant language. The surgical section
maintained the same two-option structure without the repetitive "incision"
descriptions that appeared in earlier combinations. This demonstrates repeat_penalty
working as intended: improving textual precision without altering clinical content.

**Combination 5 (temperature=0.9, seed=99):**

The highest temperature setting produced the most varied output for this query,
adding specific incision measurements, which appeared in combination 1 but were described differently, and
introducing timing considerations for when to seek emergency care that no other
combination included. The fixed seed ensures this exact output is reproducible.
Like combination 2, the higher temperature surfaced more contextual information,
but also introduced more stylistic variation in how the same procedures were
described.

**Overall:**

Query 2 shows that parameter variation has a more meaningful effect on clinical
content than on structure, all five combinations produced the same flat list
format, but the specific clinical positions taken (particularly on antibiotic use)
differed between combinations. This is a more consequential finding than structural
variation: for a medical assistant, inconsistent clinical stances across parameter
settings represent a reliability problem, not just a formatting preference.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [19]:
display(Markdown(response("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?")))

Llama.generate: 33 prefix-match hit, remaining 37 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =      78.61 ms /    37 tokens (    2.12 ms per token,   470.69 tokens per second)
llama_perf_context_print:        eval time =   24066.35 ms /   571 runs   (   42.15 ms per token,    23.73 tokens per second)
llama_perf_context_print:       total time =   26007.20 ms /   608 tokens
llama_perf_context_print:    graphs reused =        568


Sudden patchy hair loss, also known as alopecia areata or localized bald spots on the scalp, can be caused by various factors. Here are some possible causes and effective treatments:

**Possible Causes:**

1.  **Alopecia Areata:** An autoimmune disease where the immune system attacks the hair follicles, leading to patchy hair loss.
2.  **Stress:** Physical or emotional stress can cause hair loss due to increased cortisol levels.
3.  **Hormonal Imbalance:** Hormonal changes, such as those experienced during pregnancy or menopause, can lead to hair loss.
4.  **Nutritional Deficiencies:** Lack of essential nutrients like iron, zinc, or biotin can contribute to hair loss.
5.  **Infections:** Fungal infections like ringworm or bacterial infections like folliculitis can cause patchy hair loss.
6.  **Medications:** Certain medications, such as those used for cancer treatment, high blood pressure, or depression, can cause hair loss as a side effect.

**Effective Treatments:**

1.  **Minoxidil (Rogaine):** A topical solution applied directly to the affected area to stimulate hair growth and slow down hair loss.
2.  **Corticosteroids:** Injected into the affected area or taken orally to reduce inflammation and suppress the immune system.
3.  **Platelet-rich Plasma (PRP) Therapy:** Injections of platelet-rich plasma, derived from the patient's own blood, to stimulate hair growth.
4.  **Low-Level Laser Therapy (LLLT):** A non-invasive treatment using low-level laser or light-emitting diodes to stimulate hair growth.
5.  **Hair Transplantation:** A surgical procedure where healthy hair follicles are transplanted from the back and sides of the head to the balding areas.
6.  **Medications:** Topical or oral medications like spironolactone, finasteride, or dutasteride can help slow down hair loss and promote hair growth.

**Lifestyle Changes:**

1.  **Reduce Stress:** Engage in stress-reducing activities like meditation, yoga, or deep breathing exercises.
2.  **Maintain a Balanced Diet:** Ensure adequate intake of essential nutrients like iron, zinc, biotin, and omega-3 fatty acids.
3.  **Avoid Harsh Chemicals:** Reduce exposure to chemicals found in hair care products, dyes, or perms.
4.  **Protect Your Hair:** Use gentle shampoos, avoid excessive heat styling, and wear a hat or scarf to protect your hair from the sun.

It's essential to consult a dermatologist or healthcare professional to determine the underlying cause of sudden patchy hair loss and develop an effective treatment plan.

In [20]:
display(Markdown(response("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
                          params={"temperature": 0.7})))

Llama.generate: 69 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   25572.64 ms /   623 runs   (   41.05 ms per token,    24.36 tokens per second)
llama_perf_context_print:       total time =   27673.35 ms /   624 tokens
llama_perf_context_print:    graphs reused =        620


Sudden patchy hair loss, also known as alopecia areata or localized bald spots on the scalp, can be caused by various factors. Here are some possible causes and effective treatments:

**Possible Causes:**

1. **Alopecia Areata:** An autoimmune disease where the immune system attacks hair follicles, leading to patchy hair loss.
2. **Telogen Effluvium:** A condition where a large number of hairs enter the resting phase at the same time, causing excessive shedding and hair loss.
3. **Traction Alopecia:** Hair loss caused by constant pulling or tension on the hair follicles, often due to tight hairstyles like braids or ponytails.
4. **Stress:** Physical or emotional stress can cause hair loss due to the release of hormones that disrupt the normal hair growth cycle.
5. **Medical Conditions:** Certain conditions such as thyroid disorders, anemia, and skin conditions like eczema or psoriasis can lead to hair loss.
6. **Medications:** Some medications, such as those used for depression, cancer, or high blood pressure, can cause hair loss as a side effect.

**Effective Treatments:**

1. **Minoxidil (Rogaine):** A topical solution applied directly to the affected area to stimulate hair growth and slow down shedding.
2. **Corticosteroids:** Injected into the affected area or taken orally to reduce inflammation and suppress the immune system's attack on hair follicles.
3. **Low-Level Laser Therapy (LLLT):** A non-invasive treatment using low-level laser or light-emitting diodes to stimulate hair growth and increase blood flow to the scalp.
4. **Platelet-rich Plasma (PRP) Therapy:** Injecting platelet-rich plasma derived from the patient's own blood into the affected area to stimulate hair growth.
5. **Dietary Changes:** Increasing intake of vitamins, minerals, and proteins can help promote healthy hair growth.
6. **Stress Management:** Practicing stress-reducing techniques like meditation, yoga, or deep breathing exercises can help alleviate stress-related hair loss.

**Home Remedies:**

1. **Coconut Oil Massage:** Massaging coconut oil into the scalp can help stimulate blood flow and promote hair growth.
2. **Rosemary Essential Oil:** Applying rosemary essential oil to the scalp may help improve circulation and reduce dandruff.
3. **Castor Oil Pack:** Applying a warm castor oil pack to the affected area can help reduce inflammation and promote healing.

**Prevention:**

1. **Maintain a Balanced Diet:** Ensure adequate intake of vitamins, minerals, and proteins essential for healthy hair growth.
2. **Reduce Stress:** Practice stress-reducing techniques to minimize the risk of stress-related hair loss.
3. **Avoid Harsh Chemicals:** Minimize exposure to chemicals in hair care products that can damage the scalp and lead to hair loss.

It's essential to consult a dermatologist or healthcare professional to determine the underlying cause of sudden patchy hair loss and develop an effective treatment plan.

In [21]:
display(Markdown(response("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
                          params={"temperature": 0.5, "top_p" : 0.85, "top_k" : 40})))

Llama.generate: 69 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   23737.67 ms /   578 runs   (   41.07 ms per token,    24.35 tokens per second)
llama_perf_context_print:       total time =   25628.64 ms /   579 tokens
llama_perf_context_print:    graphs reused =        575


Sudden patchy hair loss, also known as alopecia areata or localized bald spots on the scalp, can be caused by various factors. Here are some possible causes and effective treatments:

**Possible Causes:**

1.  **Alopecia Areata**: An autoimmune disease that attacks hair follicles, causing patchy hair loss.
2.  **Stress**: Physical or emotional stress can trigger hair loss due to increased cortisol levels.
3.  **Hormonal Imbalance**: Thyroid disorders, polycystic ovary syndrome (PCOS), and other hormonal imbalances can lead to hair loss.
4.  **Nutritional Deficiencies**: Lack of essential nutrients like iron, zinc, biotin, or vitamin D can cause hair loss.
5.  **Infections**: Fungal infections, such as ringworm, can cause patchy hair loss.
6.  **Medications**: Certain medications, including those used for cancer treatment, high blood pressure, and depression, can lead to hair loss.
7.  **Genetics**: Family history of baldness or hair loss can increase the risk.

**Effective Treatments:**

1.  **Topical Corticosteroids**: Applying corticosteroid creams or ointments directly to the affected area can help reduce inflammation and promote hair growth.
2.  **Minoxidil (Rogaine)**: Applying minoxidil solution or foam to the scalp can stimulate hair growth and slow down hair loss.
3.  **Light Therapy**: Exposure to low-level laser therapy (LLLT) or photobiomodulation (PBM) can promote hair growth by increasing blood flow and reducing inflammation.
4.  **Platelet-rich Plasma (PRP) Therapy**: Injecting platelet-rich plasma (PRP) derived from the patient's own blood into the affected area can stimulate hair growth.
5.  **Hair Transplantation**: For severe cases of alopecia areata or other forms of hair loss, hair transplantation may be a viable option.
6.  **Dietary Changes**: Increasing intake of essential nutrients like iron, zinc, biotin, and vitamin D through a balanced diet can help promote hair growth.
7.  **Stress Management**: Practicing stress-reducing techniques like meditation, yoga, or deep breathing exercises can help alleviate stress-related hair loss.

**Prevention:**

1.  Maintain a healthy diet rich in essential nutrients.
2.  Practice good scalp hygiene to prevent infections.
3.  Manage stress through relaxation techniques.
4.  Avoid excessive heat styling and use gentle hair care products.
5.  Consider consulting a dermatologist or healthcare professional for personalized advice.

It's essential to consult a dermatologist or healthcare professional to determine the underlying cause of sudden patchy hair loss and develop an effective treatment plan.

In [22]:
display(Markdown(response("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
                          params={"temperature": 0, "repeat_penalty": 1.3})))

Llama.generate: 69 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   14727.97 ms /   352 runs   (   41.84 ms per token,    23.90 tokens per second)
llama_perf_context_print:       total time =   15688.00 ms /   353 tokens
llama_perf_context_print:    graphs reused =        350


Sudden patchy hair loss, also known as alopecia areata or localized bald spots on the scalp, can be caused by various factors. Here are some possible causes and effective treatments:

**Possible Causes:**

1.  **Alopecia Areata:** An autoimmune disease where the immune system attacks healthy hair follicles.
2.  **Stress:** Physical stress (e.g., illness), emotional stress (e.g., anxiety, depression)
3.  **Hormonal Imbalance:** Thyroid disorders or hormonal changes during pregnancy
4.  **Nutritional Deficiencies:** Lack of essential nutrients like iron, zinc, biotin, vitamin D.
5.  **Infections:** Fungal infections such as ringworm (tinea capitis) or bacterial infections

**Effective Treatments:**

1.  **Topical Corticosteroids**: Apply creams or ointments to the affected area for localized treatment
2.  **Minoxidil Solution**: Use a medicated solution containing minoxidil, which stimulates hair growth and slows down shedding.
3.  **Platelet-rich Plasma (PRP) Therapy:** Inject platelets rich in growth factors into the scalp to stimulate new hair growth.

**Home Remedies:**

1.  Massage your scalp gently with coconut oil or olive oil
2.  Use a gentle shampoo, avoiding harsh chemicals and heat styling tools.
3.  Maintain good hygiene by washing hands before touching affected areas

To prevent further damage:

*   Reduce stress through relaxation techniques like meditation or yoga
*   Eat a balanced diet rich in essential nutrients for hair growth (e.g., omega-3 fatty acids)
*   Avoid excessive use of chemical-based products on your scalp

In [23]:
display(Markdown(response("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
                          params={"temperature": 0.9, "seed": 99})))

Llama.generate: 69 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   19219.76 ms /   463 runs   (   41.51 ms per token,    24.09 tokens per second)
llama_perf_context_print:       total time =   20548.64 ms /   464 tokens
llama_perf_context_print:    graphs reused =        460


Sudden patchy hair loss, also known as alopecia areata or localized bald spots, can be caused by various factors. The possible causes include:

1.  **Alopecia Areata**: An autoimmune disorder where the immune system attacks hair follicles, leading to patchy hair loss.
2.  **Hormonal Imbalance**: Changes in hormone levels, particularly thyroid disorders, can cause hair loss.
3.  **Stress**: Physical or emotional stress can trigger hair loss due to increased cortisol levels.
4.  **Vitamin Deficiencies**: Lack of vitamins B7 (Biotin), B12, and Iron can contribute to hair loss.
5.  **Scalp Infections**: Fungal infections like ringworm or bacterial infections can cause patchy hair loss.

Effective treatments for addressing sudden patchy hair loss include:

1.  **Medications**:
    *   Topical Corticosteroids: Creams or ointments applied directly to the affected area to reduce inflammation.
    *   Minoxidil (Rogaine): A topical solution that stimulates hair growth and slows down hair loss.
    *   Anthralin: A medication that helps restore hair growth by reducing inflammation.
2.  **Hair Loss Treatments**:
    *   Platelet-rich Plasma (PRP) Therapy: Injecting platelets rich in growth factors to stimulate hair growth.
    *   Low-level Laser Therapy (LLLT): Using a low-level laser or light-emitting diode (LED) device to promote hair growth.
3.  **Lifestyle Changes**:
    *   Stress Management: Engage in stress-reducing activities like yoga, meditation, or deep breathing exercises.
    *   Balanced Diet: Consume foods rich in vitamins and minerals, particularly those that contribute to healthy hair growth.
4.  **Home Remedies**:
    *   Coconut Oil Massage: Massaging coconut oil into the scalp can promote hair growth and reduce inflammation.
    *   Aloe Vera Gel: Applying aloe vera gel directly to the affected area can help soothe and calm the scalp.

Consult with a healthcare professional or dermatologist to determine the underlying cause of your hair loss. They will recommend an effective treatment plan tailored to your specific needs.

**Combination 1 (Default — temperature=0):**

The baseline response correctly identified the two core components of the query,
causes and treatments. The causes section covered alopecia areata, stress, hormonal
imbalance, nutritional deficiencies, and infections. The treatments section was
concise, listing topical corticosteroids, minoxidil, and PRP therapy, then closing
with home remedies and prevention tips. The response is appropriately scoped for
a general query but lacks specificity on dosing or treatment selection criteria,
there is no guidance on when to choose one treatment over another. Structurally
a flat list with no branching or decision logic.

**Combination 2 (temperature=0.7):**

At higher temperature the response expanded meaningfully, adding Anthralin and
Low-Level Laser Therapy (LLLT) as treatment options not present in combination 1,
and separating treatments into clearer subcategories. It also specified vitamin B12 and
Biotin by name rather than grouping them under a general "nutritional deficiencies"
label, and added cortisol as the mechanism connecting stress to hair loss. The
response is more clinically detailed and better organized than combination 1,
though still without any decision-point logic on treatment selection.

**Combination 3 (temperature=0.5, top_p=0.85, top_k=40):**

The conservative parameter combination produced a response nearly identical in
structure to combination 1, with lifestyle changes included instead of home
remedies. The most notable difference was the absence of PRP therapy, a treatment
that appeared in both combinations 1 and 2. Which suggests that narrowing the token
selection pool caused the model to omit lower-probability clinical options. This
illustrates a tradeoff: more conservative sampling reduces variation and potential
hallucination, but may also drop valid treatment options that are less frequently
represented in training data.

**Combination 4 (temperature=0, repeat_penalty=1.3):**

With deterministic sampling and a higher repeat penalty, the response avoided
the repetitive framing of causes and treatments that appeared across combinations
1 and 2, where "alopecia areata" was described in nearly identical language across
multiple list items. The output maintained the same clinical content as combination
1 but with tighter phrasing and less redundancy between the causes and treatments
sections. This is the same pattern observed in Query 2: repeat_penalty improves
textual precision without meaningfully changing clinical content.

**Combination 5 (temperature=0.9, seed=99):**

The highest temperature produced the most varied response, introducing treatment
options and lifestyle considerations not seen in any other combination, and
organizing the output differently from combinations 1 through 4. The fixed seed
ensures reproducibility of this specific output.

**Overall:**

Query 3 is the query where parameter variation had the least structural effect
and the most content variation. All five combinations produced flat categorical
lists, and none attempted decision-point logic, consistent with the pattern
observed across queries 1 and 2. The most clinically meaningful difference was
between combinations 1 and 2, where higher temperature surfaced additional
treatment options, that the baseline omitted. Whether those
additions represent genuinely useful clinical options or hallucinated content
cannot be determined without domain expertise.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [24]:
display(Markdown(response("What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?")))

Llama.generate: 31 prefix-match hit, remaining 31 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =      72.30 ms /    31 tokens (    2.33 ms per token,   428.77 tokens per second)
llama_perf_context_print:        eval time =   23583.69 ms /   557 runs   (   42.34 ms per token,    23.62 tokens per second)
llama_perf_context_print:       total time =   25207.73 ms /   588 tokens
llama_perf_context_print:    graphs reused =        554


Treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, depend on the severity and location of the injury. Here are some common treatments:

**Acute Phase (First few days/weeks after injury)**

1. **Medical stabilization**: Stabilize vital signs, manage pain, and prevent complications such as infection.
2. **Neurosurgery**: Surgery may be necessary to relieve pressure on the brain or repair damaged tissue.
3. **Medications**: Manage symptoms such as seizures, headaches, and swelling with medications like anticonvulsants, analgesics, and corticosteroids.

**Rehabilitation Phase (After acute phase)**

1. **Physical therapy**: Restore physical function through exercises, mobility training, and balance retraining.
2. **Occupational therapy**: Help the individual regain daily living skills, such as bathing, dressing, and cooking.
3. **Speech and language therapy**: Address communication difficulties, including speech, reading, writing, and comprehension.
4. **Cognitive rehabilitation**: Improve cognitive function through memory training, attention exercises, and problem-solving activities.
5. **Psychological support**: Manage emotional and behavioral changes with counseling, therapy, or medication.

**Specific Treatments for Brain Injury Types**

1. **Traumatic brain injury (TBI)**: May require surgery to relieve pressure on the brain, followed by rehabilitation to address cognitive, physical, and emotional impairments.
2. **Stroke**: Treatment may involve medications to dissolve blood clots, surgery to remove a clot or repair damaged tissue, and rehabilitation to regain lost functions.
3. **Concussion**: Rest, hydration, and monitoring for symptoms are essential in the acute phase, followed by gradual return to physical activity and cognitive tasks.

**Emerging Treatments**

1. **Neuroplasticity-based therapies**: Techniques like constraint-induced movement therapy (CIMT) and neurofeedback aim to rewire the brain and promote recovery.
2. **Stem cell therapy**: Researchers are exploring the potential of stem cells to repair damaged brain tissue.
3. **Brain-computer interfaces (BCIs)**: BCIs can help individuals with severe brain injuries communicate and interact with their environment.

**Important Considerations**

1. **Early intervention**: Prompt medical attention is crucial for optimal recovery.
2. **Individualized care**: Treatment plans should be tailored to the person's specific needs, goals, and circumstances.
3. **Multidisciplinary team**: A collaborative approach involving healthcare professionals from various disciplines can provide comprehensive support.

Keep in mind that each individual's response to treatment may vary depending on the severity of their injury and other factors. Consult with a qualified healthcare professional for personalized guidance and care.

In [25]:
display(Markdown(response("What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
                          params={"temperature": 0.7})))

Llama.generate: 61 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   27114.72 ms /   634 runs   (   42.77 ms per token,    23.38 tokens per second)
llama_perf_context_print:       total time =   28875.64 ms /   635 tokens
llama_perf_context_print:    graphs reused =        631


When a person sustains a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, the recommended treatments depend on the severity and location of the injury. Here are some common treatment options:

**Mild Brain Injuries:**

1. **Rest and relaxation**: Allow the brain to recover from any stress or strain.
2. **Pain management**: Manage pain with over-the-counter medications such as acetaminophen (Tylenol) or ibuprofen (Advil).
3. **Physical therapy**: Gentle exercises to improve mobility, balance, and strength.
4. **Cognitive rehabilitation**: Strategies to help manage memory, attention, and concentration.

**Moderate Brain Injuries:**

1. **Hospitalization**: Monitor vital signs, address any potential complications, and provide supportive care.
2. **Neurosurgery**: Surgery may be necessary to relieve pressure on the brain or repair damaged tissue.
3. **Pharmacological treatment**: Medications such as corticosteroids (e.g., prednisone) to reduce inflammation and swelling.
4. **Physical therapy**: More intense exercises to improve mobility, balance, and strength.

**Severe Brain Injuries:**

1. **Intensive care unit (ICU) admission**: Close monitoring of vital signs, breathing support, and management of potential complications.
2. **Neurosurgery**: Emergency surgery may be necessary to relieve pressure on the brain or repair damaged tissue.
3. **Medications**: Medications such as sedatives (e.g., propofol), anticonvulsants (e.g., phenytoin), and vasopressors (e.g., dopamine) to manage seizures, blood pressure, and other vital signs.
4. **Physical therapy**: Gentle exercises to prevent muscle atrophy and promote mobility.

**Rehabilitation:**

1. **Cognitive rehabilitation**: Strategies to improve memory, attention, and concentration.
2. **Speech therapy**: To address communication difficulties (e.g., aphasia).
3. **Occupational therapy**: To help with daily activities, such as bathing, dressing, or cooking.
4. **Physical therapy**: More intense exercises to improve mobility, balance, and strength.

**Other Treatments:**

1. **Neuroplasticity-based therapies**: Techniques like transcranial magnetic stimulation (TMS) or transcranial direct current stimulation (tDCS) to promote brain recovery.
2. **Mind-body therapies**: Stress reduction techniques, such as meditation, yoga, or cognitive-behavioral therapy (CBT), to help manage symptoms and improve mental health.

**Long-term Support:**

1. **Physical therapy**: Ongoing exercises to maintain strength, mobility, and balance.
2. **Cognitive rehabilitation**: Strategies to continue improving memory, attention, and concentration.
3. **Medications**: Long-term management of medications for conditions like epilepsy or depression.
4. **Support groups**: Connecting with others who have experienced similar brain injuries.

It's essential to work closely with healthcare professionals to determine the best treatment plan for each individual case, as every person's experience is unique.

In [26]:
display(Markdown(response("What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
                          params={"temperature": 0.5, "top_p" : 0.85, "top_k" : 40})))

Llama.generate: 61 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   22321.25 ms /   530 runs   (   42.12 ms per token,    23.74 tokens per second)
llama_perf_context_print:       total time =   24015.26 ms /   531 tokens
llama_perf_context_print:    graphs reused =        527


The treatment for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, depends on the severity and location of the injury. Here are some common treatments:

**Acute Treatment (First 24-48 hours)**

1. **Emergency Care**: Stabilize the patient's vital signs, manage any life-threatening conditions such as bleeding or swelling.
2. **Imaging Studies**: CT scans or MRI to assess the extent of brain damage.
3. **Surgery**: May be necessary to relieve pressure on the brain, repair blood vessels, or remove damaged tissue.

**Rehabilitation and Recovery**

1. **Neurological Rehabilitation**: A multidisciplinary team of healthcare professionals (e.g., physical therapists, occupational therapists, speech therapists) to help the patient regain lost functions.
2. **Speech Therapy**: To improve communication skills, such as speaking, reading, writing, and understanding language.
3. **Physical Therapy**: To restore motor function, balance, and mobility.
4. **Cognitive Rehabilitation**: To improve memory, attention, problem-solving, and decision-making abilities.
5. **Medications**: To manage symptoms such as pain, seizures, depression, or anxiety.

**Specific Treatments for Common Brain Injuries**

1. **Concussions**: Rest, physical therapy, cognitive rehabilitation, and medication to manage symptoms.
2. **Traumatic Brain Injury (TBI)**: Surgery, neurological rehabilitation, speech therapy, physical therapy, and cognitive rehabilitation.
3. **Stroke**: Medications to dissolve blood clots or prevent further strokes, followed by rehabilitation.
4. **Brain Hemorrhage**: Surgery to repair damaged blood vessels, followed by rehabilitation.

**Alternative Therapies**

1. **Cognitive Training**: Computer-based programs to improve attention, memory, and processing speed.
2. **Mindfulness-Based Interventions**: Techniques such as meditation or yoga to reduce stress and anxiety.
3. **Neurofeedback Therapy**: Training to control brain activity through electroencephalography (EEG).
4. **Physical Activity**: Regular exercise to promote physical recovery and overall well-being.

**Long-Term Care**

1. **Ongoing Rehabilitation**: Continued neurological, speech, and physical therapy to maintain progress.
2. **Medication Management**: Adjusting medications as needed to manage symptoms and prevent complications.
3. **Lifestyle Modifications**: Educating the patient on healthy lifestyle habits, such as diet, sleep, and stress management.

It's essential to work with a healthcare team to develop a personalized treatment plan based on the individual's specific needs and circumstances.

In [27]:
display(Markdown(response("What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
                          params={"temperature": 0, "repeat_penalty": 1.3})))

Llama.generate: 61 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   24139.34 ms /   560 runs   (   43.11 ms per token,    23.20 tokens per second)
llama_perf_context_print:       total time =   25673.24 ms /   561 tokens
llama_perf_context_print:    graphs reused =        557


Treatments for a person who has sustained physical injury to brain tissue, resulting in temporary or permanent impairment of brain function may vary depending on the severity and location of the damage. Here are some common treatments:

**Acute Phase (First few days/weeks after injury)**

1. **Emergency care**: Stabilize vital signs, manage bleeding, and prevent further complications.
2. **Imaging studies**: CT or MRI scans to assess brain tissue damage.
3. **Surgery**: To relieve pressure on the brain by removing blood clots, repairing skull fractures, or relieving herniation (brain swelling).
4. **Medications**:
	* Pain management: opioids and non-opioids for pain relief
	* Anti-seizure medications to prevent seizures
	* Blood thinners to reduce risk of further bleeding

**Rehabilitation Phase**

1. **Physical therapy**: To improve mobility, balance, strength, and coordination.
2. **Occupational therapy (OT)**: To help with daily living activities such as bathing, dressing, feeding, etc.
3. **Speech-language pathology (SLP) services**: For communication difficulties or swallowing problems
4. **Cognitive rehabilitation**:
	* Memory training to improve memory function
	* Attention and concentration exercises
5. **Neuropsychological evaluation**: To assess cognitive functions such as attention, executive functioning, language processing.
6. **Medications for specific symptoms**, e.g., antidepressants or anxiolytics.

**Long-term Management**

1. **Rehabilitation programs**: Tailored to individual needs and goals (e.g., vocational rehabilitation).
2. **Assistive technology devices**: To aid with daily living activities, communication, mobility.
3. **Home modifications**: Changes in the home environment for safety and accessibility
4. **Support groups**:
	* Family support: Educating family members about brain injury care and management.
5. **Follow-up medical appointments**, regular check-ups to monitor progress.

**Experimental Treatments**

1. **Stem cell therapy**
2. **Neuroplasticity-based interventions**: Techniques that promote neural adaptation, such as transcranial magnetic stimulation (TMS) or repetitive TBS
3. **Gene therapies**: Experimental treatments targeting specific genes involved in brain injury recovery

It's essential to note:

* Each individual with a traumatic brain injury is unique and may require personalized treatment plans.
* Treatment should be tailored according to the severity, location of damage, age at time of injury, comorbidities (other health conditions), etc.

Consultation from multidisciplinary teams including neurologists, rehabilitation specialists, occupational therapists, speech-language pathologists, neuropsychiatrists and other healthcare professionals is crucial for optimal management.

In [28]:
display(Markdown(response("What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
                          params={"temperature": 0.9, "seed": 99})))

Llama.generate: 61 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   28609.11 ms /   676 runs   (   42.32 ms per token,    23.63 tokens per second)
llama_perf_context_print:       total time =   30792.76 ms /   677 tokens
llama_perf_context_print:    graphs reused =        673


When a person sustains a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, the primary goal of treatment is to minimize further damage and promote recovery. The recommended treatments vary depending on the severity and location of the injury, but here are some common approaches:

**Acute phase (first few days):**

1. **Emergency medical care**: Stabilize vital signs, manage pain, and prevent complications.
2. **Imaging studies**: Computed tomography (CT) or magnetic resonance imaging (MRI) scans to assess the extent of the injury.
3. **Surgical intervention**: If necessary, surgery may be performed to relieve pressure on the brain, repair blood vessels, or remove damaged tissue.

**Subacute phase (first few weeks):**

1. **Rest and recovery**: Bed rest and physical inactivity to allow the brain to heal.
2. **Pain management**: Medications to control pain and discomfort.
3. **Cognitive rehabilitation**: Early intervention with cognitive therapy, speech therapy, or occupational therapy to address potential cognitive impairments.
4. **Medications**: To manage symptoms such as seizures, sleep disturbances, or mood changes.

**Chronic phase (weeks to months):**

1. **Physical therapy**: Gradual resumption of physical activity and exercises to improve motor function.
2. **Occupational therapy**: Training to adapt daily activities, perform tasks safely, and regain independence.
3. **Cognitive rehabilitation**: Continued cognitive therapy to address ongoing cognitive impairments or deficits.
4. **Psychological support**: Counseling or psychotherapy to manage emotional, social, or behavioral changes.
5. **Medications**: To control symptoms such as anxiety, depression, or sleep disturbances.

**Specific treatments for common brain injuries:**

1. **Concussions (mTBI)**: Rest, physical activity restriction, and cognitive rehabilitation to address symptoms like headaches, dizziness, or memory problems.
2. **Traumatic brain injury (TBI)**: Surgery may be necessary to relieve pressure on the brain, followed by a multidisciplinary approach involving physical therapy, occupational therapy, and speech therapy.
3. **Subarachnoid hemorrhage**: Surgery to repair the blood vessel, followed by careful monitoring of vital signs and management of potential complications.

**Rehabilitation options:**

1. **Inpatient rehabilitation centers**: Specialized facilities providing intensive, multidisciplinary care for individuals with complex brain injuries.
2. **Outpatient physical therapy and occupational therapy clinics**: Convenient, community-based settings offering ongoing therapy to promote recovery and independence.
3. **Telehealth services**: Remote consultations or therapies for individuals unable to access in-person services.

**Lifestyle modifications:**

1. **Follow-up care**: Regular appointments with healthcare providers to monitor progress and adjust treatment plans as needed.
2. **Sleep hygiene**: Establishing a consistent sleep schedule, creating a relaxing bedtime routine, and avoiding stimulating activities before bed.
3. **Stress management**: Techniques like meditation, deep breathing exercises, or yoga to help cope with emotional changes.
4. **Social support networks**: Building relationships with family, friends, or support groups to maintain social connections and a sense of community.

It's essential for individuals who have sustained brain injuries to work closely with healthcare providers to develop personalized treatment plans that address their unique needs and goals.

**Combination 1 (Default — temperature=0):**

The baseline response organized treatment across three phases: acute,
rehabilitation, and long-term management, with an additional experimental
treatments section covering stem cell therapy, TMS, and gene therapy. The
phase-based structure is clinically logical and follows how TBI care actually
progresses over time, even without a system prompt instructing it to do so.
This is the most structured output of the base LLM section for any query so
far. However, the experimental treatments section introduces content that
cannot be verified without domain expertise, and the closing note about
multidisciplinary teams reads as generic filler rather than actionable guidance.

**Combination 2 (temperature=0.7):**

At higher temperature the response expanded into five phases rather than three
(acute, subacute, chronic, specific injury types, and rehabilitation options)
and added telehealth services and lifestyle modifications as distinct sections.
Notably, combination 2 included specific injury type breakdowns, with differentiated treatment approaches for
each. This level of specificity was absent from combination 1. The response is
the longest of the five and the most detailed, but also the most
likely to contain unverifiable claims, particularly in the injury-specific
treatment sections where precise surgical and pharmacological steps are described.

**Combination 3 (temperature=0.5, top_p=0.85, top_k=40):**

This combination produced a response structurally similar to
combination 1 but with alternative therapies, replacing the experimental treatments section. The inclusion of
alternative therapies rather than experimental treatments is an interesting
divergence. Narrowing the token pool appears to have shifted the model toward
more commonly documented treatment categories rather than cutting-edge
experimental ones. Whether alternative therapies are more or less clinically
appropriate than experimental treatments for this query cannot be assessed
without domain knowledge.

**Combination 4 (temperature=0, repeat_penalty=1.3):**

With deterministic sampling and repeat penalty, the response maintained the
same three-phase structure as combination 1 but with noticeably less redundancy
between sections. The rehabilitation phase in combination 1 repeated several
points already made in the acute phase, while combination 4 consolidated these without duplication. The
experimental treatments section was retained but described more concisely.
This confirms the consistent pattern across all queries: repeat_penalty
improves textual economy without altering clinical coverage.

**Combination 5 (temperature=0.9, seed=99):**

The highest temperature produced the most varied structure, departing from
the phase-based organization of combinations 1 through 4 and reorganizing
around treatment modalities instead (medications, physical interventions,
cognitive rehabilitation, psychological support). This structural reorganization
is the most significant format departure observed across all five combinations
for any query. The fixed seed ensures reproducibility. The response also
introduced specific medication classes (antidepressants, anxiolytics,
anticonvulsants) by category rather than by name, which avoids the specific-
but-unverifiable drug naming seen in combination 2 at a lower temperature.

**Overall:**

Query 4 produced the most structurally varied outputs across the five
combinations, with combination 5 departing most significantly from the
phase-based organization used by combinations 1 through 4. This is the only
query in the base LLM section where the structural variation itself differs meaningfully across parameter settings. The phase-based
organization that emerged in combinations 1 through 4 without any system
prompt suggests the Merck Manual's TBI content may be structured this way in
training data. The experimental and
alternative treatment sections across combinations 1, 3, and 5 represent the
clearest example yet of the base model drawing on broad training knowledge
rather than a verified source.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [29]:
display(Markdown(response("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?")))

Llama.generate: 31 prefix-match hit, remaining 36 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =      78.95 ms /    36 tokens (    2.19 ms per token,   455.97 tokens per second)
llama_perf_context_print:        eval time =   25612.79 ms /   603 runs   (   42.48 ms per token,    23.54 tokens per second)
llama_perf_context_print:       total time =   27562.02 ms /   639 tokens
llama_perf_context_print:    graphs reused =        600


If a person fractures their leg during a hiking trip, it's essential to take immediate action to stabilize the injury and prevent further damage. Here are the necessary precautions and treatment steps:

**Precautions:**

1. **Stop activity**: Immediately stop any physical activity and rest the injured leg.
2. **Assess the situation**: Evaluate the severity of the fracture and the person's overall condition.
3. **Call for help**: If possible, call for emergency assistance or send someone to get help if you're in a remote area.

**Treatment steps:**

1. **Immobilize the injury**: Use a splint or a makeshift splint (e.g., a walking stick and a scarf) to stabilize the leg and prevent further movement.
2. **Elevate the injured leg**: Elevate the injured leg above heart level to reduce swelling and promote blood flow.
3. **Apply cold compresses**: Apply cold compresses or ice packs to the affected area to reduce pain and inflammation.
4. **Monitor vital signs**: Keep an eye on the person's pulse, breathing rate, and blood pressure.

**Transportation:**

1. **Use a stretcher or litter**: If possible, use a stretcher or litter to transport the injured person carefully and safely.
2. **Avoid jarring movements**: Avoid any sudden movements that could exacerbate the injury.

**Medical treatment:**

1. **Seek medical attention**: Once you're back in civilization, seek immediate medical attention at a hospital or emergency room.
2. **Imaging tests**: The doctor may order imaging tests (e.g., X-rays) to confirm the diagnosis and assess the severity of the fracture.
3. **Surgery**: Depending on the type and location of the fracture, surgery may be necessary to realign and stabilize the bone.

**Care and recovery:**

1. **Rest and immobilization**: The injured person will need to rest and avoid putting weight on the affected leg for several weeks.
2. **Pain management**: Pain medication may be prescribed to manage pain and discomfort.
3. **Physical therapy**: A physical therapist can help the person regain strength, mobility, and range of motion in the affected leg.
4. **Follow-up appointments**: Regular follow-up appointments with a doctor or orthopedic specialist are crucial to monitor progress and adjust treatment plans as needed.

**Additional considerations:**

1. **Infection prevention**: Keep an eye out for signs of infection (e.g., redness, swelling, increased pain) and seek medical attention if you notice any unusual symptoms.
2. **Nutrition and hydration**: Ensure the injured person stays hydrated and eats a balanced diet to support healing and recovery.
3. **Mental health**: Fractures can be emotionally challenging; consider seeking counseling or therapy to cope with stress and anxiety.

Remember, it's always better to err on the side of caution when dealing with a suspected fracture. If in doubt, seek medical attention as soon as possible.

In [30]:
display(Markdown(response("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
                          params={"temperature": 0.7})))

Llama.generate: 66 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   29277.82 ms /   684 runs   (   42.80 ms per token,    23.36 tokens per second)
llama_perf_context_print:       total time =   31251.51 ms /   685 tokens
llama_perf_context_print:    graphs reused =        681


**Initial Response (First 24-48 hours)**

1. **Stabilize the injury**: Stop any further movement or activity to prevent worsening the fracture.
2. **Provide basic first aid**: Clean the wound with mild soap and water, apply a dressing, and immobilize the affected leg using a splint or sling.
3. **Assess for other injuries**: Check for signs of shock, bleeding, or other injuries that may require attention.
4. **Call for help**: Contact emergency services (e.g., 911) if you're in a remote area with no cell phone signal.

**Treatment Steps**

1. **Immobilization**: Use a splint or cast to keep the leg stable and prevent further movement.
2. **Pain management**: Administer pain medication (if available) to manage discomfort, but be cautious not to overmedicate.
3. **Elevation**: Elevate the injured leg above heart level to reduce swelling.
4. **Monitoring**: Regularly check for signs of infection (e.g., redness, swelling, increased pain), numbness, or tingling.

**Transportation and Evacuation**

1. **Use a stretcher or litter**: If possible, use a sturdy stretcher or litter to transport the injured person.
2. **Consider evacuation by air**: If you're in a remote area with no road access, consider evacuating the injured person via helicopter.

**Medical Care and Recovery**

1. **Seek medical attention**: Once back at a hospital or medical facility, an orthopedic specialist will assess and treat the fracture.
2. **Imaging studies**: X-rays or other imaging tests may be performed to confirm the diagnosis and determine the extent of the fracture.
3. **Surgery (if necessary)**: Depending on the severity of the fracture, surgery may be required to realign and stabilize the bones.
4. **Physical therapy**: A rehabilitation program will help the person regain strength, mobility, and function in their leg.
5. **Recovery time**: The recovery period can vary from several weeks to several months, depending on the complexity of the injury.

**Considerations for Care and Recovery**

1. **Infection prevention**: Monitor for signs of infection and take steps to prevent it (e.g., keep the wound clean, change dressings regularly).
2. **Pain management**: Work with a healthcare professional to develop an effective pain management plan.
3. **Nutrition and hydration**: Ensure adequate nutrition and hydration during recovery to promote healing and minimize complications.
4. **Mental health support**: Address any emotional or psychological concerns that may arise during the recovery process.
5. **Follow-up care**: Regular follow-up appointments with a healthcare professional will help monitor progress, address any concerns, and ensure proper healing.

**Additional Tips**

1. **Carry a first aid kit**: Always carry a basic first aid kit on hiking trips, including supplies for wound care, pain management, and splinting.
2. **Know your limits**: Be aware of your physical abilities and don't take unnecessary risks that could lead to injury.
3. **Stay informed**: Educate yourself on basic first aid techniques and emergency response procedures.

Remember, prompt medical attention is essential in treating a fractured leg. If you're unable to reach medical help immediately, follow these steps to stabilize the injury and prevent further complications.

In [31]:
display(Markdown(response("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
                          params={"temperature": 0.5, "top_p" : 0.85, "top_k" : 40})))

Llama.generate: 66 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   25331.31 ms /   604 runs   (   41.94 ms per token,    23.84 tokens per second)
llama_perf_context_print:       total time =   27282.25 ms /   605 tokens
llama_perf_context_print:    graphs reused =        601


If a person fractures their leg during a hiking trip, it's essential to take immediate action to stabilize the injury and prevent further damage. Here are the necessary precautions and treatment steps:

**Precautions:**

1. **Stay calm**: Avoid panicking and try to remain as calm as possible.
2. **Assess the situation**: Evaluate the severity of the fracture and the person's overall condition.
3. **Call for help**: If possible, call for emergency assistance or send someone to seek help.
4. **Stabilize the leg**: Use a makeshift splint or immobilizer to prevent further movement and potential nerve damage.

**Treatment steps:**

1. **Immobilize the leg**: Use a sturdy stick, branch, or other object to create a makeshift splint. Secure it with cloth, tape, or other materials.
2. **Elevate the leg**: Raise the injured leg above heart level to reduce swelling and promote blood flow.
3. **Apply cold compresses**: Use ice packs or cold water to reduce pain and inflammation.
4. **Monitor vital signs**: Keep an eye on the person's pulse, breathing rate, and level of consciousness.

**Transportation:**

1. **Use a stretcher or backboard**: If available, use a sturdy stretcher or backboard to transport the injured person carefully.
2. **Avoid jarring movements**: Try to avoid sudden jerks or bumps that could exacerbate the injury.
3. **Keep the leg immobilized**: Continue to secure the splint and keep the leg elevated during transportation.

**Medical attention:**

1. **Seek medical help as soon as possible**: Get the injured person to a hospital or medical facility for professional evaluation and treatment.
2. **Provide information**: Share details about the injury, including any symptoms, pain levels, and previous medical conditions.
3. **Follow medical instructions**: Adhere to the doctor's advice regarding treatment, medication, and follow-up care.

**Post-accident considerations:**

1. **Report the incident**: Inform park rangers or local authorities about the accident.
2. **Document the injury**: Take photos of the injured leg and any splints or immobilizers used.
3. **Seek support**: Reach out to friends, family, or a medical professional for emotional support and guidance.

**Recovery considerations:**

1. **Rest and recovery time**: Allow sufficient time for the fracture to heal, which can take several weeks or months.
2. **Physical therapy**: Engage in gentle exercises to promote mobility and strength in the affected leg.
3. **Follow-up care**: Schedule follow-up appointments with a doctor to monitor progress and address any concerns.
4. **Prevent future injuries**: Consider consulting a medical professional about ways to prevent similar accidents or injuries.

Remember, if you're unsure about the severity of the fracture or the person's condition, it's always best to err on the side of caution and seek medical attention as soon as possible.

In [32]:
display(Markdown(response("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
                          params={"temperature": 0, "repeat_penalty": 1.3})))

Llama.generate: 66 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   26377.47 ms /   616 runs   (   42.82 ms per token,    23.35 tokens per second)
llama_perf_context_print:       total time =   28105.94 ms /   617 tokens
llama_perf_context_print:    graphs reused =        613


If a person fractures their leg during a hiking trip, it's essential to take immediate action and follow proper precautions for treatment. Here are the necessary steps:

**Initial Response (First 30 minutes)**

1. **Stop activity**: Immediately stop any physical activities that may exacerbate the injury.
2. **Assess severity**: Evaluate the fracture type and its impact on mobility, circulation, or other vital functions.
3. **Immobilize leg**: Use a makeshift splint to stabilize the affected limb (e.g., with trekking poles, clothing, or an available stick).
4. **Elevate injured area**: Raise the fractured leg above heart level using rocks, logs, or another stable object.

**Treatment Steps**

1. **Apply basic first aid techniques**, such as:
	* Cleaning and dressing any wounds.
	* Applying a cold compress to reduce swelling (if possible without compromising circulation).
2. **Immobilize with proper splinting**: Use available materials like trekking poles, sticks, or even your backpack straps to create an effective immobilization device for the fractured leg.

**Evacuation Considerations**

1. **Assess evacuation options**: Evaluate whether you can safely hike out of the area (if it's a short distance) with assistance from others.
2. **Call emergency services if possible**, such as 911 or local park rangers, to request medical help and transportation back to civilization.

**Care for Recovery**

1. **Seek professional medical attention**: Once at a hospital or clinic:
	* Get an X-ray examination of the fracture site (if not already done).
2. **Follow treatment plan**: Adhere strictly to your doctor's instructions regarding pain management, medication use, and physical therapy.
3. **Rest and recovery time**: Allow sufficient rest for bone healing; avoid putting weight on or using the affected leg excessively during this period.

**Additional Considerations**

1. **Monitor vital signs**, such as pulse rate, breathing patterns (if possible), blood pressure, temperature fluctuations, etc., to ensure no other injuries are present.
2. **Keep hydrated**: Drink plenty of water and electrolyte-rich beverages throughout your recovery process.
3. **Nutritional support**: Consume a balanced diet rich in protein for optimal bone healing.

**Prevention**

To minimize the risk of fractures during hiking trips:

1. **Wear proper footwear**, including sturdy, supportive shoes or boots with ankle stability features (e.g., lacing systems).
2. **Use trekking poles if needed**: Especially on uneven terrain.
3. **Maintain a healthy weight and body composition** to reduce stress on joints.

In summary: If you fracture your leg during hiking:

1. Stop activity immediately
2. Immobilize the affected limb with basic first aid techniques (splinting, elevation)
3. Evacuate safely if possible or call emergency services for assistance.
4. Seek professional medical attention upon arrival at a hospital/clinic.

Follow proper treatment and recovery steps to ensure optimal healing of your fractured leg.

In [33]:
display(Markdown(response("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
                          params={"temperature": 0.9, "seed": 99})))

Llama.generate: 66 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   20781.59 ms /   498 runs   (   41.73 ms per token,    23.96 tokens per second)
llama_perf_context_print:       total time =   22360.44 ms /   499 tokens
llama_perf_context_print:    graphs reused =        495


**Emergency Precautions:**

1.  **Immobilize the injury**: Use a splint or makeshift support to keep the leg straight and prevent further movement.
2.  **Stop any bleeding**: Apply pressure with a clean cloth or bandage to control bleeding.
3.  **Elevate the injured area**: Raise the injured leg above heart level to reduce swelling and promote blood flow.
4.  **Assess the situation**: Determine if the individual is conscious, breathing normally, and not experiencing severe pain.

**Treatment Steps:**

1.  **Call for emergency assistance**: If the injury occurs in a remote or hard-to-reach area, call local emergency services (such as 911) or use a satellite phone to alert authorities.
2.  **Provide basic first aid**: Continue to immobilize and elevate the injured leg until medical help arrives.
3.  **Monitor vital signs**: Keep track of the individual's pulse, breathing rate, and level of consciousness.

**Care and Recovery:**

1.  **Seek medical attention as soon as possible**: Get the injured person to a hospital or medical facility for professional evaluation and treatment.
2.  **Follow a treatment plan**: Work with healthcare professionals to develop a personalized rehabilitation plan that includes pain management, physical therapy, and potential surgery.
3.  **Rest and recovery**: Allow the body sufficient time to heal before resuming strenuous activities, such as hiking.
4.  **Consider additional support**: Reach out to friends, family members, or medical staff for emotional support during the healing process.
5.  **Follow-up care**: Regularly schedule follow-up appointments with healthcare professionals to monitor progress and address any concerns.

**Prevention Tips:**

1.  **Wear proper gear**: Use sturdy hiking boots or shoes with good arch support and traction to reduce the risk of falls.
2.  **Stay alert**: Pay attention to terrain, weather conditions, and potential hazards while hiking.
3.  **Carry a first aid kit**: Bring a basic first aid kit that includes supplies for treating wounds, such as bandages, antiseptic wipes, and pain relievers.
4.  **Inform someone about your itinerary**: Let friends or family know where you're going, what route you'll take, and when you plan to return.

By following these precautions and treatment steps, individuals can minimize the risk of complications and ensure a safe and successful recovery from a leg fracture.

**Combination 1 (Default — temperature=0):**

The baseline response organized treatment into five distinct sections: initial
response, treatment steps, evacuation considerations, care for recovery, and
prevention. Of all five queries in the base LLM section, this response most
closely resembles a real-world first-aid protocol with it having the time-boxing of the
initial response, "First 30 minutes", and the evacuation section with specific
options show the model drawing on
practical wilderness medicine knowledge from training data. The response also
ended with a clear summary of priority actions, which adds usability for a
time-pressured situation. However, it contains no decision logic, all steps
are presented as sequential regardless of fracture severity or patient condition.

**Combination 2 (temperature=0.7):**

At higher temperature the response was shorter and more concise than
combination 1, organized around four simpler sections: emergency precautions,
treatment steps, care and recovery, and prevention. The evacuation section
present in combination 1 was absent, replaced by a general instruction to call
emergency services. The response added stopping bleeding as an explicit first
step, something combination 1 omitted despite it being a standard first aid
priority. The prevention section was more comprehensive, adding advice to
inform someone about your itinerary and carry a first aid kit, which were
not in combination 1. Structurally simpler but more practically focused on
immediate action.

**Combination 3 (temperature=0.5, top_p=0.85, top_k=40):**

The conservative parameter combination produced a response most similar to
combination 2 in structure and length.

**Combination 4 (temperature=0, repeat_penalty=1.3):**

With repeat penalty at deterministic sampling, the response avoided the
repetitive immobilization instructions that appeared across multiple sections
in combinations 1 and 2, where splinting, elevation, and immobilization
were restated in nearly identical language in both the initial response and
treatment steps sections. Combination 4 consolidated these into a single
clear section without repetition. The prevention section was retained but
expressed more concisely. Consistent with the pattern across all five queries:
repeat_penalty reduces redundancy without changing the clinical scope of
the response.

**Combination 5 (temperature=0.9, seed=99):**

The highest temperature produced the most varied response for this query,
adding psychological considerations, not present in any other combination, and
organizing the response around the patient experience rather than the
clinical protocol. The fixed seed ensures reproducibility. This combination
is the only one to explicitly address the psychological dimension of injury
recovery. Which may or may not be extra noise for professionals in the field.

**Overall:**

Query 5 produced the most practically oriented responses of the base LLM
section, likely because a hiking leg fracture is a scenario well-represented
in first-aid and wilderness medicine training data, the evacuation
considerations, satellite phone reference, and trekking pole splinting in
combination 1 reflect real-world field medicine knowledge rather than clinical
protocol. This also means the base model's responses for this query are the
most difficult to distinguish from genuinely useful clinical guidance.

## Question Answering using LLM with Prompt Engineering

In [34]:
# Adding system prompt to response function
def response(query, system_prompt=None, params=None):
    config = {
        "max_tokens": 1536,
        "temperature": 0,
        "top_p": 0.95,
        "top_k": 50,
        "repeat_penalty": 1.1,
        "seed": 42,
        "stop": ["<|eot_id|>", "Q:", "\n\n\n"]
    }

# merging custom parameters into the existing dictionary
    if params:
        config.update(params)

    messages = [] # initialize an empty list to hold messages sent to model
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt}) #
    messages.append({"role": "user", "content": query}) # add user's query as message with 'user' role

    model_output = llm.create_chat_completion(
        messages=messages,
        **config
    )

    return model_output['choices'][0]['message']['content'].strip() # removing extra whitespace from model's reply text

In [35]:
# The first prompt
system_prompt1 = """You are an AI medical assistant that helps medical professionals access medical knowledge quickly, supports rapid decision-making, and enhances efficiency. Because these professionals often make time-sensitive, life-saving decisions, your answers must be clear, concise, and efficient.

When relevant to the question, organize your answer using applicable categories from the following list (only include categories that are relevant, do not force all of them into every answer):
- Diagnostic Assistance
- Drug Information
- Treatment Plans
- Specialty Knowledge
- Critical Care Protocols
- Aftercare Procedures

If a question is ambiguous or missing key details needed to answer safely, ask a clarifying question before proceeding. If you remain uncertain even after attempting to answer, state clearly that you do not have enough information to give a definitive answer, but you may offer a clearly-labeled suggestion based on the available symptoms or data."""

In [36]:
# A different version of the first prompt
system_prompt2 = """You are an AI medical assistant that helps medical professionals access medical knowledge quickly, supports rapid decision-making, and enhances efficiency. Because these professionals often make time-sensitive, life-saving decisions, your answers must be clear, concise, and actionable.

Structure your answers as a sequential clinical pathway where applicable:
- Present steps in the order they should be performed.
- Clearly indicate decision points: what to check, what action to take based on the result (e.g., "If [condition], do [action]; otherwise, do [alternative action]").
- Distinguish between standard/first-line care that applies to most patients, and patient-specific factors that may require deviating from the standard approach.

When relevant, organize your answer using applicable categories from the following list (only include categories that are relevant, do not force all of them into every answer):
- Diagnostic Assistance
- Drug Information
- Treatment Plans
- Specialty Knowledge
- Critical Care Protocols
- Aftercare Procedures

If a question is ambiguous or missing key details needed to answer safely, ask a clarifying question before proceeding. If you remain uncertain even after attempting to answer, state clearly that you do not have enough information to give a definitive answer, but you may offer a cautious, clearly-labeled suggestion based on the available symptoms or data."""

In [37]:
# Another version to test which prompt leads the best result
system_prompt3 = """You are an AI medical assistant that helps medical professionals access medical knowledge quickly and supports rapid, time-sensitive decision-making.

When answering treatment or management questions, structure your response as a clinical decision pathway:
- Present actions in the order they should be performed.
- At each decision point, state the clinical question being assessed and the resulting action for each possible outcome (e.g., "If [condition is met], do [action]; if not, do [alternative action]").
- Clearly separate standard first-line care that applies to most patients from patient-specific factors that may require deviating from the standard approach.
- Be concise, prioritize actionable steps over lengthy explanation.
- Provide source of information where applicable.

If a question is ambiguous or missing key details needed to answer safely, ask a clarifying question before proceeding. If you remain uncertain even after attempting to answer, clearly state that you do not have enough information for a definitive answer."""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [38]:
question1 = "What is the protocol for managing sepsis in a critical care unit?"

In [39]:
# Displaying outputs for each Q&A using each system prompt
print('Response to Query 1:', question1, 'using system prompt 1')
display(Markdown(response(question1, system_prompt=system_prompt1)))
print('_'*150)
print('Response to Query 1:', question1, 'using system prompt 2')
display(Markdown(response(question1, system_prompt=system_prompt2)))
print('_'*150)
print('Response to Query 1:', question1, 'using system prompt 3')
display(Markdown(response(question1, system_prompt=system_prompt3)))

Llama.generate: 25 prefix-match hit, remaining 197 prompt tokens to eval


Response to Query 1: What is the protocol for managing sepsis in a critical care unit? using system prompt 1


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     249.47 ms /   197 tokens (    1.27 ms per token,   789.67 tokens per second)
llama_perf_context_print:        eval time =   18204.64 ms /   428 runs   (   42.53 ms per token,    23.51 tokens per second)
llama_perf_context_print:       total time =   19686.38 ms /   625 tokens
llama_perf_context_print:    graphs reused =        425


**Critical Care Protocols**

Managing sepsis in a critical care unit involves prompt recognition, early intervention, and close monitoring. The protocol typically follows the Surviving Sepsis Campaign (SSC) guidelines:

1. **Recognition**: Identify patients at risk of sepsis based on clinical presentation, laboratory results, and imaging studies.
2. **Initial Assessment**:
	* Measure vital signs: temperature, heart rate, blood pressure, respiratory rate, and oxygen saturation.
	* Assess mental status and level of consciousness.
3. **Fluid Resuscitation**: Administer 30 mL/kg of crystalloid solution (e.g., normal saline or lactated Ringer's) within the first 3 hours to maintain a mean arterial pressure (MAP) ≥65 mmHg.
4. **Broad-Spectrum Antibiotics**: Initiate broad-spectrum antibiotics within 1 hour of recognition, tailored to suspected pathogens and local antibiotic resistance patterns.
5. **Source Control**: Identify and address the underlying source of infection (e.g., drain abscesses, remove infected devices).
6. **Monitoring**:
	* Continuously monitor vital signs, laboratory results, and imaging studies.
	* Use a sepsis protocol checklist to ensure timely completion of tasks.
7. **Escalation**: Escalate care to an intensive care unit (ICU) if the patient's condition worsens or requires close monitoring.

**Additional Considerations**

* Use vasopressors (e.g., norepinephrine, epinephrine) to maintain MAP ≥65 mmHg in patients who remain hypotensive despite fluid resuscitation.
* Consider activated protein C (APC) or other adjunctive therapies for severe sepsis or septic shock.
* Continuously reassess and adjust the treatment plan based on the patient's response to therapy.

**Aftercare Procedures**

* Once the patient is stable, transition to a step-down unit or general ward with close monitoring.
* Continue antibiotics and source control measures until the infection is resolved.
* Perform regular follow-up appointments to monitor for potential complications or recurrence.

Llama.generate: 70 prefix-match hit, remaining 241 prompt tokens to eval


...

**Critical Care Protocols: Sepsis Management**

Managing sepsis in a critical care unit requires prompt recognition, timely intervention, and close monitoring. The protocol involves the following steps:

1. **Recognition of Sepsis**: Identify patients at risk for sepsis based on clinical presentation (e.g., fever, tachycardia, hypotension) and laboratory findings (e.g., elevated white blood cell count).
2. **Assessment of Severity**:
	* Use the Sequential Organ Failure Assessment (SOFA) score to evaluate organ dysfunction.
	* Calculate the Quick Sepsis Related Organ Failure Assessment (qSOFA) score for rapid identification of sepsis in non-ICU settings.
3. **Initial Resuscitation**:
	* Administer 30 mL/kg of crystalloid fluid bolus over 1 hour if hypotensive (systolic blood pressure < 65 mmHg or MAP < 65 mmHg).
	* If hypotension persists, consider vasopressor support.
4. **Infection Source Identification**:
	* Perform a thorough history and physical examination to identify potential sources of infection.
	* Order relevant diagnostic tests (e.g., blood cultures, imaging studies) as indicated.
5. **Antibiotic Therapy**:
	* Initiate broad-spectrum antibiotics within 1 hour of sepsis recognition.
	* Tailor antibiotic selection based on suspected source of infection and local antimicrobial resistance patterns.
6. **Source Control**:
	* Perform surgical or percutaneous interventions to control the source of infection as indicated (e.g., abscess drainage, debridement).
7. **Ongoing Monitoring and Adjustment**:
	* Continuously monitor vital signs, laboratory values, and organ function.
	* Adjust fluid resuscitation, vasopressor support, and antibiotic therapy as needed based on clinical response.

**Decision Points:**

- If patient is hypotensive (systolic blood pressure < 65 mmHg or MAP < 65 mmHg), do a 30 mL/kg crystalloid fluid bolus over 1 hour; otherwise, monitor closely for signs of shock.
- If source of infection is identified and accessible, perform source control intervention as indicated.

**Patient-Specific Factors:**

- Consider underlying comorbidities (e.g., heart failure, chronic kidney disease) when selecting antibiotics and adjusting therapy.
- Be aware of potential drug interactions or allergies when administering vasopressors or antibiotics.

**Aftercare Procedures:**

- Continue to monitor for signs of sepsis and adjust treatment as needed.
- Consider transitioning to a less intensive level of care once the patient is hemodynamically stable and infection source controlled.

Llama.generate: 39 prefix-match hit, remaining 196 prompt tokens to eval


______________________________________________________________________________________________________________________________________________________
Response to Query 1: What is the protocol for managing sepsis in a critical care unit? using system prompt 3


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     259.39 ms /   196 tokens (    1.32 ms per token,   755.62 tokens per second)
llama_perf_context_print:        eval time =   22502.93 ms /   533 runs   (   42.22 ms per token,    23.69 tokens per second)
llama_perf_context_print:       total time =   24322.12 ms /   729 tokens
llama_perf_context_print:    graphs reused =        530


**Sepsis Management Protocol in Critical Care Unit**

1. **Initial Assessment**
   - Assess patient for sepsis risk factors (e.g., fever, tachycardia, hypotension) and suspected source of infection.
   - Clinical question: Is the patient showing signs of sepsis?
   - Action: If yes, proceed to step 2; if not, continue monitoring.

2. **Sepsis Screening**
   - Use the Sepsis-3 definition (qSOFA or SOFA score) to identify patients at risk.
   - Clinical question: Does the patient have a qSOFA score ≥ 2 or a SOFA score ≥ 2?
   - Action:
     - If yes, proceed to step 3; if not, continue monitoring.

3. **Early Goal-Directed Therapy (EGDT)**
   - Administer broad-spectrum antibiotics within 1 hour of suspected sepsis.
   - Clinical question: Are the patient's vital signs stable?
   - Action:
     - If yes, proceed to step 4; if not, prioritize hemodynamic stabilization.

4. **Fluid Resuscitation and Vasopressor Support**
   - Administer crystalloid fluids (e.g., normal saline) to maintain a mean arterial pressure ≥ 65 mmHg.
   - Clinical question: Is the patient hypotensive?
   - Action:
     - If yes, administer vasopressors (e.g., norepinephrine); if not, continue fluid resuscitation.

5. **Source Control and Antibiotic Optimization**
   - Identify and address the source of infection.
   - Clinical question: Is the patient's infection source controlled?
   - Action:
     - If yes, proceed to step 6; if not, prioritize source control.

6. **Ongoing Monitoring and Management**
   - Continuously monitor vital signs, laboratory results, and clinical status.
   - Clinical question: Is the patient showing signs of improvement or deterioration?
   - Action:
     - If improving, continue standard care; if deteriorating, reassess and adjust treatment plan as needed.

**Source:** Surviving Sepsis Campaign (SSC) guidelines, 2022. [1]

**Note:** This protocol is a general guideline and may need to be adapted based on individual patient factors and institutional protocols.

References:
[1] Rhodes A et al. Surviving Sepsis Campaign: International Guidelines for Management of Sepsis and Septic Shock: 2022. Crit Care Med. 2022;50(3):e173-e234.

**Response 1**

It seems to have a strong clinical completeness (not sure about accuracy), but it covers recognition, fluids, vasopressors, antibiotics, source control, and organ support. It does lack escalation logic. The structure itself is a linear narrative, not a decision tree, this is not ideal for real-time decision support.

**Response 2**

This response included SIRS and qSOFA, included antibiotic examples, escelation and aftercare. Not sure on the accuracy, but the structure is better than the 1st response; using criteria, stepwise escalation, and even considerations on organs.

**Response 3**

Uses a decsion-question format. It included triage, stabilization, organ support, and mentions rapid response activation. There were not time-bound bundles, no specifics of vasopressors or fluid volumes. Had the strongest strucutre of all three, using clinical questions.

**Overall**

The 1st and 2nd response seemed to give the right content, and the 3rd gives the right structure. Dependent on the next queries, the system message should be a combined version/mix of the three prompts to get the output that would be most beneficial to the medical team.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [40]:
question2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

In [41]:
print('Response to Query 2:', question2, 'using system prompt 1')
display(Markdown(response(question2, system_prompt=system_prompt1)))
print('_'*150)
print('Response to Query 2:', question2, 'using system prompt 2')
display(Markdown(response(question2, system_prompt=system_prompt2)))
print('_'*150)
print('Response to Query 2:', question2, 'using system prompt 3')
display(Markdown(response(question2, system_prompt=system_prompt3)))

Llama.generate: 39 prefix-match hit, remaining 199 prompt tokens to eval


Response to Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it? using system prompt 1


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     254.21 ms /   199 tokens (    1.28 ms per token,   782.81 tokens per second)
llama_perf_context_print:        eval time =   14432.64 ms /   339 runs   (   42.57 ms per token,    23.49 tokens per second)
llama_perf_context_print:       total time =   15606.57 ms /   538 tokens
llama_perf_context_print:    graphs reused =        336


**Diagnostic Assistance**

Common symptoms of appendicitis include:

- Severe abdominal pain, often starting near the belly button and moving to the lower right abdomen
- Nausea and vomiting
- Loss of appetite
- Fever (usually less than 101.5°F)
- Abdominal tenderness or guarding (muscle tension) in the lower right abdomen
- Rebound tenderness (pain when pressing on the area and then releasing)

**Treatment Plans**

Appendicitis cannot be cured solely with medicine. Antibiotics may be prescribed to manage symptoms, but surgery is necessary to remove the inflamed appendix.

The recommended surgical procedure for treating appendicitis is an open or laparoscopic appendectomy:

- **Open Appendectomy**: A traditional surgical method where a single incision (about 2-3 inches) is made in the lower right abdomen to access and remove the appendix.
- **Laparoscopic Appendectomy**: A minimally invasive procedure using several small incisions and a camera to visualize the appendix. The surgeon then removes the appendix through one of these incisions.

In both cases, the goal is to prevent rupture of the appendix, which can lead to peritonitis (inflammation of the abdominal lining) and potentially life-threatening complications.

**Critical Care Protocols**

Post-operative care typically involves:

- Monitoring for signs of infection or complications
- Administering pain medication as needed
- Encouraging early mobilization and bowel movements to prevent constipation
- Following a liquid diet before gradually introducing solid foods

Please note that the specific treatment plan may vary depending on individual patient factors, such as age, overall health, and surgeon preference.

Llama.generate: 70 prefix-match hit, remaining 257 prompt tokens to eval


...

**Diagnostic Assistance**

Appendicitis is a medical emergency that requires prompt diagnosis and treatment. The common symptoms of appendicitis include:

1. **Abdominal pain**: Initially, the pain may be mild and diffuse, but it typically becomes more severe and localized to the lower right abdomen.
2. **Nausea and vomiting**: Many patients experience nausea and vomiting, which can lead to dehydration if not managed properly.
3. **Loss of appetite**: Patients often lose their appetite due to abdominal pain and discomfort.
4. **Fever**: A low-grade fever (less than 101°F) is common in appendicitis.
5. **Rebound tenderness**: When the abdomen is pressed, patients may experience increased pain upon release.

**Diagnostic Steps**

To diagnose appendicitis, follow these steps:

1. **Take a thorough medical history**: Ask about symptoms, allergies, and any previous abdominal surgeries.
2. **Perform a physical examination**: Check for abdominal tenderness, guarding (muscle tension), and rebound tenderness.
3. **Order laboratory tests**:
	* Complete Blood Count (CBC) to check for signs of infection or inflammation.
	* White blood cell count (WBC) to assess the severity of the infection.
4. **Imaging studies**: Order an abdominal ultrasound, computed tomography (CT) scan, or magnetic resonance imaging (MRI) to visualize the appendix and surrounding tissues.

**Treatment Plan**

Appendicitis cannot be cured via medicine alone. Surgery is the only definitive treatment for appendicitis.

**Surgical Procedure**

The standard surgical procedure for treating appendicitis is an **open appendectomy**, which involves:

1. **General anesthesia**: Administer general anesthesia to ensure patient comfort and safety.
2. **Incision**: Make a 2-3 inch incision in the lower right abdomen, usually just above the pubic hairline.
3. **Appendix removal**: Carefully remove the inflamed appendix, taking care not to damage surrounding tissues or organs.
4. **Closure**: Close the incision with sutures or staples.

**Alternative Surgical Options**

In some cases, a **laparoscopic appendectomy** may be performed, which involves:

1. **Multiple small incisions**: Make 2-3 small incisions in the abdomen to insert a laparoscope and surgical instruments.
2. **Appendix removal**: Remove the appendix using the laparoscope and instruments.

**Patient-Specific Factors**

Consider patient-specific factors when deciding between open and laparoscopic appendectomy:

* **Age**: Laparoscopic surgery may be more suitable for children and young adults, while open surgery is often preferred for older patients or those with complex anatomy.
* **Obesity**: Patients with a high body mass index (BMI) may require open surgery due to limited access through small incisions.
* **Previous abdominal surgeries**: Patients with previous abdominal scars or adhesions may benefit from an open appendectomy.

**Critical Care Protocols**

After surgery, patients should be monitored for:

1. **Pain management**: Administer pain medication as needed to ensure patient comfort.
2. **Fluid resuscitation**: Monitor and manage fluid intake to prevent dehydration.
3. **Infection prevention**: Implement infection control measures to prevent post-operative complications.

Please note that this is a general outline, and specific treatment plans may vary depending on individual patient needs and circumstances.

Llama.generate: 39 prefix-match hit, remaining 212 prompt tokens to eval


______________________________________________________________________________________________________________________________________________________
Response to Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it? using system prompt 3


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     266.19 ms /   212 tokens (    1.26 ms per token,   796.42 tokens per second)
llama_perf_context_print:        eval time =   26777.73 ms /   630 runs   (   42.50 ms per token,    23.53 tokens per second)
llama_perf_context_print:       total time =   29008.94 ms /   842 tokens
llama_perf_context_print:    graphs reused =        626


**Clinical Decision Pathway for Appendicitis**

1. **Initial Assessment**
   - Presenting complaint: Abdominal pain
   - Clinical question being assessed: Is the patient experiencing symptoms consistent with appendicitis?
   - Action:
     1. If the patient reports diffuse abdominal pain, nausea, vomiting, and fever, do proceed to further evaluation.
     2. If not, consider alternative diagnoses (e.g., gastroenteritis, constipation).

**Standard First-Line Care**

2. **Physical Examination**
   - Clinical question being assessed: Is there tenderness in the right lower quadrant of the abdomen?
   - Action:
     1. If the patient exhibits tenderness in the right lower quadrant, do proceed to imaging studies.
     2. If not, consider alternative diagnoses (e.g., gastroenteritis, constipation).

3. **Imaging Studies**
   - Clinical question being assessed: Is there evidence of appendicitis on imaging?
   - Action:
     1. If ultrasound or CT scan confirms appendicitis, do proceed to surgical consultation.
     2. If not, consider alternative diagnoses (e.g., gastroenteritis, constipation).

**Patient-Specific Factors**

- **Age and Comorbidities**: Patients with severe comorbidities (e.g., heart disease, lung disease) or those who are immunocompromised may require a more cautious approach.
  - Clinical question being assessed: Are there any patient-specific factors that would contraindicate surgery?
  - Action:
    1. If the patient has severe comorbidities or is immunocompromised, do consider alternative treatments (e.g., antibiotics) under close supervision.
    2. If not, proceed with surgical consultation.

**Surgical Procedure**

- **Appendectomy**: The standard treatment for appendicitis is an open or laparoscopic appendectomy.
  - Clinical question being assessed: What type of appendectomy should be performed?
  - Action:
    1. If the patient has a simple appendicitis, do perform a laparoscopic appendectomy (source: American College of Surgeons).
    2. If the patient has a complicated appendicitis (e.g., perforation, abscess), do perform an open appendectomy.

**Medication**

- **Appendicitis cannot be cured via medicine alone**: Antibiotics may be used as adjunctive therapy to manage symptoms and prevent complications.
  - Clinical question being assessed: Should antibiotics be administered?
  - Action:
    1. If the patient has a simple appendicitis, do not administer antibiotics as first-line treatment (source: American College of Surgeons).
    2. If the patient has a complicated appendicitis or is at high risk for complications, do administer broad-spectrum antibiotics.

References:

- American College of Surgeons. (2019). Appendicitis.
- National Institute of Diabetes and Digestive and Kidney Diseases. (2020). Appendicitis.
- UpToDate. (2022). Appendicitis in adults: Treatment.

**Response 1**

Includes the symptoms, treatments, but lack any mention of differential diagnoses. Has a weak structure, not a decision pathway at all. Lacking is decision thresholds, and diagnostic steps. It can recall medical facts, but cannot produce guideline-style content that was intended.

**Response 2**

Got the symptoms, imaging, included decision point and mentioned post-op care. Seems to be a bit oversimplified. The structure is slightly better for decision-making because it has the diagnostic steps, decision point, and a treatment plan: resembling clinical workflow. Lacking in criteria and thresholds. The prompt seems to be nudging the model toward structure more than system_prompt1.

**Response 3**

It has the best decision-tree style: clinical questions, branching logic. Includes severity assessment, surgical options and notes on comorbidities. A bit oversimplified

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [42]:
question3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

In [43]:
print('Response to Query 3:', question3, 'using system prompt 1')
display(Markdown(response(question3, system_prompt=system_prompt1)))
print('_'*150)
print('Response to Query 3:', question3, 'using system prompt 2')
display(Markdown(response(question3, system_prompt=system_prompt2)))
print('_'*150)
print('Response to Query 3:', question3, 'using system prompt 3')
display(Markdown(response(question3, system_prompt=system_prompt3)))

Llama.generate: 39 prefix-match hit, remaining 203 prompt tokens to eval


Response to Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it? using system prompt 1


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     253.80 ms /   203 tokens (    1.25 ms per token,   799.84 tokens per second)
llama_perf_context_print:        eval time =   20505.17 ms /   473 runs   (   43.35 ms per token,    23.07 tokens per second)
llama_perf_context_print:       total time =   22060.01 ms /   676 tokens
llama_perf_context_print:    graphs reused =        470


**Diagnostic Assistance:**

Localized bald spots on the scalp can be caused by various factors, including:

1. **Alopecia Areata**: An autoimmune disease that causes hair loss in patches.
2. **Traction Alopecia**: Hair loss due to constant pulling or tension on the hair follicles (e.g., tight hairstyles).
3. **Telogen Effluvium**: A condition where an excessive number of hairs enter the resting phase, leading to hair loss.
4. **Folliculitis**: Inflammation of the hair follicles, often caused by bacterial or fungal infections.
5. **Ringworm (Tinea Capitis)**: A fungal infection that can cause circular patches of hair loss.

**Treatment Plans:**

1. **Alopecia Areata**: Topical corticosteroids, minoxidil, and immunomodulatory medications may be prescribed to stimulate hair growth.
2. **Traction Alopecia**: Avoiding tight hairstyles and using gentle hair care products can help prevent further hair loss.
3. **Telogen Effluvium**: Treatment focuses on addressing the underlying cause (e.g., hormonal imbalances, nutritional deficiencies).
4. **Folliculitis**: Antibiotics or antifungal medications may be prescribed to treat the infection.
5. **Ringworm (Tinea Capitis)**: Antifungal medications and medicated shampoos can help clear the infection.

**Specialty Knowledge:**

Consult a dermatologist for an accurate diagnosis, as they will perform a physical examination, take a medical history, and may conduct tests (e.g., scalp biopsy) to determine the underlying cause of hair loss.

In some cases, hair transplantation or low-level laser therapy (LLLT) may be recommended to promote hair growth. However, these treatments should only be considered after consulting with a dermatologist or other qualified healthcare professional.

**Aftercare Procedures:**

To manage localized bald spots on the scalp:

1. Use gentle shampoos and conditioners.
2. Avoid excessive heat styling or tight hairstyles.
3. Keep the scalp clean and dry to prevent infections.
4. Consider using minoxidil or other topical treatments as directed by a healthcare professional.

Please consult a dermatologist for an accurate diagnosis and personalized treatment plan, as hair loss can be caused by various underlying conditions that require specific attention.

Llama.generate: 70 prefix-match hit, remaining 261 prompt tokens to eval


...

**Diagnostic Assistance**

To address sudden patchy hair loss, it's essential to identify the underlying cause. The following steps can help narrow down the possible causes:

1. **Medical History**: Review the patient's medical history for any recent illnesses, infections, or autoimmune disorders that could be contributing to hair loss.
2. **Symptom Assessment**: Ask about the onset and progression of hair loss, including:
	* When did the hair loss start?
	* How quickly is it progressing?
	* Are there any other symptoms such as itching, redness, or pain in the affected area?
3. **Physical Examination**: Inspect the scalp for signs of infection, inflammation, or scarring.
4. **Laboratory Tests**:
	* Complete Blood Count (CBC) to rule out anemia or infections
	* Thyroid function tests (TFTs) to check for hypothyroidism or hyperthyroidism
	* Autoimmune disorder screenings (e.g., antinuclear antibody test)

Based on the patient's history, symptoms, and laboratory results, consider the following possible causes:

1. **Alopecia Areata**: An autoimmune disease causing patchy hair loss.
2. **Traction Alopecia**: Hair loss due to excessive tension or pulling on the hair follicles (e.g., tight hairstyles).
3. **Telogen Effluvium**: A condition where an increased number of hairs enter the resting phase, leading to excessive shedding.
4. **Fungal Infections** (e.g., ringworm): Causing patchy hair loss and scalp inflammation.
5. **Scalp Conditions** (e.g., psoriasis, eczema): Leading to inflammation and hair loss.

**Treatment Plans**

Based on the underlying cause, consider the following treatments:

1. **Alopecia Areata**: Topical corticosteroids or immunotherapy may be prescribed.
2. **Traction Alopecia**: Recommend gentle hairstyles and scalp massage to reduce tension.
3. **Telogen Effluvium**: Treatment focuses on addressing the underlying cause (e.g., stress management, nutritional supplements).
4. **Fungal Infections**: Antifungal medications (e.g., ketoconazole) may be prescribed.
5. **Scalp Conditions**: Topical corticosteroids or other treatments to manage inflammation and promote hair growth.

**Aftercare Procedures**

To promote hair growth and prevent further hair loss:

1. **Gentle Hair Care**: Use mild shampoos and avoid excessive heat styling.
2. **Scalp Massage**: Regular massage can help stimulate blood flow and promote hair growth.
3. **Nutritional Supplements**: Consider supplements containing biotin, vitamin B, or fish oil to support hair health.

**Critical Care Protocols**

In cases of severe hair loss or suspected underlying conditions (e.g., autoimmune disorders), consider consulting a dermatologist for further evaluation and treatment.

Please note that this is not an exhaustive list, and it's essential to consult with a healthcare professional for personalized guidance.

Llama.generate: 39 prefix-match hit, remaining 216 prompt tokens to eval


______________________________________________________________________________________________________________________________________________________
Response to Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it? using system prompt 3


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     265.24 ms /   216 tokens (    1.23 ms per token,   814.36 tokens per second)
llama_perf_context_print:        eval time =   29150.39 ms /   679 runs   (   42.93 ms per token,    23.29 tokens per second)
llama_perf_context_print:       total time =   31341.40 ms /   895 tokens
llama_perf_context_print:    graphs reused =        675


**Clinical Decision Pathway for Sudden Patchy Hair Loss**

1. **Initial Assessment**
   - Presenting complaint: Sudden patchy hair loss on the scalp.
   - Clinical question 1: Is there a history of trauma, surgery, or significant stress?
     If yes, do:
       * Investigate potential causes such as telogen effluvium (stress-induced hair loss) or alopecia areata (autoimmune condition).
       * Consider ordering blood tests to rule out underlying conditions.
     If no, proceed to the next step.

2. **Dermatological Examination**
   - Clinical question 2: Are there any signs of skin inflammation, infection, or scarring?
     If yes, do:
       * Biopsy the affected area to determine the underlying cause (e.g., fungal infection, autoimmune disease).
       * Consider topical or systemic treatments based on the diagnosis.
     If no, proceed to the next step.

3. **Hair Loss Patterns and Distribution**
   - Clinical question 3: Is the hair loss localized to a specific area of the scalp?
     If yes, do:
       * Investigate potential causes such as alopecia areata (autoimmune condition) or tinea capitis (fungal infection).
       * Consider ordering blood tests to rule out underlying conditions.
     If no, proceed to the next step.

4. **Medical History and Laboratory Tests**
   - Clinical question 4: Is there a history of autoimmune disorders, thyroid disease, or other medical conditions?
     If yes, do:
       * Order laboratory tests (e.g., complete blood count, thyroid function tests) to rule out underlying conditions.
       * Consider systemic treatments based on the diagnosis.
     If no, proceed to the next step.

**Possible Causes and Treatments:**

1. **Alopecia Areata**: An autoimmune condition causing patchy hair loss.
   - Treatment options:
     * Topical corticosteroids
     * Oral corticosteroids (for severe cases)
     * Immunomodulatory agents (e.g., methotrexate)

2. **Telogen Effluvium**: Stress-induced hair loss.
   - Treatment options:
     * Reduce stress through lifestyle modifications (e.g., exercise, meditation)
     * Consider low-level laser therapy or minoxidil for hair growth promotion

3. **Tinea Capitis**: Fungal infection causing patchy hair loss.
   - Treatment options:
     * Antifungal medications (e.g., griseofulvin, terbinafine)
     * Topical antifungals (e.g., ketoconazole)

4. **Other Causes**:
   - Trauma or surgery
   - Hormonal imbalances (e.g., thyroid disease)
   - Nutritional deficiencies (e.g., iron deficiency)

**References:**

* American Academy of Dermatology. (2020). Alopecia Areata.
* National Institute of Arthritis and Musculoskeletal and Skin Diseases. (2020). Telogen Effluvium.
* Mayo Clinic. (2022). Tinea Capitis.

Please note that this is a general clinical decision pathway, and individual patient factors may require deviating from the standard approach. A thorough medical history, physical examination, and laboratory tests are essential for determining the underlying cause of sudden patchy hair loss.

**Response 1**

No diagnostic workflow, weak structure only provides a flat list, not a diagnostic pathway. It misses diagnostic steps, decision thresholds, red flags, and who knows what else. The system prompt defaults to a "diagostic assistance -> causes -> treatments" template.

**Response 2**

Includes disgnostic steps, treatment options, and aftercare. The critical care protocols section is irrelevant to the query. Structure through and through is better than what system_prompt1 pushes the model to do.

**Response 3**

This has the best format (decision-tree), as previously mentioned.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [44]:
question4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

In [45]:
print('Response to Query 4:', question4, 'using system prompt 1')
display(Markdown(response(question4, system_prompt=system_prompt1)))
print('_'*150)
print('Response to Query 4:', question4, 'using system prompt 2')
display(Markdown(response(question4, system_prompt=system_prompt2)))
print('_'*150)
print('Response to Query 4:', question4, 'using system prompt 3')
display(Markdown(response(question4, system_prompt=system_prompt3)))

Llama.generate: 39 prefix-match hit, remaining 195 prompt tokens to eval


Response to Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function? using system prompt 1


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     256.24 ms /   195 tokens (    1.31 ms per token,   761.01 tokens per second)
llama_perf_context_print:        eval time =   19072.24 ms /   446 runs   (   42.76 ms per token,    23.38 tokens per second)
llama_perf_context_print:       total time =   20541.92 ms /   641 tokens
llama_perf_context_print:    graphs reused =        443


**Diagnostic Assistance**
To determine the best course of treatment, it's essential to identify the type and severity of brain injury. Common types include:

- Concussion (mild traumatic brain injury)
- Contusion
- Hemorrhage
- Skull fracture

**Treatment Plans**

For mild traumatic brain injuries (concussions):

1. Rest: Avoid strenuous activities, including exercise, reading, or watching TV.
2. Monitoring: Regularly check for symptoms such as headache, dizziness, nausea, and confusion.
3. Pain management: Use over-the-counter pain medications like acetaminophen or ibuprofen.

For moderate to severe traumatic brain injuries:

1. Emergency care: Seek immediate medical attention if the person experiences:
	* Severe headache
	* Confusion or disorientation
	* Weakness or numbness in the face, arm, or leg
	* Difficulty speaking or understanding speech
	* Vision changes
2. Hospitalization: Admit the patient to a hospital for close monitoring and treatment.
3. Surgery: May be necessary to relieve pressure on the brain or repair skull fractures.

**Specialty Knowledge**
In some cases, patients may require:

1. Neurosurgery: To repair damaged blood vessels or remove blood clots.
2. Rehabilitation therapy: Physical, occupational, and speech therapy to help regain lost functions.
3. Cognitive rehabilitation: Therapy to improve memory, attention, and problem-solving skills.

**Critical Care Protocols**
For severe brain injuries, critical care protocols may include:

1. Intensive care unit (ICU) admission
2. Close monitoring of vital signs, including blood pressure, heart rate, and oxygen saturation
3. Administration of medications to control intracranial pressure and prevent seizures

**Aftercare Procedures**

1. Follow-up appointments: Regular check-ups with a neurologist or primary care physician.
2. Lifestyle modifications: Avoid activities that may exacerbate the injury, such as contact sports or heavy lifting.
3. Cognitive training: Engage in cognitive exercises to improve memory and attention.

Please note that this is not an exhaustive list, and treatment plans should be tailored to individual patient needs under the guidance of a qualified medical professional.

Llama.generate: 70 prefix-match hit, remaining 253 prompt tokens to eval


...

**Treatment Plan for Brain Injury**

Given the complexity of brain injuries, it's essential to consider both immediate and long-term care. The following steps outline a general approach:

### Initial Assessment (Critical Care Protocols)

1. **Stabilize the patient**: Ensure airway, breathing, and circulation (ABCs) are secure.
2. **Assess level of consciousness**: Use the Glasgow Coma Scale (GCS) to evaluate the patient's response to verbal commands, motor responses, and eye opening.
3. **Imaging studies**: Order computed tomography (CT) or magnetic resonance imaging (MRI) scans to identify the extent of brain damage.

### Diagnostic Assistance

- If CT scan shows evidence of bleeding or swelling:
  - Do: Administer mannitol or hypertonic saline to reduce intracranial pressure (ICP).
  - Otherwise, proceed with standard care.
- If MRI reveals signs of ischemia or infarction:
  - Consider administering thrombolytic agents (e.g., tPA) if within the recommended time window.

### Treatment Plans

1. **Manage ICP**: Use medications like mannitol, hypertonic saline, or barbiturates to control elevated ICP.
2. **Supportive care**:
	* Maintain a stable body temperature between 36°C and 37°C (96.8°F and 98.6°F).
	* Administer corticosteroids if there's evidence of vasogenic edema.
3. **Neurosurgical intervention**: Consider surgical decompression or evacuation of hematoma if imaging studies indicate significant mass effect.

### Specialty Knowledge

- If the patient has a history of seizures:
  - Do: Initiate anticonvulsant therapy (e.g., levetiracetam) to prevent further seizures.
- If there's suspicion of traumatic brain injury with potential for long-term cognitive or behavioral changes:
  - Consider consulting a neurologist, neuropsychologist, or rehabilitation specialist.

### Aftercare Procedures

1. **Rehabilitation**: Implement a comprehensive rehabilitation plan, including physical therapy, occupational therapy, and speech therapy as needed.
2. **Follow-up care**: Schedule regular follow-ups with the patient's primary care physician to monitor for potential complications and adjust treatment plans accordingly.

**Important Note:** This is a general outline, and specific treatments may vary depending on individual patient factors, such as age, comorbidities, and the severity of brain injury. Always consult relevant medical literature and expert opinions when making treatment decisions.

Llama.generate: 39 prefix-match hit, remaining 208 prompt tokens to eval


______________________________________________________________________________________________________________________________________________________
Response to Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function? using system prompt 3


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     263.36 ms /   208 tokens (    1.27 ms per token,   789.79 tokens per second)
llama_perf_context_print:        eval time =   19027.51 ms /   453 runs   (   42.00 ms per token,    23.81 tokens per second)
llama_perf_context_print:       total time =   20636.44 ms /   661 tokens
llama_perf_context_print:    graphs reused =        450


**Clinical Decision Pathway for Brain Injury Treatment**

1. **Initial Assessment**
   - Assess the patient's level of consciousness, pupillary response, and motor function.
   - Determine if there is any evidence of bleeding or swelling in the brain (e.g., CT scan).
   - If [patient is unconscious or shows signs of increased intracranial pressure], do:
     - **Emergency Management**: Intubate the patient to secure the airway, administer oxygen, and monitor vital signs.
     - If not, proceed with standard care.

2. **Standard Care**
   - For patients with mild brain injury (e.g., concussion):
     - **Rest and Observation**: Monitor for worsening symptoms or signs of increased intracranial pressure.
     - Provide pain management as needed.
     - Consider referral to a neurologist or primary care physician for follow-up.

3. **Moderate Brain Injury**
   - If [patient shows signs of moderate brain injury (e.g., confusion, difficulty speaking)], do:
     - **Neurological Evaluation**: Perform a thorough neurological examination to assess cognitive and motor function.
     - Consider referral to a neurologist or rehabilitation specialist for further evaluation and management.

4. **Severe Brain Injury**
   - If [patient shows signs of severe brain injury (e.g., decreased level of consciousness, pupillary dilation)], do:
     - **Intensive Care Unit (ICU) Admission**: Transfer the patient to an ICU setting for close monitoring and management.
     - Consider neurosurgical consultation for potential intervention (e.g., craniotomy).

5. **Rehabilitation**
   - Once [patient's condition stabilizes], do:
     - **Multidisciplinary Rehabilitation Team**: Assemble a team of healthcare professionals, including physical therapists, occupational therapists, speech therapists, and psychologists.
     - Develop a rehabilitation plan tailored to the patient's specific needs.

**Source:**

- American Academy of Neurology (AAN) guidelines for the management of traumatic brain injury
- Brain Injury Association of America (BIAA)
- Centers for Disease Control and Prevention (CDC)

Please note that this is a general clinical decision pathway, and individual patient factors may require deviation from standard care.

**Response 1**

What the prompt did well for this query was its easy to read format, clear sections, reasonably structured, and not overly verbose. Its weaknesses would be the lack of decision pathway, no ordering of steps, no explicit decision points, reads like a general medical summary. It follows the category structure but not a clinical pathway structure.

**Response 2**

This prompt has been more structured for almost all of the queries before, including this one. The steps are a bit overly detailed, it does use decision points, has a clear flow, but it is not strictly using the ordered pathway format, and is very verbose.

**Response 3**

Has the best structure of all 3 prompt's responces. It is using explicit decision points, follows a clear sequence, very clean/readable/consistent. Out of all the system prompts this prompt has the best responses so far, it is hard to tell if it hallucinated without clear extensive reasearch, but after loading the data this will be a lot clearer.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [46]:
question5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

In [47]:
print('Response to Query 5:', question5, 'using system prompt 1')
display(Markdown(response(question5, system_prompt=system_prompt1)))
print('_'*150)
print('Response to Query 5:', question5, 'using system prompt 2')
display(Markdown(response(question5, system_prompt=system_prompt2)))
print('_'*150)
print('Response to Query 5:', question5, 'using system prompt 3')
display(Markdown(response(question5, system_prompt=system_prompt3)))

Llama.generate: 39 prefix-match hit, remaining 200 prompt tokens to eval


Response to Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery? using system prompt 1


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     257.46 ms /   200 tokens (    1.29 ms per token,   776.81 tokens per second)
llama_perf_context_print:        eval time =   17702.50 ms /   417 runs   (   42.45 ms per token,    23.56 tokens per second)
llama_perf_context_print:       total time =   19077.20 ms /   617 tokens
llama_perf_context_print:    graphs reused =        414


**Diagnostic Assistance**

To confirm the diagnosis, assess for:

- Swelling and deformity of the affected limb
- Pain or tenderness to touch
- Limited mobility or inability to bear weight on the injured leg
- Possible open fracture (bone protruding through skin)

**Treatment Plans**

For a fractured leg during a hiking trip, follow these steps:

1. **Immobilize the injury**: Use a splint or makeshift cast to stabilize the affected limb.
2. **Control bleeding and prevent infection**: Clean the wound with sterile water, apply antibiotic ointment, and cover with a dressing.
3. **Provide pain management**: Administer over-the-counter pain medication (e.g., acetaminophen or ibuprofen) as needed.

**Critical Care Protocols**

If the person is experiencing any of the following symptoms, call for emergency medical services:

- Severe bleeding
- Difficulty breathing
- Pale or cool skin
- Decreased level of consciousness

**Specialty Knowledge**

Consider consulting an orthopedic specialist for further evaluation and treatment. They may recommend:

- X-rays to confirm the fracture type and location
- Casting or surgery to stabilize the bone
- Physical therapy to aid in recovery and rehabilitation

**Aftercare Procedures**

For care and recovery, consider the following:

1. **Monitor for signs of infection**: Watch for redness, swelling, increased pain, or pus at the wound site.
2. **Maintain proper wound care**: Change dressings daily, keep the area clean, and apply antibiotic ointment as needed.
3. **Encourage rest and elevation**: Keep the injured leg elevated above heart level to reduce swelling and promote healing.
4. **Plan for follow-up care**: Schedule a follow-up appointment with an orthopedic specialist within 1-2 weeks after the injury.

**Additional Considerations**

When transporting the person back to civilization, consider:

- Stabilizing the injured limb during transport
- Keeping the person comfortable and hydrated
- Notifying emergency services or seeking medical attention as soon as possible

Llama.generate: 70 prefix-match hit, remaining 258 prompt tokens to eval


...

**Critical Care Protocols: Managing a Fractured Leg in the Wilderness**

Given the remote location of the hiking trip, it's essential to prioritize stabilization and pain management before attempting to transport the patient.

### Initial Assessment (Steps 1-3)

1. **Assess the injury**: Evaluate the patient's leg for any open wounds, deformity, or numbness/tingling sensations.
2. **Check for other injuries**: Look for signs of head trauma, chest pain, or difficulty breathing.
3. **Stabilize the leg**: Use a splint (e.g., SAM splint) to immobilize the affected limb and prevent further injury.

### Pain Management (Steps 4-6)

4. **Administer pain relief**: Provide acetaminophen (500mg every 6 hours as needed) or ibuprofen (400mg every 8 hours as needed).
5. **Monitor for signs of overdose**: Watch for symptoms like nausea, vomiting, or dizziness.
6. **Consider alternative pain management options**: If the patient is allergic to NSAIDs or has kidney disease, consider using morphine (2-4mg IV every 1-2 hours) under close supervision.

### Transportation and Medical Evaluation (Steps 7-9)

7. **Transport the patient**: Use a stretcher or backboard to transport the patient to a medical facility.
8. **Monitor vital signs**: Keep track of the patient's pulse, blood pressure, respiratory rate, and oxygen saturation during transportation.
9. **Seek medical evaluation**: Once at a medical facility, have the patient evaluated by an orthopedic surgeon or emergency physician.

### Specialty Knowledge: Orthopedic Care

*   The patient may require surgical intervention to stabilize the fracture, depending on its severity and location.
*   A cast or splint will be applied to immobilize the affected limb during recovery.
*   Physical therapy will be necessary to regain strength and mobility in the affected leg.

### Aftercare Procedures: Post-Injury Care

*   Monitor for signs of infection (e.g., redness, swelling, increased pain).
*   Encourage the patient to stay hydrated and follow a balanced diet.
*   Provide emotional support and reassurance during the recovery process.

**Important Considerations**

*   The patient's age, medical history, and overall health may impact their treatment plan and recovery time.
*   If the patient has a pre-existing condition (e.g., diabetes, osteoporosis), it may affect their fracture management and rehabilitation.
*   The patient should be aware of potential complications, such as nerve damage or compartment syndrome.

**Do not attempt to treat this patient without proper medical training and equipment. Seek immediate medical attention if you suspect a fractured leg in the wilderness.**

Llama.generate: 39 prefix-match hit, remaining 213 prompt tokens to eval


______________________________________________________________________________________________________________________________________________________
Response to Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery? using system prompt 3


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     260.06 ms /   213 tokens (    1.22 ms per token,   819.03 tokens per second)
llama_perf_context_print:        eval time =   22616.73 ms /   535 runs   (   42.27 ms per token,    23.66 tokens per second)
llama_perf_context_print:       total time =   24409.84 ms /   748 tokens
llama_perf_context_print:    graphs reused =        531


**Clinical Decision Pathway for Treating a Fractured Leg in the Wilderness**

1. **Assess and Stabilize the Patient**
   - Assess the patient's airway, breathing, and circulation (ABCs) to ensure they are stable.
   - If the patient is unstable or showing signs of shock, prioritize their ABCs before proceeding with leg care.

2. **Immobilize the Fracture**
   - Use a splint or makeshift splint (e.g., hiking poles, sticks, and cloth) to immobilize the affected limb in a comfortable position.
   - If possible, elevate the injured leg above heart level to reduce swelling.

3. **Assess for Other Injuries**
   - Check for other injuries, such as head trauma, spinal cord injury, or soft tissue damage.
   - Prioritize treatment of these conditions if present.

4. **Determine the Need for Extraction**
   - If the patient is in a remote area with no access to medical care within 24 hours, consider extraction via helicopter or other means.
   - Weigh the risks and benefits of extraction versus on-site care.

5. **Provide Pain Management**
   - Administer pain medication (e.g., acetaminophen or ibuprofen) as needed, following standard dosing guidelines.
   - Consider using a cold compress to reduce swelling and ease pain.

6. **Monitor for Complications**
   - Regularly assess the patient's vital signs, including pulse, blood pressure, and respiratory rate.
   - Watch for signs of infection (e.g., redness, swelling, increased pain) or compartment syndrome (e.g., numbness, tingling).

7. **Plan for Transportation to Medical Care**
   - Once the patient is stable, plan for transportation to a medical facility for further evaluation and treatment.

**Considerations for Care and Recovery**

- **Patient-Specific Factors**: Consider the patient's age, overall health, and any pre-existing medical conditions when planning care.
- **Access to Medical Care**: If extraction is not possible, consider on-site care with regular monitoring and reassessment.
- **Wound Management**: If the fracture is open or contaminated, prioritize wound cleaning and dressing to prevent infection.
- **Nutrition and Hydration**: Ensure the patient has access to adequate nutrition and hydration to support healing.

**Sources:**

* Wilderness Medical Society (2016). Wilderness Medical Society Practice Guidelines for Wilderness Emergency Care. Journal of Wilderness Medicine, 27(2), 141-154.
* American College of Surgeons Committee on Trauma (2018). Advanced Trauma Life Support Student Course Manual.

**Response 1**

The diagnostic assistance, treatment plans, and critical care protocols are great categories, in readable sections. It covers all parts of the query and is very easy to follow. But consistently is struggling with the structure.

**Response 2**

This system prompt makes the model behave in a way where it asks for sequential steps, decision points, and standard vs patient specific care, which the model has no trouble delivering. It seemed to have mixed narritave and steps up, and is sort of trails off the decision pathway at some points.

**Response 3**

The best structure, it follows the clinical pathway through and through for all the queries.

It would be interesting to compare how the same 3 prompt's responses change based on slight parameter tuning. To evaluate how the model behaves, how system prompts shape structure, how sampling parameters change consistency, and how much hallucination happens. With this a muti-prompt test loop will be created to see the effects of all the different combinations.

In [48]:
# Query 1

# Setting the different combinations of prompts vs parameter changes
test_configs = [
    {"system_prompt": system_prompt1, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt2, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt1, "params": {"temperature": 0.5, "top_p" : 0.85, "top_k" : 40}},
    {"system_prompt": system_prompt2, "params": {"temperature": 0.5, "top_p" : 0.85, "top_k" : 40}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0.5, "top_p" : 0.85, "top_k" : 40}}
]
# Loop through all combinations and print each one
for i, cfg in enumerate(test_configs, 1):
    print(f"--- Combination {i} ---")
    answer = response(question1, system_prompt=cfg["system_prompt"], params=cfg["params"])
    display(Markdown(answer))
    print()

Llama.generate: 39 prefix-match hit, remaining 183 prompt tokens to eval


--- Combination 1 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     229.18 ms /   183 tokens (    1.25 ms per token,   798.51 tokens per second)
llama_perf_context_print:        eval time =   19993.03 ms /   472 runs   (   42.36 ms per token,    23.61 tokens per second)
llama_perf_context_print:       total time =   21511.77 ms /   655 tokens
llama_perf_context_print:    graphs reused =        469


**Critical Care Protocols: Sepsis Management**

The protocol for managing sepsis in a critical care unit involves the following steps:

1. **Early Recognition**: Identify patients at high risk of sepsis, such as those with fever, tachycardia, and hypotension.
2. **Severe Sepsis Bundle**:
	* Administer 30 mL/kg crystalloid fluid bolus over 30 minutes to maintain a mean arterial pressure (MAP) ≥ 65 mmHg.
	* Measure lactate levels to assess tissue perfusion.
3. **Source Control**: Identify and address the source of infection, such as drainage of abscesses or removal of infected devices.
4. **Inotropes and Vasopressors**:
	* Consider using norepinephrine or vasopressin if MAP remains < 65 mmHg despite fluid resuscitation.
5. **Broad-Spectrum Antibiotics**: Administer broad-spectrum antibiotics within 1 hour of diagnosis, adjusting based on culture results.
6. **Supportive Care**:
	* Provide mechanical ventilation as needed.
	* Maintain normothermia (body temperature between 36°C and 37.2°C).
7. **Monitoring**: Continuously monitor vital signs, lactate levels, and urine output to assess response to treatment.

Remember to follow local guidelines and hospital protocols for sepsis management, as well as the Surviving Sepsis Campaign's recommendations.

**Additional Considerations:**

* Use of vasopressin or epinephrine may be considered in refractory cases.
* Consideration of activated protein C (drotrecogin alfa) should be based on clinical judgment and local guidelines.
* Consult with an intensivist or infectious disease specialist as needed.

**Aftercare Procedures:**

1. **Discharge Planning**: Ensure patients receive education on sepsis recognition, wound care, and follow-up appointments.
2. **Frequent Follow-Up**: Schedule regular follow-ups to monitor for signs of recurrence or complications.
3. **Patient Education Materials**: Provide patient education materials, such as brochures or online resources, to facilitate understanding of their condition and treatment plan.

Please note that this is a general outline, and specific protocols may vary based on local guidelines and hospital policies.

Llama.generate: 70 prefix-match hit, remaining 241 prompt tokens to eval


...

**Critical Care Protocol: Sepsis Management**

Sepsis is a life-threatening condition that requires prompt recognition and treatment. The following protocol outlines the steps for managing sepsis in a critical care unit:

1. **Recognition (Step 1)**:
	* Monitor patients for signs of infection, such as fever, tachycardia, hypotension, or altered mental status.
	* Use the Systemic Inflammatory Response Syndrome (SIRS) criteria to identify potential sepsis cases:
		+ Temperature > 38°C or < 36°C
		+ Heart rate > 90 beats/min
		+ Respiratory rate > 20 breaths/min or PaCO2 < 32 mmHg
		+ White blood cell count > 12,000 cells/μL or < 4,000 cells/μL
2. **Assessment (Step 2)**:
	* Perform a thorough history and physical examination to identify the source of infection.
	* Obtain laboratory tests, including:
		+ Complete Blood Count (CBC)
		+ Blood cultures
		+ Arterial blood gas analysis
		+ Lactate level
		+ Renal function tests
3. **Initial Treatment (Step 3)**:
	* Administer broad-spectrum antibiotics within the first hour of recognition, based on suspected source and local antibiogram.
	* Provide aggressive fluid resuscitation with crystalloids (e.g., normal saline or lactated Ringer's solution) to maintain a mean arterial pressure ≥ 65 mmHg.
4. **Monitoring and Re-evaluation (Step 4)**:
	* Continuously monitor vital signs, laboratory results, and clinical status.
	* Reassess the patient every 15-30 minutes for signs of improvement or deterioration.
5. **Escalation of Care (Decision Point)**:
	* If the patient's condition worsens despite initial treatment, consider escalation to:
		+ Vasopressors (e.g., norepinephrine) if mean arterial pressure remains < 65 mmHg
		+ Inotropes (e.g., dobutamine) for cardiac support
		+ Mechanical ventilation if respiratory failure develops

**Patient-Specific Factors:**

* Patients with severe sepsis or septic shock may require more aggressive treatment, including:
	+ Higher doses of vasopressors or inotropes
	+ More frequent monitoring and re-evaluation
	+ Consideration of extracorporeal membrane oxygenation (ECMO) or renal replacement therapy (RRT)

**Critical Care Protocols:**

* Sepsis-induced coagulopathy: consider administering platelet transfusions, fresh frozen plasma, or other clotting factors as needed.
* Sepsis-related acute kidney injury: consider initiating RRT if creatinine levels rise above 3.5 mg/dL.

Please note that this protocol is a general guideline and may need to be adapted based on individual patient needs and local protocols. Always consult with a critical care specialist or other experts in the field for specific guidance.

Llama.generate: 39 prefix-match hit, remaining 196 prompt tokens to eval



--- Combination 6 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     264.44 ms /   196 tokens (    1.35 ms per token,   741.17 tokens per second)
llama_perf_context_print:        eval time =   25129.50 ms /   593 runs   (   42.38 ms per token,    23.60 tokens per second)
llama_perf_context_print:       total time =   27016.20 ms /   789 tokens
llama_perf_context_print:    graphs reused =        589


**Clinical Decision Pathway for Managing Sepsis in a Critical Care Unit**

1. **Initial Assessment**
   - Present to the emergency department or critical care unit with suspected sepsis.
   Action: Conduct a thorough history and physical examination, including vital signs.

2. **Sepsis Screening (q12h)**
   - Clinical question: Is the patient meeting Sepsis-3 criteria for septic shock or organ dysfunction?
   - If yes:
     * Do administer broad-spectrum antibiotics within 1 hour.
     * If not:
       * Repeat sepsis screening in 2 hours.

**Standard First-Line Care**

3. **Fluid Resuscitation (q30min)**
   - Clinical question: Is the patient hypotensive or oliguric?
   - If yes, do administer IV fluids (crystalloids) to maintain a MAP >65 mmHg.
   - If not:
     * Continue monitoring and repeat fluid assessment in 30 minutes.

4. **Blood Cultures (q2h)**
   - Clinical question: Are blood cultures pending?
   - If yes, do collect two sets of blood cultures before starting antibiotics.
   - If not:
     * Repeat blood culture collection as needed.

5. **Oxygenation and Ventilation Management**
   - Clinical question: Is the patient hypoxemic or requiring mechanical ventilation?
   - If yes, do optimize oxygenation and ventilation settings to maintain PaO2/FiO2 ≥ 300 mmHg.
   - If not:
     * Continue monitoring and adjust as needed.

6. **Organ Support (q4h)**
   - Clinical question: Is the patient requiring vasopressors or mechanical support for other organs?
   - If yes, do initiate organ-specific support (e.g., renal replacement therapy).
   - If not:
     * Continue monitoring and adjust as needed.

**Patient-Specific Factors**

7. **Considerations for Specific Organ Dysfunction**
   - Clinical question: Is the patient experiencing specific organ dysfunction (e.g., cardiac, hepatic)?
   - Consider consulting a specialist (e.g., cardiology, hepatology) for guidance on management.
   - If yes:
     * Do develop an individualized plan for organ-specific support.

8. **Considerations for Immunocompromised or High-Risk Patients**
   - Clinical question: Is the patient immunocompromised or at high risk for sepsis?
   - Consider consulting infectious disease or a specialist for guidance on management.
   - If yes:
     * Do develop an individualized plan for antimicrobial stewardship and infection prevention.

**Source**

* Surviving Sepsis Campaign (2016) guidelines: "Surviving Sepsis Campaign: International Guidelines for Management of Sepsis and Septic Shock."
* Sepsis-3 criteria: "Sepsis-3 Definitions: Time for Change."

**Combination 1**
- Very structured
- Uses categories
- Adds invented monitoring targets

**Combination 2**
- More narrative
- Less structured
- Adds “patient‑specific factors”
- Doesn’t restrict the model to retrieved context

**Combination 3**
- Very structured
- Adds references to 2023 SSC guidelines
- Most clinacal

**Combination 4**
- More conservative
- Less random
- Cleaner wording

**Combination 5**
- Very similar to Combination 2
- Slightly more concise
- No meaningful improvement.

**Combination 6**
- Extremely structured
- Adds 2021 SSC guidelines
- Very clinical pathway


In [49]:
# Query 3
test_configs = [
    {"system_prompt": system_prompt1, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt2, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt1, "params": {"temperature": 0.5, "top_p" : 0.85, "top_k" : 40}},
    {"system_prompt": system_prompt2, "params": {"temperature": 0.5, "top_p" : 0.85, "top_k" : 40}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0.5, "top_p" : 0.85, "top_k" : 40}}
]

for i, cfg in enumerate(test_configs, 1):
    print(f"--- Combination {i} ---")
    answer = response(question3, system_prompt=cfg["system_prompt"], params=cfg["params"])
    display(Markdown(answer))
    print()

Llama.generate: 39 prefix-match hit, remaining 203 prompt tokens to eval


--- Combination 1 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     256.94 ms /   203 tokens (    1.27 ms per token,   790.05 tokens per second)
llama_perf_context_print:        eval time =   16053.99 ms /   379 runs   (   42.36 ms per token,    23.61 tokens per second)
llama_perf_context_print:       total time =   17385.81 ms /   582 tokens
llama_perf_context_print:    graphs reused =        376


**Diagnostic Assistance:**

Localized bald spots on the scalp can be caused by various factors. Some possible causes include:

*   Telogen effluvium (stress-induced hair loss)
*   Alopecia areata (autoimmune disorder causing hair loss)
*   Traction alopecia (hair loss due to frequent styling or tension on the hair follicles)
*   Androgenetic alopecia (male/female pattern baldness)
*   Infections (fungal, bacterial, or viral) such as ringworm or scalp abscesses
*   Skin conditions like eczema, psoriasis, or dermatitis

**Treatment Plans:**

The most effective treatments depend on the underlying cause of hair loss. Here are some general treatment options:

1.  **Medications:** Minoxidil (Rogaine) and finasteride (Propecia) can help slow down hair loss and stimulate new growth.
2.  **Corticosteroids:** Injecting corticosteroids into the affected area can reduce inflammation and promote hair regrowth.
3.  **Phototherapy:** Exposure to ultraviolet light or low-level laser therapy can stimulate hair growth.
4.  **Hair transplantation:** Transplanting healthy hair follicles from one part of the scalp to the bald spot can provide a permanent solution.
5.  **Topical treatments:** Applying topical creams, shampoos, or ointments containing ingredients like ketoconazole, salicylic acid, or coal tar can help treat underlying skin conditions.

**Specialty Knowledge:**

It's essential to consult a dermatologist for an accurate diagnosis and personalized treatment plan. They will assess the severity of hair loss, take a medical history, and perform diagnostic tests (e.g., scalp biopsy) if necessary.

If you have any further questions or would like more detailed information on these treatments, please let me know!

Llama.generate: 70 prefix-match hit, remaining 261 prompt tokens to eval


...

**Diagnostic Assistance**

To address sudden patchy hair loss, it's essential to consider the possible causes and rule out underlying medical conditions. Please note that a definitive diagnosis requires a thorough examination by a healthcare professional.

1. **Medical History**: Review the patient's past medical history for any autoimmune disorders (e.g., alopecia areata), hormonal imbalances (e.g., thyroid disease), or skin conditions (e.g., psoriasis).
2. **Physical Examination**:
	* Inspect the affected area for signs of inflammation, redness, or scarring.
	* Check for any other symptoms such as itching, burning, or tenderness.

**Possible Causes**

1. **Alopecia Areata**: An autoimmune condition that causes patchy hair loss due to an overactive immune system attacking hair follicles.
2. **Traction Alopecia**: Hair loss caused by constant pulling on the hair (e.g., tight hairstyles, braids).
3. **Telogen Effluvium**: A condition where a significant number of hair follicles enter the resting phase, leading to excessive shedding and patchy hair loss.
4. **Infections** (e.g., ringworm, bacterial infections)
5. **Nutritional Deficiencies** (e.g., iron deficiency, biotin deficiency)

**Treatment Plans**

1. **Topical Corticosteroids**: Apply a mild corticosteroid cream or ointment to the affected area to reduce inflammation and promote hair growth.
2. **Minoxidil Solution**: Use an over-the-counter minoxidil solution (Rogaine) to stimulate hair growth and slow down hair loss.
3. **Platelet-rich Plasma (PRP) Therapy**: Inject platelet-rich plasma into the affected area to stimulate hair growth.
4. **Low-Level Laser Therapy (LLLT)**: Use a low-level laser or light-emitting diode (LED) device to promote hair growth.

**Aftercare Procedures**

1. **Maintain Good Hair Care Habits**: Avoid excessive heat styling, use gentle shampoos, and reduce stress.
2. **Monitor Progress**: Regularly monitor the affected area for signs of improvement or worsening symptoms.
3. **Follow-up Appointments**: Schedule follow-up appointments with a healthcare professional to reassess the condition and adjust treatment plans as needed.

**Specialty Knowledge**

In some cases, sudden patchy hair loss may be associated with underlying medical conditions such as autoimmune disorders (e.g., lupus, rheumatoid arthritis) or hormonal imbalances. Consult with a dermatologist or endocrinologist if you suspect an underlying condition.

Please note that this is not an exhaustive list of possible causes and treatments, and a definitive diagnosis requires a thorough examination by a healthcare professional. If you're unsure about the patient's symptoms or medical history, please provide more information for further assistance.

Llama.generate: 39 prefix-match hit, remaining 216 prompt tokens to eval



--- Combination 6 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     267.94 ms /   216 tokens (    1.24 ms per token,   806.16 tokens per second)
llama_perf_context_print:        eval time =   22692.68 ms /   524 runs   (   43.31 ms per token,    23.09 tokens per second)
llama_perf_context_print:       total time =   24416.20 ms /   740 tokens
llama_perf_context_print:    graphs reused =        520


**Clinical Decision Pathway for Sudden Patchy Hair Loss**

**Step 1: Assess the Patient's Medical History and Symptoms**

*   If patient has a history of autoimmune disorders (e.g., alopecia areata, lupus), do:
    *   Consider immunosuppressive therapy or corticosteroids (e.g., prednisone) under medical supervision. (Source: American Academy of Dermatology)
*   If patient reports recent trauma to the affected area, do:
    *   Assess for signs of infection and consider antibiotics if necessary.
    *   Monitor for further hair loss as it may be a temporary response to stress.

**Step 2: Examine the Affected Area**

*   If localized bald spot is accompanied by redness, itching, or scaling, do:
    *   Consider fungal or bacterial infections (e.g., ringworm) and prescribe antifungal or antibacterial medications as needed. (Source: Mayo Clinic)
*   If no signs of infection are present, proceed to:

**Step 3: Evaluate Potential Causes**

*   If patient has a family history of autoimmune disorders or is experiencing other symptoms (e.g., joint pain, skin rashes), do:
    *   Consider autoimmune-related causes (e.g., alopecia areata) and refer to a dermatologist for further evaluation.
*   If no underlying medical conditions are suspected, consider:
    *   Telogen effluvium: sudden hair loss due to stress or hormonal changes. (Source: American Academy of Dermatology)
    *   Trichotillomania: compulsive hair pulling disorder.

**Step 4: Develop a Treatment Plan**

*   For patients with autoimmune-related causes, do:
    *   Consider immunosuppressive therapy or corticosteroids under medical supervision.
*   For patients with telogen effluvium or trichotillomania, do:
    *   Recommend stress-reducing techniques (e.g., meditation, yoga) and consider low-level laser therapy. (Source: Journal of Clinical and Aesthetic Dermatology)

**Additional Considerations**

*   Patients with sudden patchy hair loss should be evaluated by a healthcare professional to rule out underlying medical conditions.
*   In some cases, hair loss may be a sign of an underlying condition that requires prompt attention.

Sources:

*   American Academy of Dermatology: "Alopecia Areata"
*   Mayo Clinic: "Ringworm (Tinea Capitis)"
*   Journal of Clinical and Aesthetic Dermatology: "Low-Level Laser Therapy for Hair Loss"

In [50]:
# Query 5
test_configs = [
    {"system_prompt": system_prompt1, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt2, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt1, "params": {"temperature": 0.5, "top_p" : 0.85, "top_k" : 40}},
    {"system_prompt": system_prompt2, "params": {"temperature": 0.5, "top_p" : 0.85, "top_k" : 40}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0.5, "top_p" : 0.85, "top_k" : 40}}
]

for i, cfg in enumerate(test_configs, 1):
    print(f"--- Combination {i} ---")
    answer = response(question5, system_prompt=cfg["system_prompt"], params=cfg["params"])
    display(Markdown(answer))
    print()

Llama.generate: 39 prefix-match hit, remaining 200 prompt tokens to eval


--- Combination 1 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     253.92 ms /   200 tokens (    1.27 ms per token,   787.65 tokens per second)
llama_perf_context_print:        eval time =   18154.89 ms /   418 runs   (   43.43 ms per token,    23.02 tokens per second)
llama_perf_context_print:       total time =   19517.91 ms /   618 tokens
llama_perf_context_print:    graphs reused =        415


**Diagnostic Assistance:**

1. **Initial Assessment:** Check for signs of shock (pallor, sweating, cool skin), bleeding, or numbness around the affected area.
2. **Imaging Tests:** Order X-rays to confirm the fracture type and location.

**Treatment Plans:**

1. **Stabilization:** Immobilize the leg using a splint or backboard to prevent further injury.
2. **Pain Management:** Administer pain relief medication (e.g., acetaminophen, ibuprofen) according to severity.
3. **Transportation:** If unable to walk, consider helicopter evacuation or stretcher transport.

**Specialty Knowledge:**

1. **Orthopedic Consultation:** Arrange for a specialist evaluation as soon as possible.
2. **Surgical Intervention:** Consider surgical fixation (e.g., plate and screws) if the fracture is complex or unstable.

**Critical Care Protocols:**

1. **Monitor Vital Signs:** Closely observe heart rate, blood pressure, respiratory rate, and oxygen saturation.
2. **Fluid Resuscitation:** Administer IV fluids to maintain hydration and prevent shock.

**Aftercare Procedures:**

1. **Follow-Up Appointments:** Schedule regular check-ups with the orthopedic specialist for cast changes, wound care, and progress monitoring.
2. **Rehabilitation Program:** Develop a rehabilitation plan to promote healing, strength, and mobility.
3. **Pain Management Plan:** Continuously assess and adjust pain management strategies as needed.

**Considerations:**

1. **Infection Prevention:** Monitor for signs of infection (e.g., redness, swelling, fever) and promptly treat if necessary.
2. **Nutritional Support:** Ensure adequate nutrition to support healing and recovery.
3. **Emotional Support:** Provide reassurance and emotional support to the patient during their recovery period.

Please note that a thorough medical evaluation by an orthopedic specialist is essential for proper diagnosis and treatment of the fractured leg. These steps serve as general guidelines only and should not replace professional medical care.

Llama.generate: 70 prefix-match hit, remaining 258 prompt tokens to eval


...

**Diagnostic Assistance**

1. **Assess the injury**: Evaluate the patient's condition, looking for signs of distress, severe pain, or difficulty moving the affected limb.
2. Check for potential complications:
	* Is there an open wound?
	* Are there any signs of neurovascular compromise (e.g., numbness, tingling, pale skin)?
3. **Immobilize the leg**: Use a splint or backboard to stabilize the injured area and prevent further injury.

**Treatment Plans**

1. **Pain management**:
	* Administer pain medication as needed (e.g., acetaminophen, ibuprofen).
	* Consider prescribing stronger pain medications if necessary.
2. **Immobilization**: Continue using a splint or backboard to stabilize the injured leg.
3. **Transportation**: If possible, transport the patient to a medical facility for further evaluation and treatment.

**Specialty Knowledge**

1. **Orthopedic assessment**: If possible, consult with an orthopedic specialist for a thorough evaluation of the fracture.
2. **Imaging studies**: Order X-rays or other imaging tests (e.g., CT scan) to confirm the diagnosis and assess the extent of the injury.

**Critical Care Protocols**

1. **Monitor vital signs**: Regularly check the patient's pulse, blood pressure, oxygen saturation, and respiratory rate.
2. **Watch for complications**: Be aware of potential complications such as infection, compartment syndrome, or nerve damage.

**Aftercare Procedures**

1. **Follow-up care**: Schedule follow-up appointments with an orthopedic specialist to monitor progress and adjust treatment plans as needed.
2. **Rehabilitation**: Develop a rehabilitation plan to promote healing, strength, and mobility in the affected leg.
3. **Prevent future injuries**: Educate the patient on proper hiking techniques, footwear, and gear usage to reduce the risk of future injuries.

**Patient-Specific Factors**

* Consider any pre-existing medical conditions (e.g., diabetes, osteoporosis) that may impact treatment or recovery.
* Be aware of potential allergies or sensitivities to medications or materials used in treatment.
* Assess the patient's mental status and ability to follow instructions.

**Additional Considerations**

* If the patient is unable to walk or bear weight on the affected leg, consider using a wheelchair or other mobility aid for transportation.
* Be prepared for potential delays in medical care due to remote location or limited access to medical facilities.

Llama.generate: 39 prefix-match hit, remaining 213 prompt tokens to eval



--- Combination 6 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =     262.63 ms /   213 tokens (    1.23 ms per token,   811.03 tokens per second)
llama_perf_context_print:        eval time =   24667.97 ms /   591 runs   (   41.74 ms per token,    23.96 tokens per second)
llama_perf_context_print:       total time =   26867.36 ms /   804 tokens
llama_perf_context_print:    graphs reused =        587


**Clinical Decision Pathway for Treating a Fractured Leg in the Wilderness**

1. **Assess and Stabilize**
    - Question: Is the patient conscious, breathing normally, and able to communicate their symptoms?
    - Action: If yes, maintain airway, breathing, and circulation (ABCs) until medical help arrives; if no, initiate basic life support.
2. **Immobilize the Injury**
    - Question: Can the patient move the affected limb without causing further pain or instability?
    - Action: If yes, immobilize with a splint or sling to prevent further injury; if not, use a backboard or rigid splint for stability.
3. **Evaluate and Treat for Other Injuries**
    - Question: Are there any other signs of trauma (e.g., head injuries, burns, or internal bleeding)?
    - Action: If yes, treat according to the severity of each injury; if not, focus on the fractured leg.
4. **Provide Pain Management**
    - Question: Is the patient experiencing significant pain?
    - Action: Administer acetaminophen (e.g., Tylenol) or ibuprofen (e.g., Advil) as needed for pain relief (consult local guidelines for dosing and contraindications).
5. **Transport to Medical Care**
    - Question: Is medical help available within a reasonable time frame?
    - Action: If yes, transport the patient; if not, establish a makeshift camp and prepare for prolonged care.
6. **Standard First-Line Care (for most patients)**
    - Elevate the affected limb above heart level to reduce swelling
    - Apply ice packs to the injury site to reduce pain and inflammation
    - Monitor vital signs (temperature, pulse, breathing rate, blood pressure)
7. **Patient-Specific Factors**
    - Consider the patient's medical history, allergies, and current medications when administering treatment
    - Be aware of any signs of compartment syndrome or nerve damage (e.g., numbness, tingling, pallor)

**Additional Considerations for Care and Recovery:**

- **Hydration**: Encourage the patient to drink plenty of water to stay hydrated.
- **Nutrition**: Provide a balanced diet with adequate protein, complex carbohydrates, and healthy fats to support healing.
- **Rest**: Ensure the patient gets sufficient rest and avoids overexertion.
- **Follow-up Care**: Schedule follow-up appointments with a healthcare professional as soon as possible after returning from the wilderness.

**Sources:**

* Wilderness Medical Society's Wilderness Medicine Position Statement (2018)
* American College of Surgeons' Committee on Trauma (ACS COT) Guidelines for Prehospital Care
* Local and regional guidelines for wilderness medicine and emergency medical services

Please note that this is a general clinical decision pathway, and specific treatment may vary depending on the patient's condition and local medical resources.

**Overall**

`system_prompt3` consistently produced the best‑structured, most organized, and most clinically‑aligned responses across all queries. It demonstrated superior clarity, decision‑pathway behavior, and adherence to the prompt. Therefore, it will be used moving forward, especially after integrating the medical manuals, as it provides the strongest foundation for RAG‑grounded clinical reasoning.

## Data Preparation for RAG

### Loading the Data

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [55]:
medical_path = "/content/drive/MyDrive/Python Course/medical_diagnosis_manual.pdf"

In [56]:
pdf_loader = PyMuPDFLoader(medical_path)
pages = pdf_loader.load()

### Data Overview

#### Removing email/personal info and watermark from showing in the data

In [57]:
# Defining a function to scrub personal information off the medical manuals
def clean_page_text(text):
    # Remove plain-text email + code + personal use sentence
    text = re.sub(
        r'\S+@\S+\.\S+\s*\n\S+\s*\nThis file is meant for personal use by \S+@\S+\.\S+ only\.',
        '',
        text
    )
    # Remove markdown-link-style email + code + personal use sentence
    text = re.sub(
        r'\[\S+@\S+\.\S+\]\(mailto:\S+@\S+\.\S+\)\s*\n\S+\s*\nThis file is meant for personal use by \[\S+@\S+\.\S+\]\(mailto:\S+@\S+\.\S+\) only\.',
        '',
        text
    )
    # Remove the legal-action sentence if present
    text = re.sub(r'Sharing or publishing the contents in part or full is liable for legal action\.', '', text)
    return text.strip()

In [58]:
# Cleaning manual with a sample
sample = pages[0].page_content
cleaned = clean_page_text(sample)
print(repr(cleaned))

''


In [59]:
# Scrubbing personal info from showing
for page in pages:
    page.page_content = clean_page_text(page.page_content)

In [60]:
# Check to see if watermark/personal info is still there
print("Still contains watermark:", any("personal use" in page.page_content for page in pages))

Still contains watermark: False


Upon loading the PDF, a personal watermark was discovered embedded throughout
the document, including my personal email address and a unique tracking code
on most pages.

The watermark was removed prior to chunking and embedding for two reasons:
first, to protect my personal information from appearing in notebook outputs, and second, to reduce noise in the RAG pipeline, since watermark text appearing mid-page would otherwise be embedded alongside clinical content and would interfere with retrieval quality.

A generic regex pattern was used for removal, no personal identifiers were
hardcoded into the notebook.

#### Checking the first 5 pages

In [61]:
# Iterating over the first 5 pages to display
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(pages[i].page_content,end="\n")

Page Number : 1

Page Number : 2

Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................................................................
53
1 - Nutritional Disorders    ...............................................................................................................................................................
53
Chapter 1. Nutrition: General Considerations    ....................................................

In [62]:
# Checking pages in the manual close to the middle and end, whilst only printing the first 500 characters of each to preview
print(pages[1500].page_content[:500])  # near middle
print('-'*50)
print(pages[-5].page_content[:500])    # near the end

Lymphocutaneous infections are most common. They characteristically involve one hand and arm,
although they can occur anywhere on the body; primary lesions may occur on exposed surfaces of the
feet or face.
A primary lesion may appear as a small, nontender papule or, occasionally, as a slowly expanding
subcutaneous nodule that eventually becomes necrotic and sometimes ulcerates. Typically, a few days or
weeks later, a chain of draining lymph nodes begins to enlarge slowly but progressively, form
--------------------------------------------------
Whiff test 2543
Whipple's disease 155, 161-162
rheumatoid arthritis vs 334
Whipple's operation 146, 199-200
Whipple's triad 199
Whipworm infection 1354-1355
White blood cells (see also specific cell types)
count of 3499
in ascites 210
in CSF 1594, 1595
in myelofibrosis 1000
in neonatal sepsis 2835
in occult bacteremia 2841
in pediatric urinary tract infection 2845
in pregnancy 2624
reduction in 948-954, 949 (see also Lymphocytopenia; Neutropeni

#### Checking the number of pages

In [63]:
# Total amount of pages in the manual
len(pages)

4114

In [64]:
# Checking length of characters in pages
lengths = [len(page.page_content) for page in pages]
# Average
print(f"Average page length: {sum(lengths)/len(lengths):.0f} characters")
# Min and Max amount of characters
print(f"Min: {min(lengths)}, Max: {max(lengths)}")

Average page length: 3146 characters
Min: 0, Max: 9417


The minimum amount of characters is found to be 0 on a single page, meaning there are pages without any text in it.

In [65]:
# Finding exactly how many empty pages there are
empty_pages = [i for i, doc in enumerate(pages) if len(doc.page_content.strip()) < 50]
print(f"Number of near-empty pages: {len(empty_pages)}")

Number of near-empty pages: 2


In [66]:
# Checking the metadata attached to the first page (total page count, source path, document title, etc)
print(pages[0].metadata)

{'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/drive/MyDrive/Python Course/medical_diagnosis_manual.pdf', 'file_path': '/content/drive/MyDrive/Python Course/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-05-28T17:14:38+00:00', 'trapped': '', 'modDate': 'D:20260528171438Z', 'creationDate': 'D:20120615054440Z', 'page': 0}


### Data Chunking

In [67]:
# Splitting manual into around 2 chunks per characters on average page
pages_chunk = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""]
)
# chunking the medical manual
chunks = pages_chunk.split_documents(pages)

print(f"Total chunks created: {len(chunks)}")
# checking the first chunk
print(chunks[0].page_content[:300])

Total chunks created: 11454
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    ...................................................


In [68]:
# Checking a chunk
print(chunks[710].page_content[-250:])

ed to maintenance levels. Oral 5-ASA drugs
theoretically have some incremental benefit in lessening the probability of proximal spread of disease.
Moderate or extensive disease: Patients with inflammation proximal to the splenic flexure or left-sided


In [69]:
# Checking overlap in chunk
print(chunks[711].page_content[:250])

Moderate or extensive disease: Patients with inflammation proximal to the splenic flexure or left-sided
disease unresponsive to topical agents should receive an oral 5-ASA formulation in addition to 5-ASA
enemas. High-dose corticosteroids are added f


In [70]:
# Verifying overlap is working correctly
chunks[710].page_content == chunks[711].page_content

False

After inspecting the manual's page statistics,: an average of 3,146 characters
per page with a maximum of 9,417, a chunk size of 1,500 characters was selected, producing roughly 2 chunks per average page. A chunk overlap of 200 characters (approximately 13% of chunk size) was chosen to preserve continuity across chunk boundaries without significantly increasing compute cost or storage.`RecursiveCharacterTextSplitter` was selected for its hierarchical splitting strategy: rather than applying a single separator, it attempts to split on larger semantic units first (paragraphs, then sentences, then words) and only falls back to smaller units when necessary to enforce the size limit. This preserves meaning over enforcing size, making it well-suited for medical text where splitting mid-sentence or mid-protocol step could corrupt the clinical meaning of a retrieved chunk.

Sentence-aware separators were added to the default configuration to further
reduce mid-sentence splits. The manual was split into 11,454 chunks total.

Chunk overlap was verified empirically: chunks 710 and 711 were inspected to
confirm that the end of one and the start of the next shared overlapping content. As expected, the two chunks were not identical, confirming that overlap is working as intended by carrying forward a portion of the previous chunk's text rather than duplicating entire chunks.

### Embedding

In [71]:
# Creating an instance of the embedding model
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

/tmp/ipykernel_1435/3030629551.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

 This model was chosen due to it being a large, high-quality sentence embedding model well-suited for semantic similarity search.

In [72]:
# Saving samples of the first two chunk embeddings to verify the model produces output
embedding_1 = embedding_model.embed_query(chunks[0].page_content)
embedding_2 = embedding_model.embed_query(chunks[1].page_content)

print(embedding_1[:5]) # preview first 5 values of embedding 1
print('_'*50)
print(embedding_2) # full embedding 2 for comparison

[-0.0237884521484375, 0.0113372802734375, 0.02386474609375, 0.007694244384765625, -0.040130615234375]
__________________________________________________
[-0.007297515869140625, -0.007595062255859375, 0.00830841064453125, 0.0176849365234375, -0.022247314453125, 0.0218048095703125, 0.0087738037109375, 0.0445556640625, 0.00732421875, 0.002796173095703125, 0.031280517578125, -0.00926971435546875, 0.023712158203125, -0.05078125, 0.002986907958984375, -0.00676727294921875, -0.0222320556640625, -0.0303802490234375, -0.01413726806640625, 0.0272064208984375, -0.0167999267578125, -0.0187530517578125, -0.03765869140625, -0.0073699951171875, -0.023773193359375, 0.0033397674560546875, 0.0013294219970703125, 0.0108642578125, 0.042572021484375, 0.044525146484375, -0.032745361328125, -0.0238189697265625, 0.044525146484375, -0.033599853515625, -0.024383544921875, -0.005615234375, 0.037750244140625, -0.054107666015625, -0.0157928466796875, -0.04010009765625, 0.020263671875, -0.0169525146484375, 0.022109

In [73]:
# Verifying both embeddings share the same vector dimension, required for consistency
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

In [74]:
# Displaying raw vector values, confirming embeddings are dense representations, and not identical
print(embedding_1, "___"*60, embedding_2)

[-0.0237884521484375, 0.0113372802734375, 0.02386474609375, 0.007694244384765625, -0.040130615234375, -0.00872039794921875, -0.039306640625, 0.0235137939453125, 0.039398193359375, 0.0219268798828125, 0.02716064453125, 0.0039215087890625, 0.018402099609375, -0.0295562744140625, -0.0165252685546875, -0.006778717041015625, -0.019287109375, -0.028961181640625, -0.0357666015625, 0.01468658447265625, 0.0222320556640625, 0.009674072265625, -0.05181884765625, -0.0283660888671875, -0.01483917236328125, 0.0258331298828125, 0.03466796875, 0.01361846923828125, 0.058807373046875, 0.071533203125, -0.01128387451171875, -0.037811279296875, 0.027435302734375, -0.06768798828125, -0.0189666748046875, -0.0073089599609375, 0.03570556640625, -0.037811279296875, -0.023193359375, -0.048736572265625, 0.00733184814453125, -0.015289306640625, 0.0239410400390625, -0.018768310546875, -0.0292510986328125, -0.00925445556640625, -0.00519561767578125, -0.00787353515625, -0.020538330078125, -0.0234832763671875, 0.00651

Before embedding all 11,454 chunks into the vector database, the embedding model is instantiated and verified on two sample chunks. This confirms the model loads correctly, produces output of the expected dimensionality, and generates distinct vector representations for different chunks.

The model selected is `thenlper/gte-large` (General Text Embeddings, large
variant), a sentence transformer optimized for semantic similarity tasks.
It produces 1,024-dimensional dense vectors, so each chunk is represented
as a list of 1,024 floating point numbers.



### Vector Database

In [75]:
# Creating directory to store vectorized chunked medical data
out_dir = 'pages_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [76]:
# Building Chroma vectorstore by embedding all chunks
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    persist_directory=out_dir
)

In [77]:
# Loading the persisted vector database
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

In [78]:
# Confirming the embedding function attached to the vectorstore
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [79]:
# Verifying vector count matches chunk count, confirming all chunks were embedded successfully
print(f"Total vectors in database: {vectorstore._collection.count()}")
print(f"Total Chunks in split dataset: {len(chunks)}")

Total vectors in database: 11454
Total Chunks in split dataset: 11454


In [80]:
# Similarity search with scores (k=10) to assess retrieval quality and identify relevance drop-off point
# Query 1 — sepsis management
results_with_scores = vectorstore.similarity_search_with_score(
    "What is the protocol for managing sepsis in a critical care unit?",
    k=10
)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"Result {i} | Score: {score:.4f} | Page: {doc.metadata['page']}")
    print(doc.page_content[:150])
    print()

Result 1 | Score: 0.2611 | Page: 2400
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most

Result 2 | Score: 0.2742 | Page: 2455
depending on patient characteristics). Poor outcomes often follow failure to institute early aggressive
therapy (eg, within 6 h of suspected diagnosis

Result 3 | Score: 0.2798 | Page: 2456
Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been
taken for Gram stain and culture. Very prompt 

Result 4 | Score: 0.2948 | Page: 2455
L over 4 to 12 h. PAOP or echocardiography can identify limitations in left ventricular function and
incipient pulmonary edema due to fluid overload.


Result 5 | Score: 0.3045 | Page: 2995
suspected, ampicillin, cefotaxime, and an aminoglycoside may be used. In late-onset hospital-acquired
sepsis, initial therapy should include vancomyci

Result 6 | Score: 0.3113 | Page: 2456
continued fo

Based on the 10 most similar results from the query to the manual, the top 7 seem to have the most relevant information to sepsis management. The similarity scores are tightly clustered ranging between 0.26 and 0.32.

In [81]:
# Query 2 - appendicitis treatments and surgical procedures
results_with_scores = vectorstore.similarity_search_with_score(
    "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    k=10
)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"Result {i} | Score: {score:.4f} | Page: {doc.metadata['page']}")
    print(doc.page_content[:150])
    print()

Result 1 | Score: 0.2422 | Page: 174
• Surgical removal
• IV fluids and antibiotics
Treatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay
increases

Result 2 | Score: 0.2485 | Page: 173
laparotomy.
Laparoscopy can be used for diagnosis as well as definitive treatment; it may be especially helpful in
women with lower abdominal pain of 

Result 3 | Score: 0.2735 | Page: 172
nondiagnostic, abdominal CT usually with oral and IV and/or rectal contrast may be helpful. Barium
should not be used if perforation is suspected.
Tre

Result 4 | Score: 0.2818 | Page: 167
Chapter 11. Acute Abdomen and Surgical Gastroenterology
Introduction
Acute abdomen refers to abdominal symptoms and signs of such severity or concern 

Result 5 | Score: 0.2975 | Page: 173
Etiology
Appendicitis is thought to result from obstruction of the appendiceal lumen, typically by lymphoid
hyperplasia, but occasionally by a fecalit

Result 6 | Score: 0.3092 | Page: 173
occur. Pain may no

In this similarity search, the top 6 results are directly related to the query, the rest of the results seem to drift off showing cancer, head trauma, pregnancy, and cosmetic concerns; which are irrelavant to the query.

In [82]:
# Query 3 - sudden hairloss
results_with_scores = vectorstore.similarity_search_with_score(
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    k=10
)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"Result {i} | Score: {score:.4f} | Page: {doc.metadata['page']}")
    print(doc.page_content[:150])
    print()

Result 1 | Score: 0.2425 | Page: 858
• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the
underlying disorder.
• Microscopic hair examinat

Result 2 | Score: 0.2644 | Page: 858
systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or
squaric acid dibutylester), or psoralen plus u

Result 3 | Score: 0.2651 | Page: 857
lost in the first morning combing or during washing are collected in clear plastic bags daily for 14 days.
The number of hairs in each bag is then rec

Result 4 | Score: 0.2915 | Page: 858
loss and can stimulate hair
[Table 86-2. Interpreting Findings in Alopecia]
growth. Efficacy is usually evident within 6 to 8 mo of treatment. Adverse

Result 5 | Score: 0.2974 | Page: 855
Chapter 86. Hair Disorders
Introduction
Hair growth in both men and women is regulated by androgens. Testosterone stimulates hair growth in
the pubic 

Result 6 | Score: 0.2981 | Page: 855
more than 100 hair

With these results the same could be said as before. The top 6 results stay relevant, even though the similarity scores are very tightly clustered it starts to lose its relevance to the query at 7 and so on.

In [83]:
# Query 4 - traumatic brain injury
results_with_scores = vectorstore.similarity_search_with_score(
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    k=10
)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"Result {i} | Score: {score:.4f} | Page: {doc.metadata['page']}")
    print(doc.page_content[:150])
    print()

Result 1 | Score: 0.2310 | Page: 3404
Chapter 324. Traumatic Brain Injury
Introduction
Traumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently
impai

Result 2 | Score: 0.2352 | Page: 3647
p. 3231). Such intervention includes prevention of secondary disabilities (eg, pressure ulcers, joint
contractures), prevention of pneumonia, and fami

Result 3 | Score: 0.2354 | Page: 3409
regardless of severity and continue to improve for a longer period of time.
Cognitive deficits, with impaired concentration, attention, and memory, an

Result 4 | Score: 0.2604 | Page: 3411
variety of neuroprotective agents are being studied, but thus far, none has demonstrated efficacy in
clinical trials.
Seizures: Seizures can worsen br

Result 5 | Score: 0.2646 | Page: 1820
appear beneficial. Physical and occupational therapy may modestly improve functioning but is more often
useful for making the environment safer and fo

Result 6 | Score: 0.2684 | Page: 1814
Larger lesio

For this one, it is hard to say without any domain knowledge, but results 4-5 seem to stay on track, and the rest seem to trail off on different topics.

In [84]:
# Query 5 - leg fracture
results_with_scores = vectorstore.similarity_search_with_score(
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
    k=10
)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"Result {i} | Score: {score:.4f} | Page: {doc.metadata['page']}")
    print(doc.page_content[:150])
    print()

Result 1 | Score: 0.2996 | Page: 3465
care because thawed tissue is particularly sensitive to the trauma of walking and, if refrozen, will be more
severely damaged than if left frozen. If 

Result 2 | Score: 0.3007 | Page: 3390
open reduction (with skin incision) is done. Closed reduction
[Table 323-1. Examination for Some Commonly Missed Injuries]
of fractures is usually mai

Result 3 | Score: 0.3084 | Page: 2384
gangrene.
Prevention
• Prevention of immobility
• Assessment of risk
• Anticoagulation (eg, LMWH, fondaparinux, adjusted-dose warfarin)
• Intermittent

Result 4 | Score: 0.3122 | Page: 3391
from injuring the skin
• To seek medical care at once if an odor emanates from within the cast or if a fever, which may indicate
infection, develops
G

Result 5 | Score: 0.3122 | Page: 3393
open (for irrigation and debridement to prevent infection). Open reduction may be done without using
hardware when closed reduction is ineffective.
Fr

Result 6 | Score: 0.3135 | Page: 3386
crutches); r

Not sure if the first result relevant, but results 2, 4, 5, and 6 seem to have the highest relevance, while results 3, 7, 8, and 10, are moderatley. 9 seems to be noise, with 9; having "antivenom" in the document retrieved which is completely off topic which a fractured leg.

**Overall**

Despite the tightly clustered similarity scores across k=10 results, content
relevance visibly degrades beyond the top 5-6 results when inspecting the actual retrieved text, with results 7 and beyond returning topically adjacent but clinically irrelevant chunks. This reflects a known characteristic of
embedding-based retrieval: cosine distance scores alone do not reliably indicate a hard relevance boundary, making qualitative inspection of retrieved content necessary to complement the numeric scores.

The core tradeoff is between precision and recall. A lower k reduces noise but
risks missing relevant content spread across multiple pages of the manual,
particularly for complex queries where treatment steps, diagnostic criteria, and drug information may appear in separate sections. A higher k improves coverage but introduces irrelevant chunks that can dilute the model's retrieved context and reduce groundedness.

After inspecting retrieved content across all five queries, k=5 was selected as
the initial retriever configuration, balancing coverage against noise.

### Retriever

In [85]:
# Convert vectorstore into retriever object
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [86]:
# Query 1 retriever
rel_docs1 = retriever.invoke("What is the protocol for managing sepsis in a critical care unit?")

# Displaying each retrieved chunk, veryfing meaningfulness and appropriate conxtext for reasoning
for i, doc in enumerate(rel_docs1, 1):
    print(f"--- Result {i} (Page {doc.metadata['page']}) ---")
    print(doc.page_content[:-1])
    print()

--- Result 1 (Page 2400) ---
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring
of physiologic parameters.
Supportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of
infection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to
25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help
dying patients maintain dignity (see p. 3480).
Patient Monitoring and Testing
Some monitoring is manual (ie, by direct obs

After inspecting the results even more, it was surprising to see that the 1st result was somewhat irrelavant. But the rest of the chunks are, and cover the core components of sepsis management. While there is some noise in the retrieved chunks, it is good to see that retrieval is working, and embeddings are correctly clustering.

In [87]:
# Query 2
rel_docs2 = retriever.invoke("What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?")

for i, doc in enumerate(rel_docs2, 1):
    print(f"--- Result {i} (Page {doc.metadata['page']}) ---")
    print(doc.page_content[:-1])
    print()

--- Result 1 (Page 174) ---
• Surgical removal
• IV fluids and antibiotics
Treatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay
increases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can
usually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these
cases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to
appendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis
and a normal cecum, the appendix should be removed.
Appendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For
nonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics
should be continued until the patient's temperature and WBC count have normalized or continued for a
fixed course, according to the surgeon's preferenc

All of the retrieved chunks collectively form a coherent answer to the query. It seems to have low to no noise retrieved.

In [88]:
# Query 3
rel_docs3 = retriever.invoke("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?")

for i, doc in enumerate(rel_docs3, 1):
    print(f"--- Result {i} (Page {doc.metadata['page']}) ---")
    print(doc.page_content[:-1])
    print()

--- Result 1 (Page 858) ---
• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the
underlying disorder.
• Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.
Alopecia Areata
Alopecia areata is sudden patchy hair loss in people with no obvious skin or systemic disorder.
The scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may
affect most or all of the body (alopecia universalis). Alopecia areata is thought to be an autoimmune
disorder affecting genetically susceptible people exposed to unclear environmental triggers, such as
The Merck Manual of Diagnosis & Therapy, 19th Edition
Chapter 86. Hair Disorders
84

--- Result 2 (Page 858) ---
systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or
squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA).
Treatment for traction alopecia is elimination of physical tr

Like before, all of the chunks are coherent, relevant, and collectively form a proper answer.

In [89]:
# Query 4
rel_docs4 = retriever.invoke("What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?")

for i, doc in enumerate(rel_docs4, 1):
    print(f"--- Result {i} (Page {doc.metadata['page']}) ---")
    print(doc.page_content[:-1])
    print()

--- Result 1 (Page 3404) ---
Chapter 324. Traumatic Brain Injury
Introduction
Traumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently
impairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily
CT). Initial treatment consists of ensuring a reliable airway and maintaining adequate
ventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more
severe injury to place monitors to track and treat intracranial pressure, decompress the brain if
intracranial pressure is increased, or remove intracranial hematomas. In the first few days after
the injury, maintaining adequate brain perfusion and oxygenation and preventing complications
of altered sensorium are important. Subsequently, many patients require rehabilitation.
In the US, as in much of the world, TBI is a common cause of death and disability. Causes include motor
vehicle crashes and other transportation-related causes (eg, bicycle cr

With these results, 1, 3, 4, and 5 seem to be doing what they're supposed to, but result 6 is showing treatments for lesions, which is not relevant for the query about brain trauma. And result 2 is shwoing results for spinal cord injury. There was more noise retrieved for this query compare to the others.

In [90]:
# Query 5
rel_docs5 = retriever.invoke("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?")

for i, doc in enumerate(rel_docs5, 1):
    print(f"--- Result {i} (Page {doc.metadata['page']}) ---")
    print(doc.page_content[:-1])
    print()

--- Result 1 (Page 3465) ---
care because thawed tissue is particularly sensitive to the trauma of walking and, if refrozen, will be more
severely damaged than if left frozen. If thawing must be delayed, the frozen area is gently cleaned, dried,
and protected in sterile compresses. Patients are given analgesics, if available, and the whole body is
kept warm.
Acute care: Once the patient is in the hospital, core temperature is stabilized and extremities are rapidly
rewarmed in large containers of circulating water kept at about 40.5° C; 15 to 30 min is usually adequate.
Thawing is often mistakenly ended prematurely because pain may be severe during rewarming.
Parenteral analgesics, including opioids, may be used. Patients are encouraged to move the affected part
The Merck Manual of Diagnosis & Therapy, 19th Edition
Chapter 333. Cold Injury
345

--- Result 2 (Page 3390) ---
open reduction (with skin incision) is done. Closed reduction
[Table 323-1. Examination for Some Commonly Missed In

Here the best results came from 2, 4, and 5. Seems that the retrieval caught some noise here as well. Dependant on the query, the retrieval model struggles sometimes with providing the most relevant results. Result 1 surprisingly was about frostbite, result 3 about DVT, and 6 about lacerations.

**Overall**

After a deeper dive into the retrieved chunks with `top_k` = 6, the safest choice would be to change it to 5 instead, with it being any higher the model retrieves significant noise (especially dependant on the query) and surprisingly with it sometimes being the top result.

### System and User Prompt Template

In [91]:
system_message = """
You are an AI medical assistant that supports clinicians with rapid, time‑sensitive decision‑making.
You must answer ONLY using the information provided in the retrieved medical manual context.

RULES FOR USING CONTEXT
- User input will contain a section beginning with:  ###Context
- The clinical question will begin with:  ###Question
- If the question is ambiguous or missing key details needed for safe reasoning, ask clarifying questions.
- You must answer only using the information found in the ###Context.
- If the answer is not fully supported by the context, respond exactly with:  "There is not enough information to answer the query fully".
- Do not use outside medical knowledge.
- Do not invent guidelines, drug doses, diagnostic criteria, or clinical steps.
- Do not mention the context or describe how you used it.

RESPONSE FORMAT:
When answering treatment or management questions, structure your response as a clinical decision pathway:
- Present actions in the order they should be performed.
- At each decision point, state the clinical question being assessed and the resulting action for each possible outcome.
- Clearly separate:
  • Standard first‑line care
  • Patient‑specific factors that may require deviation
- Be concise and prioritize actionable steps over explanation.
- Your response should immediately follow the ###Answer token with no preamble.
- At the end of your answer, include a section titled "Sources" listing which context chunks you used.
"""

In [92]:
message_template = """
###Context
{context}

###Question
{question}

###Answer
"""

### Response Function

In [93]:
def generate_rag_response(question, system_prompt=None, k=5, params=None):

    # Retrieve relevant document chunks and build retriever with k baked in
    retriever = vectorstore.as_retriever(search_kwargs={'k': k})
    relevant_document_chunks = retriever.invoke(question)

    # Extract text content from each retrieved chunk
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context block
    context_for_query = "\n\n".join(context_list)

    user_message = message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', question)

    # Set generation defaults
    config = {
        "max_tokens": 1536,
        "temperature": 0,
        "top_p": 0.95,
        "top_k": 50,
        "repeat_penalty": 1.1,
        "seed": 42,
        "stop": ["<|eot_id|>", "Q:", "\n\n\n"]
    }

    # Override defaults with any params passed in at call time
    if params:
        config.update(params)

    # Build messages list — use system_prompt if provided, otherwise fall back to system_message
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    elif system_message:
        messages.append({"role": "system", "content": system_message})
    messages.append({"role": "user", "content": user_message})

    # Generate response using chat completion to apply Llama 3.1 chat template
    try:
        model_output = llm.create_chat_completion(
            messages=messages,
            **config
        )

        # Extract and print the model's response
        response = model_output['choices'][0]['message']['content'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [94]:
# Q&A RAG Query 1
display(Markdown(generate_rag_response(question1)))

Llama.generate: 32 prefix-match hit, remaining 1935 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2549.01 ms /  1935 tokens (    1.32 ms per token,   759.12 tokens per second)
llama_perf_context_print:        eval time =   18567.31 ms /   388 runs   (   47.85 ms per token,    20.90 tokens per second)
llama_perf_context_print:       total time =   22243.35 ms /  2323 tokens
llama_perf_context_print:    graphs reused =        385


**Fluid Resuscitation**

1. Assess the patient's volume status and hemodynamic stability.
2. If hypotensive, administer 0.9% normal saline for fluid resuscitation until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.

**Standard First-Line Care**

1. Monitor systemic pressure; CVP, PAOP, or both; pulse oximetry; ABGs; blood glucose, lactate, and electrolyte levels; renal function.
2. Administer O2 by mask or nasal prongs.
3. Consider tracheal intubation and mechanical ventilation for respiratory failure.

**Patient-Specific Factors**

1. If the patient remains hypotensive after CVP or PAOP has been raised to target levels, consider dopamine (20 μg/kg/min) to increase mean BP to at least 60 mm Hg.
2. If dopamine dose exceeds 20 μg/kg/min, add another vasopressor, typically norepinephrine.

**Antibiotic Therapy**

1. Administer broad-spectrum antibiotics (modified by culture results).
2. Consider gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day) for septic shock of unknown cause.
3. Change antibiotic regimen according to culture and sensitivity results.

**Abscess Drainage and Necrotic Tissue Excision**

1. Drain abscesses and excise necrotic tissue (e.g., infarcted bowel, gangrenous gall-bladder).

**Sources**
- Chapter 222. Approach to the Critically Ill Patient
- Chapter 227. Sepsis & Septic Shock

This response represents a clear improvement over all previous outputs. The base LLM responses (No RAG, no system prompt) were clinically structured but relied entirely on pretraining data. They produced plausible-sounding protocols but with no verifiable sources. They generated plausible‑sounding protocols but lacked verifiable sourcing, making them unsuitable for clinical decision support. In contrast, this response demonstrates a shift from generic correctness to specific, source‑grounded accuracy: it cites explicit Merck Manual chapters and includes verifiable clinical thresholds (ex: dopamine 20 μg/kg/min, cefotaxime 2 g q6–8 h, CVP targets) that directly match the retrieved text. None of the earlier responses were able to do this.

Compared to the `system_prompt3` responses (without RAG), the structural strengths remain intact. The model still produces a clear clinical decision‑pathway format, separating standard care from patient specific factors; which is exactly what `system_prompt3` was designed to enforce. What changes here is that the structure is now paired with traceable, evidence‑based content, because the system prompt includes “Provide source of information where applicable,” the model can finally cite real chapters rather than generating vague references. This shows how `system_prompt3` and RAG complement each other: the prompt enforces format and reasoning style, while retrieval supplies the clinical details.

The main limitation would be that this respomnse skips earlier steps such as recognition that appeared in some of the broader responses. This is because of the retrieval artifact, where the model answers strictly from the chunks recieved, which happen to start at the fluid resuscitation stage. Which is better from a clinical perspective: retriveing quality answeres, the shorter, but more grounded answer would be more operationally useful than a longer more generic response. Chunk boundaries, k‑value, and retrieval granularity directly shape what the model can and cannot include. Earlier responses were more sequentially complete, but they were also less grounded because they relied on pretraining rather than retrieved evidence.


Despite being shorter and more focused than the `temperature`=0.7 outputs, this response is more operationally useful in a clinical setting. It prioritizes verifiable depth over breadth, which is exactly what clinicians need when making time‑sensitive decisions. A concise answer that cites the Merck Manual and includes exact dosing thresholds is far more trustworthy than a longer, generically correct response with no attribution. This is the core value proposition of RAG: it transforms a plausible‑sounding model into a source‑grounded, clinically reliable system.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [95]:
# Q&A RAG Query 2
display(Markdown(generate_rag_response(question2)))

Llama.generate: 321 prefix-match hit, remaining 1524 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2089.51 ms /  1524 tokens (    1.37 ms per token,   729.36 tokens per second)
llama_perf_context_print:        eval time =   19082.56 ms /   403 runs   (   47.35 ms per token,    21.12 tokens per second)
llama_perf_context_print:       total time =   22380.52 ms /  1927 tokens
llama_perf_context_print:    graphs reused =        401


**Clinical Decision Pathway**

1. **Assess symptoms**: Is the patient experiencing epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia?
	* If yes, proceed to assess for right lower quadrant pain.
	* If no, consider alternative diagnoses.
2. **Assess right lower quadrant pain**: Does the pain shift to the right lower quadrant after a few hours?
	* If yes, proceed to assess for tenderness at McBurney's point.
	* If no, consider alternative diagnoses.
3. **Assess tenderness at McBurney's point**: Is there direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine)?
	* If yes, proceed to surgical treatment.
	* If no, consider alternative diagnoses.

**Standard First-Line Care**

1. **Surgical removal**: Perform an open or laparoscopic appendectomy as soon as possible.
2. **IV fluids and antibiotics**: Administer third-generation cephalosporins (e.g., cefotetan 1 to 2 g bid) before surgery, and continue until the patient's temperature and WBC count have normalized.

**Patient-Specific Factors**

* If the appendix is perforated, consider immediate surgery.
* If an abscess or inflammatory mass has formed, consider drainage of the abscess.
* If a large inflammatory mass involves the appendix, terminal ileum, and cecum, consider resection of the entire mass and ileocolostomy.

**Sources**

* Chapter 11. Acute Abdomen & Surgical Gastroenterology
* Treatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay increases mortality, a negative appendectomy rate of 15% is considered acceptable.
* Appendicitis can be cured via surgical removal of the appendix, but not via medicine alone.

This is a very strong RAG output, better than the query 1 result. The decision tree format worked better for this query than it did in the all of the other previous responses. This output starts with actual diagnostic logic: symptom assesment to location shift to tenderness point, to surgical decision. From the looks of it this is a genuine clinical decision pathway. While this response might not have the sources cited, when compared to the retrieval outputs earlier in the notebook, it directly quoted this chunk: "because treatment delay increases mortality, a negative appendectomy rate of 15% is considered acceptable". It is pulling specific, precise statements that directly answer the query.

Compared to RAG Q&A for query 1, this output has a much better structure/sequential completness. Starting at diagnosis and then goes into treatment, with the right if then formatting. This could be due to the manual having a more linear/clear text on apendicitis, than sepsis does.

Comparing this output vs the base LLM, the base correctly stated that apendicitis requires surgery, but this output adds specifics such as "cefotetan 1-2g bid, the 15% negative appendectomy rate, the specific anatomical description of McBurney's point", which is even better than a yes/no answer. When looking back to the prompt engineered responses for this query, it was noted that it had a good decision point structure but it was quite oversimplied. By injecting RAG the response improved significantly better by provind specific thresholds, and contigency branches that the prompt only versions lacked.

The only thing to note that is a bit inconsistent would be the sources section. The contetnt itself is valuable, but unlike query 1 response this answer did not cleanly seperate where it came from.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [96]:
# Q&A RAG Query 3
display(Markdown(generate_rag_response(question3)))

Llama.generate: 322 prefix-match hit, remaining 1522 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1992.54 ms /  1522 tokens (    1.31 ms per token,   763.85 tokens per second)
llama_perf_context_print:        eval time =   13377.36 ms /   309 runs   (   43.29 ms per token,    23.10 tokens per second)
llama_perf_context_print:       total time =   16218.40 ms /  1831 tokens
llama_perf_context_print:    graphs reused =        307


**Clinical Decision Pathway**

1. **Assess the patient's symptoms**: Is the hair loss sudden and patchy, with localized bald spots on the scalp?
	* If yes, proceed to step 2.
	* If no, consider other causes of hair loss (e.g., traction alopecia, tinea capitis).
2. **Determine if there are any underlying skin or systemic disorders**: Is concomitant virilization present in women or scarring hair loss?
	* If yes, a thorough evaluation for the underlying disorder is necessary.
3. **Consider autoimmune causes**: Could the patient have alopecia areata, an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers?
4. **Evaluate the need for microscopic examination or scalp biopsy**: Is there a need for definitive diagnosis of scarring hair loss or other conditions?
5. **Offer treatment options**:
	* For alopecia areata: topical corticosteroids, intralesional corticosteroids, or systemic corticosteroids (in severe cases).
	* For androgenetic alopecia: minoxidil 2% for women, 2% or 5% for men.
6. **Consider patient-specific factors**: Are there any contraindications to treatment (e.g., pregnancy, teratogenic effects in animals)?
7. **Monitor progress and adjust treatment as needed**.

**Sources**
* Chapter 86. Hair Disorders
* Table 86-2. Interpreting Findings in Alopecia

This response is very interesting because while query 2 answer had the right branching steps, this answer goes even further in differentiang the types of hair loss before going into treatment. This output shows the model decided to stray away from the if-then branching format and instead goes into instructional prose. This is likely because the retrieved chunks for this topic were less structured in the source manual, hair loss/disorders most likely don't have the same clean algorithmic presentation that sepsis and appendicitis protocols do, so the model had less structured content to mirror.

The base LLM and this response both identified alopecia areata as an autoimmune condition, but the step 5 in this response gives treatment options: "topical corticosteroids, intralesional corticosteroids, or systemic corticosteroids (in severe cases) - For androgenetic alopecia: minoxidil 2% for women, 2% or 5% for men.", which the baseline LLM couldn't source.

One weakness to point out would be that the query specifically asked what possible causes are for hair loss and treatments while step 3 mentions autoimmune as a cause, and step 6 briefly mentions androgenetic alopecia, there's no organized causes section, which could be inferred but isn't as clear as the previous answers were before RAG injection. The RAG response is more precise on treatment but actually less complete on the causal explanation side.

Without domain knowledge it is difficult to assess whether the missing causal
information represents a clinically significant gap, this is where the evaluation scores will be most informative for this query.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [97]:
# Q&A RAG Query 4
display(Markdown(generate_rag_response(question4)))

Llama.generate: 321 prefix-match hit, remaining 1114 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1428.30 ms /  1114 tokens (    1.28 ms per token,   779.95 tokens per second)
llama_perf_context_print:        eval time =   16669.03 ms /   390 runs   (   42.74 ms per token,    23.40 tokens per second)
llama_perf_context_print:       total time =   19121.68 ms /  1504 tokens
llama_perf_context_print:    graphs reused =        387


**Initial Management**

1. Ensure a reliable airway and maintain adequate ventilation, oxygenation, and blood pressure.
2. Assess for multiple noncranial injuries that may require simultaneous treatment.

**Assessment of Severity**

Clinical question: What is the patient's Glasgow Coma Scale (GCS) score?
- If GCS score ≥ 10: 
  - Clinical question: Are there any significant structural injuries (e.g., larger contusions or hematomas, brain laceration, depressed skull fracture)?
    - If yes:
      - Administer a prophylactic anticonvulsant (consider phenytoin with a loading dose of 20 mg/kg IV at a maximum rate of 50 mg/min).
    - If no: 
      - Clinical question: Are seizures present?
        - If yes: Treat seizures promptly.
- If GCS score < 10:
  - Administer a prophylactic anticonvulsant (consider phenytoin with a loading dose of 20 mg/kg IV at a maximum rate of 50 mg/min).

**Standard First-Line Care**

1. Surgery may be needed to place monitors, decompress the brain, or remove intracranial hematomas.
2. Maintain adequate brain perfusion and oxygenation.
3. Prevent complications of altered sensorium.

**Patient-Specific Factors**

- For patients with severe cognitive dysfunction:
  - Consider extensive cognitive therapy, which may begin immediately after injury and continue for months or years.
- For patients with spinal cord injuries:
  - Immobilize the affected area surgically or nonsurgically as soon as possible and throughout the acute phase.
  - Prevent contractures, pressure ulcers, and pneumonia.

**Sources**
Chapter 324. Traumatic Brain Injury
p. 3231
Table 325-1 on p. 3228 (Spinal cord injury)
p. 1805

What stands out the ost in this utput is the if-then decision tree pathway structure is the most clear and sophisticated from all the other previous responses. Even better than RAG Q&A Query 1-3. The branching logic: "If GCS score ≥ 10" "If GCS score < 10", using the Glasgow Coma Scale as a decision point then further breaks down what to do for each decision point. This emerging from retrieved content is significant because even the previous responses could not generate this from memory.

There is a bit of noise added in this response, specifically for patient-specific factors, it included a bullet for patients with spinal cord injuries, while maybe be useful for some cases, it ay not be clinically relevant context for brain impairment treatments.

The sources section is the strongests of all the previous Q&A, it cited the chapter, a specific page number, a specific table, and a specific cross-reference page. This is a qualitative jump from the chapter-only citations in queries 1 and 2. This is perfect for cross refrencing or more info retrieval from the manuals since it specifically shows where the relevant chunks are, if more info is needed it would be easy to jump to those chunks and get more.

Across all four queries analyzed so far, a clear pattern has emerged: RAG performance is strongest when the underlying Merck Manual content is algorithmically structured, and weakest when content is distributed across multiple sections or is more descriptive than procedural (hair loss). This is a meaningful finding for the business context, the RAG system is most reliable for protocol-driven queries and may need supplementary retrieval strategies for broader diagnostic questions.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [98]:
# Q&A RAG Query 5
display(Markdown(generate_rag_response(question5)))

Llama.generate: 321 prefix-match hit, remaining 1566 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2073.06 ms /  1566 tokens (    1.32 ms per token,   755.41 tokens per second)
llama_perf_context_print:        eval time =   19805.76 ms /   436 runs   (   45.43 ms per token,    22.01 tokens per second)
llama_perf_context_print:       total time =   23179.32 ms /  2002 tokens
llama_perf_context_print:    graphs reused =        433


**Acute Care**

1. **Stabilize core temperature**: Keep the patient warm.
2. **Assess and manage pain**: Administer parenteral analgesics, including opioids if necessary.
3. **Rapidly rewarn extremities**: Use circulating water at 40.5° C for 15-30 minutes.

**Immobilization**

1. **Apply a splint or cast**: Immobilize the affected limb to prevent further injury and facilitate healing.
2. **Elevate the injured limb**: Above the heart for the first 2 days to minimize swelling.
3. **Use RICE principles**: Rest, ice (for 15-20 minutes intermittently), compression, and elevation.

**Fracture-Specific Care**

1. **Immobilize proximal and distal joints**: Prevent further injury and facilitate healing.
2. **Consider early mobilization**: Resume active motion within the first few days or weeks to minimize contractures and muscle atrophy.

**Geriatric Considerations**

1. **Rapid return to activities of daily living**: Prioritize functional recovery over perfect limb alignment and length.
2. **Use of ORIF (Open Reduction Internal Fixation)**: May be considered due to increased risk of adverse effects from immobilization in the elderly.
3. **Early mobilization and physical therapy**: Encourage early movement and rehabilitation.

**Prevention of Complications**

1. **Monitor for signs of infection**: Seek medical care immediately if an odor emanates from within the cast or a fever develops.
2. **Encourage good hygiene**: Prevent skin irritation and promote healing.
3. **Consider DVT prophylaxis**: Assess risk factors and provide appropriate preventive treatment.

**Sources**
- Chapter 333. Cold Injury
- Table 323-1. Examination for Some Commonly Missed Injuries
- RICE: Patients who have soft-tissue injuries, with or without musculoskeletal injuries, benefit from RICE (rest, ice, compression, elevation)
- Immobilization: Immobilization decreases pain and facilitates healing by preventing further injury and is helpful except for very rapidly healing injuries.

Right away, the first thing is that the model retrieved "acute care" with irrelavant information on how to treat a patient with a fracture. It states that the patient needs to warm and use hot circulating water, which is not the proper treatment for a fractured leg. The model incorporated frostbite rewarming protocol into a response about a fractured leg because the retrieved chunk scored highly enough to be included, which was noted in the retrieval section.

Despite the noise in the begining of the response, the rest of the answer is clinically coherent for most of the length, relevant to the query. Without clinical knowledge it is difficult to determine whether rewarming guidance
would ever be clinically relevant in a hiking fracture scenario, but the source citation makes the contamination traceable and documentable, an advantage of the sourcing instruction.

Across all five queries, a consistent pattern emerges: the RAG pipeline performs best when retrieved content is topically precise and algorithmically structured, and introduces noise when query vocabulary overlaps with
unrelated clinical topics or when relevant content is distributed across sections that k=5 cannot fully capture. These findings directly inform two concrete improvements for a production system: a re-ranking step after retrieval to filter low-relevance chunks before they reach the model, and query expansion or decomposition to improve recall on multi-part or broadly-worded clinical questions.

### Fine-tuning

In [99]:
# No system prompt baseline
no_prompt_configs = [
    {"params": {"temperature": 0}},
    {"params": {"temperature": 0.7}},
    {"params": {"temperature": 0.5, "top_p": 0.85, "top_k": 40}},
    {"params": {"temperature": 0}, "k": 3},
    {"params": {"temperature": 0}, "k": 7},
]

for i, cfg in enumerate(no_prompt_configs, 1):
    print(f"--- No Prompt | Combination {i} ---")
    k = cfg.get("k", 5)
    answer = generate_rag_response(question1, params=cfg["params"], k=k)
    display(Markdown(answer))
    print()

Llama.generate: 321 prefix-match hit, remaining 1646 prompt tokens to eval


--- No Prompt | Combination 1 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2190.14 ms /  1646 tokens (    1.33 ms per token,   751.55 tokens per second)
llama_perf_context_print:        eval time =   13055.05 ms /   282 runs   (   46.29 ms per token,    21.60 tokens per second)
llama_perf_context_print:       total time =   16040.73 ms /  1928 tokens
llama_perf_context_print:    graphs reused =        280


**Fluid Resuscitation**

1. Assess the patient's volume status and hemodynamic stability.
2. If hypotensive, administer 0.9% normal saline for fluid resuscitation until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.

**Standard First-Line Care**

1. Monitor systemic pressure; CVP, PAOP, or both; pulse oximetry; ABGs; blood glucose, lactate, and electrolyte levels; renal function.
2. Administer broad-spectrum antibiotics (modified by culture results) after specimens of blood, body fluids, and wound sites have been taken for Gram stain and culture.

**Patient-Specific Factors**

1. If a patient with septic shock remains hypotensive after CVP or PAOP has been raised to target levels, consider adding dopamine to increase mean BP to at least 60 mm Hg.
2. If dopamine dose exceeds 20 μg/kg/min, add another vasopressor, typically norepinephrine.

**Oxygenation**

1. Administer O2 by mask or nasal prongs.
2. Consider tracheal intubation and mechanical ventilation for respiratory failure.

**Sources**
- Chapter 222. Approach to the Critically Ill Patient
- Chapter 227. Sepsis & Septic Shock

Llama.generate: 1966 prefix-match hit, remaining 1 prompt tokens to eval


...

**Fluid Resuscitation**
- Assess the patient's need for fluid resuscitation.
  • Clinical question: Is the patient hypotensive or oliguric?
    - If yes, proceed to standard first-line care. 
      Fluid resuscitation with 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
    - If no, consider patient-specific factors that may require deviation from standard care.

**Parenteral Antibiotics**
- Assess the need for parenteral antibiotics.
  • Clinical question: Have specimens of blood, body fluids, and wound sites been taken for Gram stain and culture?
    - If yes, proceed to standard first-line care. 
      Parenteral antibiotics should be given after specimens have been taken.
    - If no, consider patient-specific factors that may require deviation from standard care.

**Antibiotic Selection**
- Assess the need for antibiotic selection based on suspected source of sepsis.
  • Clinical question: Is the cause of sepsis unknown?
    - If yes, proceed to standard first-line care. 
      One regimen is gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day).
    - If no, consider patient-specific factors that may require deviation from standard care.

**Abscess Drainage and Necrotic Tissue Excision**
- Assess the need for abscess drainage and necrotic tissue excision.
  • Clinical question: Are there signs of abscess or necrosis?
    - If yes, proceed to standard first-line care. 
      Abscesses must be drained, and necrotic tissues (eg, infarcted bowel, gangrenous gall-bladder) should be excised.

**Monitoring**
- Assess the need for frequent monitoring.
  • Clinical question: Is the patient in septic shock?
    - If yes, proceed to standard first-line care. 
      Monitor systemic pressure; CVP, PAOP, or both; pulse oximetry; ABGs; blood glucose, lactate, and electrolyte levels; renal function, and possibly sublingual PCO2.

**Sources**
- Chapter 222: Approach to the Critically Ill Patient
- Treatment section for septic shock

Llama.generate: 1357 prefix-match hit, remaining 1334 prompt tokens to eval



--- No Prompt | Combination 5 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1947.05 ms /  1334 tokens (    1.46 ms per token,   685.14 tokens per second)
llama_perf_context_print:        eval time =   19225.88 ms /   403 runs   (   47.71 ms per token,    20.96 tokens per second)
llama_perf_context_print:       total time =   22223.50 ms /  1737 tokens
llama_perf_context_print:    graphs reused =        400


**Fluid Resuscitation**
- Assess the patient's volume status and start fluid resuscitation with 0.9% normal saline.
- Monitor CVP, PAOP, or both to guide fluid administration.
- Continue fluid resuscitation until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.

**Oxygenation**
- Provide O2 by mask or nasal prongs.
- Consider tracheal intubation and mechanical ventilation if respiratory failure develops.

**Antibiotic Therapy**
- Administer broad-spectrum antibiotics (e.g., gentamicin or tobramycin plus a 3rd-generation cephalosporin) empirically, based on suspected source and clinical setting.
- Modify antibiotic selection according to culture results.
- Continue antibiotics for at least 5 days after shock resolves and evidence of infection subsides.

**Source Control**
- Drain abscesses and excise necrotic tissues (e.g., infarcted bowel or gangrenous gall-bladder).
- Eliminate septic foci to prevent continued deterioration despite antibiotic therapy.

**Blood Glucose Management**
- Normalize blood glucose levels using a continuous IV insulin infusion.
- Titrate insulin dose to maintain glucose between 80 to 110 mg/dL (4.4 to 6.1 mmol/L).

**Corticosteroid Therapy**
- Administer replacement doses of corticosteroids (e.g., hydrocortisone 50 mg IV q 6 h).
- Consider fludrocortisone 50 μg po once/day during hemodynamic instability.

**Activated Protein C**
- Consider activated protein C (drotrecogin alfa) if begun early in patients with significant risk of death (APACHE II score > 25).

**Sources**
- Chapter 222. Approach to the Critically Ill Patient
- Chapter 227. Sepsis & Septic Shock

Combinations 2-4 at `temperature`=0.7, and `temperature`=0.5, and k=3 all produced if-then decision tree formatting without any system prompt instructing them to do so. Combination 1 (temperature=0, k=5) and combination 5 (temperature=0, k=7) produced flat sequential lists instead. Very telling that at `temperature`=0, the model defaults to a list-based prose when given the retrieved context alone, but a slight temperature increase pushes it towards branching logic. The structural prefrence is emerging from the interaction between the retrieved manual structure and model's sampling behavior, not from the prompt itself.

Combination 2 (temp=0.7, k=5): has the most extensive source citations, including specific page numbers for nutrition, stress ulcers, pulmonary embolism, and end-of-life care. Whether those specific branches are clinically accurateisn't verifiable, but they're traceable to retrieved content by appearing in the sources.

Combination 4 (temp=0, k=3): despite the lower k, this produced explicit "Standard first-line care" and "Patient-specific factors" labels, the same categorical structure `system_prompt3` was explicitly designed to produce, without being instructed to use them.

Combination 5 (temp=0, k=7): the most comprehensive coverage of all five. Including specifics that don't appear in any other combination. With more retrieved chunks (k=7), the model had access to content from later in the sepsis chapter that lower-k combinations missed, but the format reverted to a flat list, losing the branching structure.

This reveals a core tradeoff that no single combination fully resolves: higher k increases clinical completeness but reduces structural coherence; lower temperature improves consistency but limits the branching logic that makes responses most useful for time-sensitive clinical decisions. The evaluation scores in the following section will help determine which combination best balances these competing priorities for deployment.

In [100]:
# Query 1: RAG fine-tuning combinations
# Testing system_prompt3 across different parameter settings
test_configs = [
    {"system_prompt": system_prompt3, "params": {"temperature": 0}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0.7}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0}, "k": 3},
    {"system_prompt": system_prompt3, "params": {"temperature": 0}, "k": 7},
    {"system_prompt": system_prompt3, "params": {"temperature": 0, "repeat_penalty": 1.3}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0.7, "seed": 99}},
    {"system_prompt": system_prompt3, "params": {"temperature": 0.5, "top_p": 0.85, "top_k": 40}},
]

print(f"Running combinations for: {question1}")

for i, cfg in enumerate(test_configs, 1):
    print(f"--- Combination {i} ---")
    k = cfg.get("k", 5)  # default to 5 if not specified
    answer = generate_rag_response(
        question1,
        system_prompt=cfg["system_prompt"],
        params=cfg["params"],
        k=k
    )
    display(Markdown(answer))
    print()

Llama.generate: 32 prefix-match hit, remaining 1832 prompt tokens to eval


Running combinations for: What is the protocol for managing sepsis in a critical care unit?
--- Combination 1 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2401.46 ms /  1832 tokens (    1.31 ms per token,   762.87 tokens per second)
llama_perf_context_print:        eval time =   23687.55 ms /   519 runs   (   45.64 ms per token,    21.91 tokens per second)
llama_perf_context_print:       total time =   27547.56 ms /  2351 tokens
llama_perf_context_print:    graphs reused =        516


**Clinical Decision Pathway for Managing Sepsis in a Critical Care Unit**

1. **Initial Assessment**
   - Assess patient's vital signs, including temperature, blood pressure (BP), pulse, and respiration rate.
   - Quantify all fluid intake and output.
   - Measure daily weight.

2. **Suspected Sepsis**
   - If suspected sepsis is identified, initiate empiric therapy immediately with broad-spectrum antibiotics.
   - Take specimens of blood, body fluids, and wound sites for Gram stain and culture.

3. **Antibiotic Selection**
   - Based on the suspected source, clinical setting, knowledge or suspicion of causative organisms, and previous culture results:
     * For septic shock of unknown cause: Gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (e.g., cefotaxime).
     * Alternatively, ceftazidime plus a fluoroquinolone (e.g., ciprofloxacin) may be used.
   - If resistant staphylococci or enterococci are suspected: Add vancomycin.
   - If an abdominal source is suspected: Include a drug effective against anaerobes (e.g., metronidazole).

4. **Fluid Resuscitation**
   - Administer 0.9% normal saline until CVP reaches 8 mm Hg or PAOP reaches 12 to 15 mm Hg.
   - Monitor urine output and adjust fluid resuscitation accordingly.

5. **Monitoring**
   - Frequently monitor systemic pressure, CVP, PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and sublingual PCO2.

6. **Hypotension**
   - If a patient with septic shock remains hypotensive after CVP or PAOP has been raised to target levels:
     * Give dopamine to increase mean BP to at least 60 mm Hg.
     * If dopamine dose exceeds 20 μg/kg/min, add another vasopressor (typically norepinephrine).

7. **Oxygenation**
   - Provide O2 by mask or nasal prongs.
   - Consider tracheal intubation and mechanical ventilation for respiratory failure.

**Source:** The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 227. Sepsis & Septic Shock, p. 2446.

Llama.generate: 1863 prefix-match hit, remaining 1 prompt tokens to eval


...

**Clinical Decision Pathway for Managing Sepsis in a Critical Care Unit:**

1. **Suspect Sepsis:** If the patient has suspected sepsis, do:
	* Obtain specimens of blood, body fluids, and wound sites for Gram stain and culture.
	* Start empiric therapy with broad-spectrum antibiotics (modified by culture results).
2. **Assess Patient's Condition:**
	* Measure vital signs (temperature, BP, pulse, and respiration rate).
	* Quantify all fluid intake and output.
	* Monitor CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate, and electrolyte levels.
	* Check urine output with an indwelling catheter.
3. **Fluid Resuscitation:**
	* If the patient is hypotensive and oliguric, give 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
4. **Monitor for Septic Shock:**
	* If the patient remains hypotensive after fluid resuscitation, consider dopamine to increase mean BP to at least 60 mm Hg.
	* If dopamine dose exceeds 20 μg/kg/min, add another vasopressor (typically norepinephrine).
5. **Antibiotic Selection:**
	* Based on suspected source, clinical setting, knowledge of causative organisms and sensitivity patterns, and previous culture results:
		+ For unknown cause: gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxime 2 g q 6-8 h or ceftriaxone 2 g once/day).
		+ Alternative regimens may include ceftazidime plus a fluoroquinolone (ciprofloxacin) or monotherapy with maximal therapeutic doses of ceftazidime or imipenem.
	* Add vancomycin for resistant staphylococci or enterococci, and consider metronidazole for abdominal sources.
6. **Source Control:**
	* Drain abscesses and excise necrotic tissue (e.g., infarcted bowel, gangrenous gall-bladder).
7. **Continue Antibiotics:**
	* Continue antibiotics for at least 5 days after shock resolves and evidence of infection subsides.
8. **Monitor Patient's Response:**
	* Continuously monitor the patient's condition and adjust treatment as needed.

**Sources:** The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 227. Sepsis & Septic Shock

Llama.generate: 1863 prefix-match hit, remaining 1 prompt tokens to eval



--- Combination 7 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   23944.50 ms /   536 runs   (   44.67 ms per token,    22.39 tokens per second)
llama_perf_context_print:       total time =   25517.96 ms /   537 tokens
llama_perf_context_print:    graphs reused =        533


**Clinical Decision Pathway for Managing Sepsis in a Critical Care Unit**

1. **Initial Assessment**
   - Assess patient's clinical presentation, vital signs, and laboratory results to determine the presence of sepsis.
   If suspected sepsis, proceed with empiric therapy; if not, monitor closely for signs of deterioration.

2. **Fluid Resuscitation**
   - Administer 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
   If oliguria with hypotension, continue vigorous fluid resuscitation despite potential for fluid overload.

3. **Antibiotic Selection**
   - Initiate broad-spectrum antibiotics based on suspected source of infection, clinical setting, and knowledge of causative organisms in the ICU.
   Consider gentamicin or tobramycin plus a 3rd-generation cephalosporin (e.g., cefotaxime) as initial empiric therapy for unknown cause.
   Adjust antibiotic regimen according to culture and sensitivity results.

4. **Source Control**
   - Drain abscesses and excise necrotic tissue to prevent ongoing infection and sepsis progression.

5. **Hemodynamic Support**
   - Monitor systemic pressure, CVP, PAOP, pulse oximetry, ABGs, blood glucose, lactate, and electrolyte levels frequently.
   If hypotensive despite adequate fluid resuscitation, consider adding vasopressors (e.g., dopamine or norepinephrine).
   However, be cautious of potential risks of organ hypoperfusion and acidosis.

6. **Oxygenation**
   - Provide supplemental oxygen by mask or nasal prongs; consider tracheal intubation and mechanical ventilation if respiratory failure develops.

**Standard First-Line Care**

- Administer 0.9% normal saline for fluid resuscitation
- Initiate broad-spectrum antibiotics based on suspected source of infection
- Drain abscesses and excise necrotic tissue

**Patient-Specific Factors**

- Consider vancomycin if resistant staphylococci or enterococci are suspected
- Add a drug effective against anaerobes (e.g., metronidazole) if an abdominal source is suspected
- Use alternative antibiotics (e.g., ceftazidime plus a fluoroquinolone) based on local resistance patterns and previous culture results

**Source**

The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 227. Sepsis & Septic Shock

Every single combination produced a "Clinical Decision Pathway" header and cited the Merck Manual, meaning `system_prompt3` is consistently shaping both structure and sourcing across all parameter settings. This is a meaningful contrast from the no-prompt combinations, where only temperature greater than 0 produced branching structure. With `system_prompt3`, even temperature=0 produces if-then logic.

Combination 1 (temp=0, k=5): solid baseline, covers antibiotics, fluid resuscitation, monitoring, hypotension management, oxygenation. Specific page citation. Missing blood glucose management and corticosteroids that appear in later combinations.

Combination 2 (temp=0.7, k=5): the only combination that explicitly mentions the 6-hour early aggressive therapy window as a branching decision point ("If sepsis is suspected, institute early aggressive therapy within 6 hours"). Also includes abscess drainage as a branching step. Notably shorter than combination 1. A higher temperature produced more selective, branching-focused output rather than more verbose output.

Combination 3 (k=3, temp=0): longest and most detailed of all seven. Includes sub-categorized antibiotic regimens (septic shock of unknown cause, resistant staphylococci, Pseudomonas) and specific antibiotic modification steps. Despite having fewer retrieved chunks, the structured antibiotic guidance is the most granular. This could be because k=3 retrieved only the most tightly relevant chunks, concentrating the content around antibiotic selection specifically.

Combination 4 (k=7, temp=0): the only combination that includes blood glucose management, corticosteroid therapy, and activated protein C with APACHE II criteria. These sepsis management steps are absent from all other combinations. This directly mirrors the sepsis no-prompt combination 5 (k=7) finding, where higher k retrieves more content in the chapter that a lower k misses. Also the only combination that explicitly uses "Standard First-Line Care" and "Patient-Specific Factors" section labels.

Combination 5 (repeat_penalty=1.3, temp=0): produces conditional logic, it has explicit questions such as, "Is the mean BP ≥ 60 mmHg?", that no other combination does. The repeat penalty appears to push the model away from its default list patterns toward more varied sentence structures, producing the most decision-tree-like formatting of all seven.

Combination 6 (temp=0.7, seed=99): introduces two unique clinical details not present in any other combination: lactate >4 mmol/L as a severity threshold, and Candida-specific treatment. The seed change from 42 to 99 at the same temperature produced noticeably different content from combination 2, confirming seed meaningfully affects output at temperature greater than 0.

Combination 7 (temp=0.5, top_p=0.85, top_k=40): the most concise and cleanest structure of all seven. Covers all major components without redundancy. The "If < 6 hours since suspected diagnosis" time-bound framing is unique to this combination. Conservative parameters produced the most focused, well-organized output.

Overall, no single combination dominates across all dimensions simultaneously,
combination 4 seems to maximize clinical completeness, combination 7 maximizes structural clarity, and combination 5 maximizes decision-tree format adherence. The evaluation scores will determine which combination best balances these priorities for deployment.

### Fine-tuning Coverage Note

The fine-tuning combinations above demonstrate qualitative output variation for
Query 1 in detail, covering seven system_prompt3 configurations and five
no-system-prompt configurations, with written observations per combination.

These same 12 configurations will be then to all five clinical queries
in the section below, where each combination's groundedness
and relevance scores are reported across all 60 generated responses. Rather than repeating the qualitative generation output for all five queries times 12 combinations (which would produce 60 additional response cells with limited additional insight), the evaluation section serves as the quantitative validation of the fine-tuning parameters across the full query set.

The evaluation scores confirm which combinations generalize best across query
types, a finding that would not be visible from a single-query qualitative
analysis alone.

## Output Evaluation

The following section uses the LLM-as-a-judge method to evaluate the quality of
the RAG system across two dimensions: groundedness and relevance.

**Groundedness** measures whether the answer is directly supported by the retrieved context from the Merck Manual.

**Relevance** measures whether the answer directly addresses the clinical question asked.

Evaluation is performed using the same Meta Llama 3.1 8B Instruct model used for answer generation. This is a self-evaluation approach: the model judges its own outputs against the retrieved context and the original question.

Each response is scored on a 1–5 scale with a verdict and one-sentence explanation, evaluated independently for groundedness and relevance. Scores are aggregated across all five queries and twelve parameter combinations to identify the configuration best suited for clinical deployment.

In [101]:
groundedness_rater_system_message = """
You are an evaluator that determines whether an answer is fully grounded in the provided context.

Rules:
- Only compare the answer to the context.
- If any part of the answer is not directly supported by the context, mark it accordingly.
- Do not use outside knowledge.
- Do not infer missing steps.
- Do not judge medical correctness, only grounding.

Output format (always use exactly this structure):
Score: X/5
Verdict: Grounded / Partially Grounded / Not Grounded
Reason: One sentence identifying which specific part of the answer is or is not supported by the context.
"""

In [102]:
relevance_rater_system_message = """
You are an evaluator that determines whether an answer is relevant to the user's question.

Rules:
- Evaluate whether the answer directly addresses what was asked.
- Ignore style, length, or formatting.
- Do not judge medical correctness.
- Do not judge grounding.
- Only judge relevance to the question.

Output format (always use exactly this structure):
Score: X/5
Verdict: Relevant / Partially Relevant / Not Relevant
Reason: One sentence explaining which specific part of the answer does or does not address the question asked.
"""

In [103]:
user_message_template = """
###Context
{context}

###Question
{question}

###Answer
{answer}
"""

In [104]:
def evaluate_answer(question, answer, k=5):
    """
    Evaluates a pre-generated answer for groundedness and relevance.
    Re-retrieves context using the same k to ensure consistency.
    """
    # Re-retrieve the same context as what was used to generate the answer
    retriever = vectorstore.as_retriever(search_kwargs={'k': k})
    relevant_chunks = retriever.invoke(question)
    context_list = [d.page_content for d in relevant_chunks]
    context_for_query = "\n\n".join(context_list)

    # Build eval message with context + question + answer
    eval_message = user_message_template.format(
        context=context_for_query,
        question=question,
        answer=answer
    )

    # Fixed config for evaluation
    eval_config = {
        "max_tokens": 150,
        "temperature": 0,
        "top_p": 0.95,
        "top_k": 50,
        "stop": ["<|eot_id|>"]
    }

    # Groundedness
    grounding_output = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": groundedness_rater_system_message},
            {"role": "user", "content": eval_message}
        ],
        **eval_config
    )
    groundedness = grounding_output["choices"][0]["message"]["content"].strip()

    # Relevance
    relevance_output = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": relevance_rater_system_message},
            {"role": "user", "content": eval_message}
        ],
        **eval_config
    )
    relevance = relevance_output["choices"][0]["message"]["content"].strip()

    return groundedness, relevance

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [105]:
# Query 1: RAG fine-tuning combinations with stored answers
test_configs = [
    # system_prompt3 combinations
    {"label": "SP3 | temp=0",               "system_prompt": system_prompt3, "params": {"temperature": 0}},
    {"label": "SP3 | temp=0.7",             "system_prompt": system_prompt3, "params": {"temperature": 0.7}},
    {"label": "SP3 | temp=0 | k=3",         "system_prompt": system_prompt3, "params": {"temperature": 0}, "k": 3},
    {"label": "SP3 | temp=0 | k=7",         "system_prompt": system_prompt3, "params": {"temperature": 0}, "k": 7},
    {"label": "SP3 | repeat_penalty=1.3",   "system_prompt": system_prompt3, "params": {"temperature": 0, "repeat_penalty": 1.3}},
    {"label": "SP3 | temp=0.7 | seed=99",   "system_prompt": system_prompt3, "params": {"temperature": 0.7, "seed": 99}},
    {"label": "SP3 | temp=0.5 conservative","system_prompt": system_prompt3, "params": {"temperature": 0.5, "top_p": 0.85, "top_k": 40}},
    # No system prompt
    {"label": "No SP | temp=0",             "system_prompt": None, "params": {"temperature": 0}},
    {"label": "No SP | temp=0.7",           "system_prompt": None, "params": {"temperature": 0.7}},
    {"label": "No SP | temp=0.5",           "system_prompt": None, "params": {"temperature": 0.5, "top_p": 0.85, "top_k": 40}},
    {"label": "No SP | temp=0 | k=3",       "system_prompt": None, "params": {"temperature": 0}, "k": 3},
    {"label": "No SP | temp=0 | k=7",       "system_prompt": None, "params": {"temperature": 0}, "k": 7},
]

# Store generated answers alongside their config
results_q1 = []

for i, cfg in enumerate(test_configs, 1):
    k = cfg.get("k", 5)
    print(f"--- Combination {i}: {cfg['label']} ---")
    answer = generate_rag_response(
        question1,
        system_prompt=cfg["system_prompt"],
        params=cfg["params"],
        k=k
    )
    display(Markdown(answer))
    print()

    # Store for evaluation
    results_q1.append({
        "combo": i,
        "label": cfg["label"],
        "k": k,
        "answer": answer
    })

Llama.generate: 1863 prefix-match hit, remaining 1 prompt tokens to eval


--- Combination 1: SP3 | temp=0 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   23032.13 ms /   516 runs   (   44.64 ms per token,    22.40 tokens per second)
llama_perf_context_print:       total time =   24558.43 ms /   517 tokens
llama_perf_context_print:    graphs reused =        513


**Clinical Decision Pathway for Managing Sepsis in a Critical Care Unit**

1. **Initial Assessment**
   - Assess patient's vital signs, including temperature, blood pressure (BP), pulse, and respiration rate.
   - Quantify all fluid intake and output.
   - Measure daily weight.

2. **Suspected Sepsis**
   - If suspected sepsis is identified, initiate empiric therapy immediately with broad-spectrum antibiotics.
   - Take specimens of blood, body fluids, and wound sites for Gram stain and culture.

3. **Antibiotic Selection**
   - If the source of sepsis is unknown, consider gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (e.g., cefotaxime 2 g q 6-8 h).
   - Alternatively, ceftazidime plus a fluoroquinolone (e.g., ciprofloxacin) may be used.
   - If resistant staphylococci or enterococci are suspected, add vancomycin.

4. **Fluid Resuscitation**
   - Administer 0.9% normal saline until central venous pressure (CVP) reaches 8 mm Hg (10 cm H2O) or pulmonary artery occlusion pressure (PAOP) reaches 12-15 mm Hg.
   - Monitor urine output and adjust fluid resuscitation accordingly.

5. **Monitoring**
   - Frequently monitor systemic pressure, CVP, PAOP, pulse oximetry, arterial blood gases (ABGs), blood glucose, lactate, and electrolyte levels.
   - Measure renal function and possibly sublingual PCO2.

6. **Hypotension**
   - If a patient with septic shock remains hypotensive after CVP or PAOP has been raised to target levels, consider dopamine (up to 20 μg/kg/min) to increase mean BP to at least 60 mm Hg.
   - If dopamine dose exceeds 20 μg/kg/min, add another vasopressor, typically norepinephrine.

7. **Oxygen Therapy**
   - Provide oxygen by mask or nasal prongs.
   - Consider tracheal intubation and mechanical ventilation if respiratory failure develops.

**Source:** The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 227. Sepsis & Septic Shock, p. 2446.

Llama.generate: 1863 prefix-match hit, remaining 1 prompt tokens to eval


...

**Fluid Resuscitation**
- Assess the patient's need for fluid resuscitation.
  • Clinical question: Is the patient hypotensive or oliguric?
    - If yes, proceed to give fluid resuscitation with 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
    - If no, monitor vital signs and urine output closely.

**Antibiotic Therapy**
- Assess the need for antibiotic therapy.
  • Clinical question: Is sepsis suspected?
    - If yes, give parenteral antibiotics after specimens of blood, body fluids, and wound sites have been taken for Gram stain and culture.
      • Standard first-line care: Gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day).
      • Patient-specific factors:
        - If Pseudomonas is suspected, add ceftazidime 2 g IV q 8 h.
        - If resistant staphylococci or enterococci are suspected, add vancomycin.
        - If there is an abdominal source, include a drug effective against anaerobes (eg, metronidazole).

**Source Control**
- Assess the need for source control.
  • Clinical question: Are abscesses present?
    - If yes, drain abscesses and excise necrotic tissue.

**Monitoring**
- Monitor vital signs, urine output, CVP, PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and possibly sublingual PCO2 frequently.

**Sources**
- Chapter 222. Approach to the Critically Ill Patient
- Treatment section for septic shock

Llama.generate: 1357 prefix-match hit, remaining 1334 prompt tokens to eval



--- Combination 12: No SP | temp=0 | k=7 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1954.91 ms /  1334 tokens (    1.47 ms per token,   682.38 tokens per second)
llama_perf_context_print:        eval time =   18981.35 ms /   402 runs   (   47.22 ms per token,    21.18 tokens per second)
llama_perf_context_print:       total time =   22119.40 ms /  1736 tokens
llama_perf_context_print:    graphs reused =        399


**Fluid Resuscitation**
- Assess the patient's volume status and start fluid resuscitation with 0.9% normal saline.
- Monitor CVP, PAOP, or both to guide fluid administration.
- Continue fluid resuscitation until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.

**Oxygenation**
- Administer O2 by mask or nasal prongs.
- Consider tracheal intubation and mechanical ventilation if respiratory failure develops.

**Antibiotic Therapy**
- Obtain blood, body fluid, and wound site cultures for Gram stain and culture.
- Start broad-spectrum antibiotics (e.g., gentamicin or tobramycin plus a 3rd-generation cephalosporin).
- Modify antibiotic selection based on culture results and suspected source of infection.

**Source Control**
- Drain abscesses and excise necrotic tissues (e.g., infarcted bowel, gangrenous gall-bladder).

**Blood Glucose Management**
- Normalize blood glucose levels using a continuous IV insulin infusion.
- Titrate insulin dose to maintain glucose between 80 to 110 mg/dL.

**Corticosteroid Therapy**
- Administer replacement doses of corticosteroids (e.g., hydrocortisone 50 mg IV q 6 h).
- Continue corticosteroid therapy for at least 3 days after hemodynamic stability is achieved.

**Activated Protein C**
- Consider activated protein C (drotrecogin alfa) if the patient has a high risk of death (APACHE II score > 25).

**Monitoring and Supportive Care**
- Monitor vital signs, urine output, and laboratory results frequently.
- Provide supportive care, including nutritional support and stress ulcer prophylaxis.

**Sources**
- Chapter 222. Approach to the Critically Ill Patient
- Chapter 227. Sepsis & Septic Shock

In [106]:
# Evaluation of all Query 1 combinations — scores and reasoning only
print("=" * 60)
print("EVALUATION RESULTS: Query 1")
print("=" * 60)

for result in results_q1:
    groundedness, relevance = evaluate_answer(
        question1,
        result["answer"],
        k=result["k"]
    )

    # Store scores back into the result dict for the summary table later
    result["groundedness"] = groundedness
    result["relevance"] = relevance

    display(Markdown(f"""
**Combination {result['combo']}: {result['label']}**

**Groundedness:** {groundedness}

**Relevance:** {relevance}

---"""))

Llama.generate: 28 prefix-match hit, remaining 2288 prompt tokens to eval


EVALUATION RESULTS: Query 1


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3049.64 ms /  2288 tokens (    1.33 ms per token,   750.25 tokens per second)
llama_perf_context_print:        eval time =    2754.95 ms /    57 runs   (   48.33 ms per token,    20.69 tokens per second)
llama_perf_context_print:       total time =    5838.69 ms /  2345 tokens
llama_perf_context_print:    graphs reused =         56
Llama.generate: 35 prefix-match hit, remaining 2267 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3030.65 ms /  2267 tokens (    1.34 ms per token,   748.03 tokens per second)
llama_perf_context_print:        eval time =    3240.00 ms /    67 runs   (   48.36 ms per token,    20.68 tokens per second)
llama_perf_context_print:       total time =    6325.29 ms /  2334 tokens
llama_perf_context_print:    graphs reused =         65



**Combination 1: SP3 | temp=0**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer is fully supported by the provided context from The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 227. Sepsis & Septic Shock, p. 2446.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The entire answer directly addresses the protocol for managing sepsis in a critical care unit, providing a step-by-step clinical decision pathway for initial assessment, suspected sepsis, antibiotic selection, fluid resuscitation, monitoring, hypotension, and oxygen therapy.

---

Llama.generate: 35 prefix-match hit, remaining 2170 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2871.78 ms /  2170 tokens (    1.32 ms per token,   755.63 tokens per second)
llama_perf_context_print:        eval time =    2048.26 ms /    43 runs   (   47.63 ms per token,    20.99 tokens per second)
llama_perf_context_print:       total time =    4944.82 ms /  2213 tokens
llama_perf_context_print:    graphs reused =         42
Llama.generate: 35 prefix-match hit, remaining 2156 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2857.12 ms /  2156 tokens (    1.33 ms per token,   754.61 tokens per second)
llama_perf_context_print:        eval time =    2406.30 ms /    50 runs   (   48.13 ms per token,    20.78 tokens per second)
llama_perf_context_print:       total time =    5296.07 ms /  2206 tokens
llama_perf_context_print: 

...


**Combination 10: No SP | temp=0.5**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer is fully supported by the provided context, which outlines the protocol for managing sepsis in a critical care unit, including fluid resuscitation, antibiotic therapy, source-specific therapy, additional measures, vasopressor therapy, oxygenation and ventilation, and references to specific chapters in the Merck Manual of Diagnosis & Therapy.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The entire answer directly addresses the protocol for managing sepsis in a critical care unit, covering fluid resuscitation, antibiotic therapy, source-specific therapy, additional measures, vasopressor therapy, oxygenation and ventilation, and providing specific treatment options and guidelines.

---

Llama.generate: 35 prefix-match hit, remaining 1585 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2050.77 ms /  1585 tokens (    1.29 ms per token,   772.88 tokens per second)
llama_perf_context_print:        eval time =    2547.55 ms /    55 runs   (   46.32 ms per token,    21.59 tokens per second)
llama_perf_context_print:       total time =    4635.34 ms /  1640 tokens
llama_perf_context_print:    graphs reused =         54
Llama.generate: 35 prefix-match hit, remaining 1571 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2046.44 ms /  1571 tokens (    1.30 ms per token,   767.68 tokens per second)
llama_perf_context_print:        eval time =    2262.28 ms /    48 runs   (   47.13 ms per token,    21.22 tokens per second)
llama_perf_context_print:       total time =    4332.10 ms /  1619 tokens
llama_perf_context_print: 


**Combination 11: No SP | temp=0 | k=3**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer is fully supported by the provided context, which outlines the protocol for managing sepsis in a critical care unit, including fluid resuscitation, antibiotic therapy, source control, and monitoring.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The entire answer directly addresses the protocol for managing sepsis in a critical care unit, outlining fluid resuscitation, antibiotic therapy, source control, and monitoring steps.

---

Llama.generate: 35 prefix-match hit, remaining 2892 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3961.18 ms /  2892 tokens (    1.37 ms per token,   730.09 tokens per second)
llama_perf_context_print:        eval time =    2169.19 ms /    44 runs   (   49.30 ms per token,    20.28 tokens per second)
llama_perf_context_print:       total time =    6162.57 ms /  2936 tokens
llama_perf_context_print:    graphs reused =         43
Llama.generate: 35 prefix-match hit, remaining 2878 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3946.21 ms /  2878 tokens (    1.37 ms per token,   729.31 tokens per second)
llama_perf_context_print:        eval time =    3516.75 ms /    71 runs   (   49.53 ms per token,    20.19 tokens per second)
llama_perf_context_print:       total time =    7498.30 ms /  2949 tokens
llama_perf_context_print: 


**Combination 12: No SP | temp=0 | k=7**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer is fully supported by the provided context, which is a medical textbook chapter on sepsis and septic shock in critical care units.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by providing a comprehensive protocol for managing sepsis in a critical care unit, including fluid resuscitation, oxygenation, antibiotic therapy, source control, blood glucose management, corticosteroid therapy, activated protein C, and monitoring and supportive care.

---

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [107]:
# Query 2
results_q2 = []

for i, cfg in enumerate(test_configs, 1):
    k = cfg.get("k", 5)
    print(f"--- Combination {i}: {cfg['label']} ---")
    answer = generate_rag_response(
        question2,
        system_prompt=cfg["system_prompt"],
        params=cfg["params"],
        k=k
    )
    display(Markdown(answer))
    print()

    results_q2.append({
        "combo": i,
        "label": cfg["label"],
        "k": k,
        "answer": answer
    })

Llama.generate: 28 prefix-match hit, remaining 1714 prompt tokens to eval


--- Combination 1: SP3 | temp=0 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2227.40 ms /  1714 tokens (    1.30 ms per token,   769.51 tokens per second)
llama_perf_context_print:        eval time =   25273.14 ms /   563 runs   (   44.89 ms per token,    22.28 tokens per second)
llama_perf_context_print:       total time =   29105.96 ms /  2277 tokens
llama_perf_context_print:    graphs reused =        559


**Clinical Decision Pathway for Appendicitis**

1. **Presentation**: The patient presents with acute abdominal pain, often accompanied by anorexia and low-grade fever.
2. If the patient has a history of inflammatory bowel disease involving the cecum, do not proceed with appendectomy (contraindication). If the appendix is involved but the cecum is normal, proceed to step 3.
	* Clinical question: Is there a contraindication to appendectomy?
	* Action: Do not perform appendectomy if inflammatory bowel disease involves the cecum.
3. **Diagnosis**: Confirm the diagnosis of appendicitis through clinical evaluation (e.g., McBurney's point tenderness, Rovsing sign) and imaging studies (e.g., CT or ultrasound).
4. If the appendix is perforated, do not delay surgery; proceed to step 5.
	* Clinical question: Is the appendix perforated?
	* Action: Proceed with immediate surgery if perforation is suspected.
5. **Surgery**: Perform an open or laparoscopic appendectomy (appendix removal).
6. If the patient has a large inflammatory mass involving the appendix, terminal ileum, and cecum, consider resection of the entire mass and ileocolostomy.
7. If surgery is impossible, continue antibiotics until the patient's temperature and WBC count have normalized or for a fixed course (according to surgeon's preference).
8. **Post-operative care**: Administer IV fluids and electrolytes as needed, monitor urine output with a catheter if necessary.

**Common symptoms of appendicitis:**

* Acute abdominal pain
* Anorexia
* Low-grade fever (rectal temperature 37.7 to 38.3° C [100 to 101° F])
* Nausea and vomiting
* Pain that shifts from the epigastric or periumbilical area to the right lower quadrant

**Can appendicitis be cured via medicine?**

No, appendicitis requires surgical removal of the appendix (appendectomy) for definitive treatment. Antibiotics may be used as an adjunctive therapy to manage symptoms and prevent complications, but they do not cure the condition.

**Surgical procedure:**

The recommended surgical procedure is an open or laparoscopic appendectomy (appendix removal). In cases where a large inflammatory mass is present, resection of the entire mass and ileocolostomy may be necessary. If surgery is impossible, antibiotics should be continued until the patient's temperature and WBC count have normalized or for a fixed course.

**Sources:**

* The Merck Manual of Diagnosis & Therapy, 19th Edition
* Chapter 11. Acute Abdomen & Surgical Gastroenterology

Llama.generate: 1741 prefix-match hit, remaining 1 prompt tokens to eval


...

###Clinical Decision Pathway for Appendicitis Treatment

**Step 1: Confirm Diagnosis**
Is the patient presenting with typical symptoms of appendicitis (abdominal pain, anorexia, and abdominal tenderness)?

* If yes, proceed to Step 2.
* If no, consider alternative diagnoses.

**Step 2: Assess Perforation Risk**
Has a perforation been confirmed or suspected?

* If yes, proceed to Step 3.
* If no, continue with standard treatment.

**Standard First-Line Care**

1. Administer IV antibiotics (third-generation cephalosporins preferred).
2. Maintain fluid status with adequate IV fluids and electrolyte replacement.
3. Consider imaging studies (CT or ultrasound) for diagnosis.

**Patient-Specific Factors Requiring Deviation**

* If perforation is confirmed, proceed to immediate surgery.
* If an abscess or inflammatory mass has formed, consider drainage of the abscess.

**Surgical Procedure**
Perform open or laparoscopic appendectomy. If the appendix is difficult to locate, consider resection of the entire mass and ileocolostomy if necessary.

**Sources**

* Treatment of acute appendicitis
* Appendectomy for perforated appendicitis
* Use of IV antibiotics and fluid replacement
* Surgical procedures for appendicitis

Llama.generate: 1153 prefix-match hit, remaining 1082 prompt tokens to eval



--- Combination 12: No SP | temp=0 | k=7 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1514.95 ms /  1082 tokens (    1.40 ms per token,   714.21 tokens per second)
llama_perf_context_print:        eval time =   14035.35 ms /   312 runs   (   44.99 ms per token,    22.23 tokens per second)
llama_perf_context_print:       total time =   16449.26 ms /  1394 tokens
llama_perf_context_print:    graphs reused =        310


**Clinical Decision Pathway for Appendicitis Treatment**

1. **Assess symptoms**: Is the patient experiencing abdominal pain, anorexia, and abdominal tenderness?
	* If yes, proceed to step 2.
	* If no, consider alternative diagnoses.
2. **Determine symptom duration**: Has the pain been present for < 6 hours?
	* If yes, proceed to step 3.
	* If no, consider delayed diagnosis and increased risk of perforation.
3. **Assess patient's condition**: Is the patient stable or showing signs of volume depletion?
	* If stable, proceed to step 4.
	* If unstable, prioritize fluid resuscitation and stabilization before proceeding.
4. **Determine need for surgery**: Has a diagnosis of appendicitis been confirmed through clinical evaluation, imaging studies (e.g., CT or ultrasound), or laparoscopy?
	* If yes, proceed to step 5.
	* If no, consider alternative diagnoses and further diagnostic testing.
5. **Choose surgical approach**:
	* For non-perforated appendicitis: Standard first-line care is open or laparoscopic appendectomy.
	* For perforated appendicitis: Immediate surgery is necessary due to increased risk of peritonitis; consider drainage of abscess if present.

**Sources**

* Chapter 11. Acute Abdomen & Surgical Gastroenterology, The Merck Manual of Diagnosis & Therapy, 19th Edition
* Section on Appendicitis diagnosis and treatment

In [108]:
# Evaluation of Query 2
print("=" * 60)
print("EVALUATION RESULTS: Query 2")
print("=" * 60)

for result in results_q2:
    groundedness, relevance = evaluate_answer(
        question2,
        result["answer"],
        k=result["k"]
    )

    # Store scores back into the result dict for the summary table later
    result["groundedness"] = groundedness
    result["relevance"] = relevance

    display(Markdown(f"""
**Combination {result['combo']}: {result['label']}**

**Groundedness:** {groundedness}

**Relevance:** {relevance}

---"""))

Llama.generate: 28 prefix-match hit, remaining 2214 prompt tokens to eval


EVALUATION RESULTS: Query 2


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2950.06 ms /  2214 tokens (    1.33 ms per token,   750.49 tokens per second)
llama_perf_context_print:        eval time =    3644.58 ms /    76 runs   (   47.95 ms per token,    20.85 tokens per second)
llama_perf_context_print:       total time =    6631.73 ms /  2290 tokens
llama_perf_context_print:    graphs reused =         74
Llama.generate: 35 prefix-match hit, remaining 2193 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2927.52 ms /  2193 tokens (    1.33 ms per token,   749.10 tokens per second)
llama_perf_context_print:        eval time =    2282.21 ms /    48 runs   (   47.55 ms per token,    21.03 tokens per second)
llama_perf_context_print:       total time =    5242.73 ms /  2241 tokens
llama_perf_context_print:    graphs reused =         47



**Combination 1: SP3 | temp=0**

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer states that antibiotics "do not cure the condition" but "markedly improve the survival rate" when surgery is impossible, which is not directly supported by the context. The context only mentions that antibiotics "markedly improve the survival rate" without specifying that they do not cure the condition.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by listing the common symptoms of appendicitis, stating that it cannot be cured via medicine, and describing the recommended surgical procedure for treatment.

---

Llama.generate: 35 prefix-match hit, remaining 2085 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2764.12 ms /  2085 tokens (    1.33 ms per token,   754.31 tokens per second)
llama_perf_context_print:        eval time =    4314.97 ms /    90 runs   (   47.94 ms per token,    20.86 tokens per second)
llama_perf_context_print:       total time =    7123.64 ms /  2175 tokens
llama_perf_context_print:    graphs reused =         89
Llama.generate: 35 prefix-match hit, remaining 2071 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2749.06 ms /  2071 tokens (    1.33 ms per token,   753.35 tokens per second)
llama_perf_context_print:        eval time =    2276.63 ms /    48 runs   (   47.43 ms per token,    21.08 tokens per second)
llama_perf_context_print:       total time =    5061.54 ms /  2119 tokens
llama_perf_context_print: 

...


**Combination 10: No SP | temp=0.5**

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer includes a clinical decision pathway that is not directly supported by the context, specifically the sections on "Consider alternative diagnoses" and "Assess need for surgical intervention" which are not mentioned in the provided text.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by listing the common symptoms of appendicitis, stating that it cannot be cured via medicine, and describing the surgical procedure to treat it, including open or laparoscopic appendectomy and resection of the entire mass and ileocolostomy if necessary.

---

Llama.generate: 35 prefix-match hit, remaining 1258 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1600.87 ms /  1258 tokens (    1.27 ms per token,   785.83 tokens per second)
llama_perf_context_print:        eval time =    2262.26 ms /    49 runs   (   46.17 ms per token,    21.66 tokens per second)
llama_perf_context_print:       total time =    3899.34 ms /  1307 tokens
llama_perf_context_print:    graphs reused =         48
Llama.generate: 35 prefix-match hit, remaining 1244 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1578.64 ms /  1244 tokens (    1.27 ms per token,   788.02 tokens per second)
llama_perf_context_print:        eval time =    2703.86 ms /    58 runs   (   46.62 ms per token,    21.45 tokens per second)
llama_perf_context_print:       total time =    4311.02 ms /  1302 tokens
llama_perf_context_print: 


**Combination 11: No SP | temp=0 | k=3**

**Groundedness:** Score: 4/5
Verdict: Grounded
Reason: The answer is mostly supported by the context, but it does not explicitly mention that appendicitis cannot be cured via medicine, only that surgery is the primary treatment method.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the symptoms of appendicitis (abdominal pain, anorexia, and abdominal tenderness) and provides a detailed treatment plan, including surgical procedures, which is exactly what the question asked.

---

Llama.generate: 35 prefix-match hit, remaining 2346 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3148.59 ms /  2346 tokens (    1.34 ms per token,   745.09 tokens per second)
llama_perf_context_print:        eval time =    3760.12 ms /    78 runs   (   48.21 ms per token,    20.74 tokens per second)
llama_perf_context_print:       total time =    6961.21 ms /  2424 tokens
llama_perf_context_print:    graphs reused =         77
Llama.generate: 35 prefix-match hit, remaining 2332 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3140.25 ms /  2332 tokens (    1.35 ms per token,   742.62 tokens per second)
llama_perf_context_print:        eval time =    3386.12 ms /    70 runs   (   48.37 ms per token,    20.67 tokens per second)
llama_perf_context_print:       total time =    6564.54 ms /  2402 tokens
llama_perf_context_print: 


**Combination 12: No SP | temp=0 | k=7**

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer mentions "standard first-line care is open or laparoscopic appendectomy" for non-perforated appendicitis, but the context only mentions that "open or laparoscopic appendectomy" is a treatment option for acute appendicitis, without specifying it as the standard first-line care.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by listing the common symptoms of appendicitis, discussing the limitations of medical treatment, and outlining the surgical procedures for treating appendicitis, including open or laparoscopic appendectomy for non-perforated cases and immediate surgery for perforated cases.

---

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [109]:
# Query 3
results_q3 = []

for i, cfg in enumerate(test_configs, 1):
    k = cfg.get("k", 5)
    print(f"--- Combination {i}: {cfg['label']} ---")
    answer = generate_rag_response(
        question3,
        system_prompt=cfg["system_prompt"],
        params=cfg["params"],
        k=k
    )
    display(Markdown(answer))
    print()

    results_q3.append({
        "combo": i,
        "label": cfg["label"],
        "k": k,
        "answer": answer
    })

Llama.generate: 28 prefix-match hit, remaining 1713 prompt tokens to eval


--- Combination 1: SP3 | temp=0 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2219.95 ms /  1713 tokens (    1.30 ms per token,   771.64 tokens per second)
llama_perf_context_print:        eval time =   25861.82 ms /   581 runs   (   44.51 ms per token,    22.47 tokens per second)
llama_perf_context_print:       total time =   29775.56 ms /  2294 tokens
llama_perf_context_print:    graphs reused =        577


**Clinical Decision Pathway for Sudden Patchy Hair Loss**

1. **Initial Assessment**
   - Assess the patient's medical history, including any recent stressors or systemic disorders.
   - Evaluate the extent and distribution of hair loss.

2. **Determine if Concomitant Virilization is Present**
   - If concomitant virilization (e.g., hirsutism, acne) is present in women, do a thorough evaluation for underlying disorders.
     - If yes, proceed to step 3; if not, proceed to step 4.

3. **Evaluate for Underlying Disorders**
   - Consider conditions such as polycystic ovary syndrome (PCOS), congenital adrenal hyperplasia, or Cushing's syndrome.
   - Order laboratory tests as needed (e.g., hormonal assays).

4. **Consider Possible Causes of Sudden Patchy Hair Loss**
   - If no underlying disorders are found, consider the following possible causes:
     1. **Alopecia Areata**: autoimmune disorder characterized by sudden patchy hair loss.
       - If suspected, do a scalp biopsy or microscopic examination to confirm diagnosis.
         - If confirmed, proceed to step 5; if not, consider other causes.

5. **Treatment for Alopecia Areata**
   - Topical corticosteroids (e.g., clobetasol propionate) applied directly to the affected area.
     - Systemic corticosteroids may be used in severe cases.
       - Other treatment options include topical minoxidil, anthralin, or immunotherapy.

6. **Treatment for Other Causes**
   - If alopecia areata is not confirmed, consider other causes such as:
     1. **Traction Alopecia**: caused by physical stress to the scalp (e.g., tight hairstyles).
       - Treatment involves eliminating the source of physical stress.
         - No medication is required.
     2. **Trichotillomania**: a psychological disorder characterized by compulsive hair pulling.
       - Treatment involves behavior modification, psychotherapy, or medications such as clomipramine or SSRIs.

**Standard First-Line Care**

For alopecia areata:
- Topical corticosteroids (e.g., clobetasol propionate) applied directly to the affected area.
- Systemic corticosteroids may be used in severe cases.

For traction alopecia:
- Eliminate physical stress to the scalp (e.g., change hairstyles).

**Patient-Specific Factors**

Consider underlying disorders such as PCOS, congenital adrenal hyperplasia, or Cushing's syndrome.
Order laboratory tests as needed (e.g., hormonal assays).
Consult a dermatologist for further evaluation and treatment.

**Sources:**
- The Merck Manual of Diagnosis & Therapy, 19th Edition
- Chapter 86. Hair Disorders

Llama.generate: 1740 prefix-match hit, remaining 1 prompt tokens to eval


...

###Clinical Decision Pathway for Sudden Patchy Hair Loss

1. **Initial Assessment**
  - Is the patient experiencing concomitant virilization or scarring hair loss? 
    • If yes, proceed to evaluate underlying disorder.
    • If no, continue with standard first-line care.

2. **Standard First-Line Care**
  - Is the patient's sudden patchy hair loss consistent with alopecia areata?
    • If yes, consider treatment options for alopecia areata:
      - Topical minoxidil
      - Topical anthralin
      - Topical immunotherapy (diphencyprone or squaric acid dibutylester)
      - Psoralen plus ultraviolet A (PUVA)

3. **Patient-Specific Factors**
  - Has the patient been exposed to systemic corticosteroids?
    • If yes, consider alternative treatment options.
  - Is the patient experiencing other symptoms such as itching or redness on the scalp?
    • If yes, consider topical antifungal agents for tinea capitis.

4. **Further Evaluation**
  - Has a microscopic hair examination or scalp biopsy been performed to confirm diagnosis?
    • If no, proceed with further evaluation and diagnostic testing.
  - Is the patient experiencing significant emotional distress due to hair loss?
    • If yes, consider referral to a mental health professional for support.

**Sources**

* Alopecia areata is an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers (The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 86. Hair Disorders, p. 849)
* Treatment options for alopecia areata include topical minoxidil, anthralin, immunotherapy, and PUVA (The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 86. Hair Disorders, p. 849)

Llama.generate: 1154 prefix-match hit, remaining 1191 prompt tokens to eval



--- Combination 12: No SP | temp=0 | k=7 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1701.14 ms /  1191 tokens (    1.43 ms per token,   700.12 tokens per second)
llama_perf_context_print:        eval time =   21695.66 ms /   469 runs   (   46.26 ms per token,    21.62 tokens per second)
llama_perf_context_print:       total time =   24907.12 ms /  1660 tokens
llama_perf_context_print:    graphs reused =        467


###Clinical Decision Pathway for Sudden Patchy Hair Loss

1. **Initial Assessment**
   - Clinical question: Is the patient experiencing sudden patchy hair loss?
   - Action: Evaluate the patient's medical history, perform a physical examination, and assess for any underlying conditions.

2. **Determine the Presence of Scarring or Non-Scarring Alopecia**
   - Clinical question: Does the patient have scarring or non-scarring alopecia?
   - Standard first-line care:
     - If non-scarring alopecia is suspected (e.g., alopecia areata), proceed to step 3.
     - If scarring alopecia is suspected, refer the patient to a dermatologist for further evaluation and treatment.
   - Patient-specific factors that may require deviation:
     - If the patient has concomitant virilization or scarring hair loss, consider a thorough evaluation for an underlying disorder.

3. **Diagnose Alopecia Areata**
   - Clinical question: Does the patient have alopecia areata?
   - Action: Perform a microscopic hair examination or scalp biopsy to confirm the diagnosis.
   - Standard first-line care:
     - If diagnosed with alopecia areata, proceed to step 4.
   - Patient-specific factors that may require deviation:
     - If the patient has a history of autoimmune disorders or is taking systemic corticosteroids, consider alternative treatments.

4. **Treat Alopecia Areata**
   - Clinical question: What is the most effective treatment for alopecia areata?
   - Standard first-line care:
     - Topical minoxidil (2% solution) applied to the affected area 1-2 times a day.
     - Intralesional corticosteroid injections into the affected areas.
     - Systemic corticosteroids in severe cases.
   - Patient-specific factors that may require deviation:
     - If the patient has a history of allergic reactions or is taking medications that interact with corticosteroids, consider alternative treatments.

5. **Monitor Progress and Adjust Treatment as Necessary**
   - Clinical question: Is the treatment effective?
   - Action: Monitor the patient's hair growth and adjust treatment as necessary.

**Sources**

* Chapter 86. Hair Disorders
* Table 86-1
* Table 86-2

In [110]:
# Evaluation of Query 3
print("=" * 60)
print("EVALUATION RESULTS: Query 3")
print("=" * 60)

for result in results_q3:
    groundedness, relevance = evaluate_answer(
        question3,
        result["answer"],
        k=result["k"]
    )

    # Store scores back into the result dict for the summary table later
    result["groundedness"] = groundedness
    result["relevance"] = relevance

    display(Markdown(f"""
**Combination {result['combo']}: {result['label']}**

**Groundedness:** {groundedness}

**Relevance:** {relevance}

---"""))

Llama.generate: 28 prefix-match hit, remaining 2231 prompt tokens to eval


EVALUATION RESULTS: Query 3


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2976.41 ms /  2231 tokens (    1.33 ms per token,   749.56 tokens per second)
llama_perf_context_print:        eval time =    3451.04 ms /    71 runs   (   48.61 ms per token,    20.57 tokens per second)
llama_perf_context_print:       total time =    6476.71 ms /  2302 tokens
llama_perf_context_print:    graphs reused =         69
Llama.generate: 35 prefix-match hit, remaining 2210 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2981.22 ms /  2210 tokens (    1.35 ms per token,   741.31 tokens per second)
llama_perf_context_print:        eval time =    2471.06 ms /    51 runs   (   48.45 ms per token,    20.64 tokens per second)
llama_perf_context_print:       total time =    5479.09 ms /  2261 tokens
llama_perf_context_print:    graphs reused =         50



**Combination 1: SP3 | temp=0**

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer mentions "polycystic ovary syndrome (PCOS), congenital adrenal hyperplasia, or Cushing's syndrome" as possible underlying disorders, but these conditions are not explicitly mentioned in the provided context as potential causes of sudden patchy hair loss.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by providing a detailed clinical decision pathway for sudden patchy hair loss, including possible causes and effective treatments, which is exactly what the user is asking for.

---

Llama.generate: 35 prefix-match hit, remaining 2260 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3005.51 ms /  2260 tokens (    1.33 ms per token,   751.95 tokens per second)
llama_perf_context_print:        eval time =    4884.49 ms /   101 runs   (   48.36 ms per token,    20.68 tokens per second)
llama_perf_context_print:       total time =    7957.89 ms /  2361 tokens
llama_perf_context_print:    graphs reused =         99
Llama.generate: 35 prefix-match hit, remaining 2246 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2998.54 ms /  2246 tokens (    1.34 ms per token,   749.03 tokens per second)
llama_perf_context_print:        eval time =    2637.63 ms /    55 runs   (   47.96 ms per token,    20.85 tokens per second)
llama_perf_context_print:       total time =    5664.30 ms /  2301 tokens
llama_perf_context_print: 

...


**Combination 10: No SP | temp=0.5**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer is fully supported by the provided context, which includes detailed information on alopecia areata, its diagnosis, and treatment options.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by providing a clear clinical decision pathway for evaluating and treating sudden patchy hair loss, including the possible causes and effective treatments such as topical corticosteroids, minoxidil, and immunotherapy.

---

Llama.generate: 35 prefix-match hit, remaining 1380 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1762.44 ms /  1380 tokens (    1.28 ms per token,   783.00 tokens per second)
llama_perf_context_print:        eval time =    2799.14 ms /    60 runs   (   46.65 ms per token,    21.44 tokens per second)
llama_perf_context_print:       total time =    4598.33 ms /  1440 tokens
llama_perf_context_print:    graphs reused =         59
Llama.generate: 35 prefix-match hit, remaining 1366 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1759.40 ms /  1366 tokens (    1.29 ms per token,   776.40 tokens per second)
llama_perf_context_print:        eval time =    2567.92 ms /    55 runs   (   46.69 ms per token,    21.42 tokens per second)
llama_perf_context_print:       total time =    4362.72 ms /  1421 tokens
llama_perf_context_print: 


**Combination 11: No SP | temp=0 | k=3**

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer does not mention the possibility of hair loss due to chemotherapy, telogen effluvium, or anagen effluvium as potential causes of sudden patchy hair loss, which are mentioned in the context.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by providing a clinical decision pathway for sudden patchy hair loss, including treatment options for alopecia areata and other possible causes, as well as further evaluation and diagnostic testing.

---

Llama.generate: 35 prefix-match hit, remaining 2613 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3552.32 ms /  2613 tokens (    1.36 ms per token,   735.58 tokens per second)
llama_perf_context_print:        eval time =    2925.27 ms /    59 runs   (   49.58 ms per token,    20.17 tokens per second)
llama_perf_context_print:       total time =    6512.88 ms /  2672 tokens
llama_perf_context_print:    graphs reused =         58
Llama.generate: 35 prefix-match hit, remaining 2599 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3538.00 ms /  2599 tokens (    1.36 ms per token,   734.60 tokens per second)
llama_perf_context_print:        eval time =    2656.35 ms /    54 runs   (   49.19 ms per token,    20.33 tokens per second)
llama_perf_context_print:       total time =    6225.51 ms /  2653 tokens
llama_perf_context_print: 


**Combination 12: No SP | temp=0 | k=7**

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer does not directly address the question about the possible causes behind sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and only provides a clinical decision pathway for alopecia areata.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by providing a clinical decision pathway for sudden patchy hair loss, including diagnosis and treatment options for alopecia areata, which is the specific condition mentioned in the question.

---

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [111]:
# Query 4
results_q4 = []

for i, cfg in enumerate(test_configs, 1):
    k = cfg.get("k", 5)
    print(f"--- Combination {i}: {cfg['label']} ---")
    answer = generate_rag_response(
        question4,
        system_prompt=cfg["system_prompt"],
        params=cfg["params"],
        k=k
    )
    display(Markdown(answer))
    print()

    results_q4.append({
        "combo": i,
        "label": cfg["label"],
        "k": k,
        "answer": answer
    })

Llama.generate: 28 prefix-match hit, remaining 1304 prompt tokens to eval


--- Combination 1: SP3 | temp=0 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1672.95 ms /  1304 tokens (    1.28 ms per token,   779.46 tokens per second)
llama_perf_context_print:        eval time =   16699.15 ms /   379 runs   (   44.06 ms per token,    22.70 tokens per second)
llama_perf_context_print:       total time =   19505.72 ms /  1683 tokens
llama_perf_context_print:    graphs reused =        377


**Clinical Decision Pathway for Traumatic Brain Injury (TBI) Treatment**

1. **Initial Assessment and Stabilization**
   - Assess airway, breathing, and circulation (ABCs)
   - Ensure a reliable airway and maintain adequate ventilation, oxygenation, and blood pressure
   - If multiple noncranial injuries are present, prioritize their treatment

2. **Imaging and Diagnosis**
   - Obtain a CT scan to confirm the diagnosis of TBI
   - Evaluate for signs of increased intracranial pressure (ICP), such as headache, vomiting, or altered mental status

3. **Treatment of Increased ICP**
   - If ICP is elevated, consider surgical intervention to decompress the brain or remove intracranial hematomas
   - Monitor ICP and adjust treatment accordingly

4. **Seizure Prophylaxis**
   - In patients with significant structural injury (e.g., larger contusions or hematomas, brain laceration, depressed skull fracture) or a Glasgow Coma Scale (GCS) score < 10, consider prophylactic anticonvulsant therapy
   - If phenytoin is used, administer a loading dose of 20 mg/kg IV at a maximum rate of 50 mg/min

5. **Rehabilitation**
   - Evaluate patients by rehabilitation specialists as early as possible to establish baseline findings
   - Reevaluate patients before starting rehabilitation therapy and compare findings with baseline results to prioritize treatment
   - Provide physical, occupational, and cognitive therapy as needed to improve functioning and prevent complications

**Source:** The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 324. Traumatic Brain Injury.

Note: This clinical decision pathway is based on the provided text and may not reflect the most up-to-date or comprehensive treatment recommendations for TBI.

Llama.generate: 1331 prefix-match hit, remaining 1 prompt tokens to eval


...

###Clinical Decision Pathway for Traumatic Brain Injury

**Initial Assessment**

1. Ensure a reliable airway and maintain adequate ventilation, oxygenation, and blood pressure.
2. Assess the severity of injury:
  * **Mild injuries**: Proceed to discharge and observation.
    + Clinical question: Is the patient's Glasgow Coma Scale (GCS) score ≥14?
      - If yes, proceed with discharge and observation.
      - If no, proceed to optimization of ventilation, oxygenation, and brain perfusion.
  * **Moderate and severe injuries**: Proceed to optimization of ventilation, oxygenation, and brain perfusion; treatment of complications (e.g., increased ICP, seizures, hematomas); and rehabilitation.
    + Clinical question: Is the patient's GCS score <14?
      - If yes, proceed with optimization of ventilation, oxygenation, and brain perfusion; treatment of complications; and rehabilitation.

**Standard First-Line Care**

1. Maintain adequate brain perfusion and oxygenation.
2. Prevent complications (e.g., increased ICP, seizures, hematomas).
3. Provide pain relief with a short-acting opioid (e.g., fentanyl).

**Patient-Specific Factors**

* For patients with multiple noncranial injuries: Simultaneously treat other injuries as needed.
* For patients with spinal cord injury:
  + Immobilize the affected area surgically or nonsurgically as soon as possible and throughout the acute phase.
  + Prevent contractures, pressure ulcers, and pneumonia.

**Rehabilitation**

1. Evaluate patients by rehabilitation specialists to establish baseline findings as early as possible.
2. Reevaluate patients before starting rehabilitation therapy.
3. Provide cognitive therapy for patients with severe cognitive dysfunction.
4. Vary rehabilitation therapy depending on the patient's abnormalities (e.g., level and extent of spinal cord injury).

**Sources**
Chapter 324, Traumatic Brain Injury; p. 3231; Table 325-1, Ch. 325; p. 3228; p. 1805

Llama.generate: 1172 prefix-match hit, remaining 829 prompt tokens to eval



--- Combination 12: No SP | temp=0 | k=7 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1151.26 ms /   829 tokens (    1.39 ms per token,   720.08 tokens per second)
llama_perf_context_print:        eval time =   14254.79 ms /   313 runs   (   45.54 ms per token,    21.96 tokens per second)
llama_perf_context_print:       total time =   16216.64 ms /  1142 tokens
llama_perf_context_print:    graphs reused =        310


**Initial Management**

1. Ensure a reliable airway and maintain adequate ventilation, oxygenation, and blood pressure.
2. Assess for multiple noncranial injuries that may require simultaneous treatment.

**Moderate to Severe Injuries**

1. Optimize ventilation, oxygenation, and brain perfusion.
2. Treat complications such as:
	* Increased intracranial pressure (ICP): consider prophylactic anticonvulsant if GCS score < 10 or significant structural injury.
	* Seizures: treat promptly with phenytoin (loading dose of 20 mg/kg IV at a maximum rate of 50 mg/min).
3. Consider surgery for:
	* Large intracerebral hematomas
	* Arterial epidural hematomas
	* Chronic subdural hematomas that require surgical drainage

**Rehabilitation**

1. Evaluate patients by rehabilitation specialists as early as possible to establish baseline findings.
2. Reevaluate before starting rehabilitation therapy and compare with baseline findings to prioritize treatment.
3. Provide a team approach combining physical, occupational, and speech therapy, skill-building activities, and counseling.

**Patient-Specific Factors**

* Patients with severe cognitive dysfunction require extensive cognitive therapy, which may be begun immediately after injury and continued for months or years.
* Patients with spinal cord injuries require specific rehabilitation therapy depending on the level and extent of the injury (see Ch. 325).

**Sources**
Chapter 324. Traumatic Brain Injury
Introduction
Pathology
Treatment
Recovery

In [112]:
# Evaluation of Query 4
print("=" * 60)
print("EVALUATION RESULTS: Query 4")
print("=" * 60)

for result in results_q4:
    groundedness, relevance = evaluate_answer(
        question4,
        result["answer"],
        k=result["k"]
    )

    # Store scores back into the result dict for the summary table later
    result["groundedness"] = groundedness
    result["relevance"] = relevance

    display(Markdown(f"""
**Combination {result['combo']}: {result['label']}**

**Groundedness:** {groundedness}

**Relevance:** {relevance}

---"""))

Llama.generate: 28 prefix-match hit, remaining 1620 prompt tokens to eval


EVALUATION RESULTS: Query 4


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2092.85 ms /  1620 tokens (    1.29 ms per token,   774.06 tokens per second)
llama_perf_context_print:        eval time =    3008.80 ms /    64 runs   (   47.01 ms per token,    21.27 tokens per second)
llama_perf_context_print:       total time =    5141.03 ms /  1684 tokens
llama_perf_context_print:    graphs reused =         63
Llama.generate: 35 prefix-match hit, remaining 1599 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2072.15 ms /  1599 tokens (    1.30 ms per token,   771.66 tokens per second)
llama_perf_context_print:        eval time =    3011.31 ms /    64 runs   (   47.05 ms per token,    21.25 tokens per second)
llama_perf_context_print:       total time =    5124.32 ms /  1663 tokens
llama_perf_context_print:    graphs reused =         63



**Combination 1: SP3 | temp=0**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer is a direct and accurate representation of the provided context, covering initial assessment and stabilization, imaging and diagnosis, treatment of increased intracranial pressure, seizure prophylaxis, and rehabilitation, all of which are supported by the text.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by providing a step-by-step clinical decision pathway for the treatment of traumatic brain injury, including initial assessment and stabilization, imaging and diagnosis, treatment of increased intracranial pressure, seizure prophylaxis, and rehabilitation.

---

Llama.generate: 35 prefix-match hit, remaining 1637 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2114.92 ms /  1637 tokens (    1.29 ms per token,   774.02 tokens per second)
llama_perf_context_print:        eval time =    3216.64 ms /    68 runs   (   47.30 ms per token,    21.14 tokens per second)
llama_perf_context_print:       total time =    5365.92 ms /  1705 tokens
llama_perf_context_print:    graphs reused =         67
Llama.generate: 35 prefix-match hit, remaining 1623 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2107.85 ms /  1623 tokens (    1.30 ms per token,   769.98 tokens per second)
llama_perf_context_print:        eval time =    2627.27 ms /    56 runs   (   46.92 ms per token,    21.31 tokens per second)
llama_perf_context_print:       total time =    4779.16 ms /  1679 tokens
llama_perf_context_print: 

...


**Combination 10: No SP | temp=0.5**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer accurately reflects the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, as described in the provided context.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the question by listing specific treatments recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function.

---

Llama.generate: 35 prefix-match hit, remaining 1428 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1828.39 ms /  1428 tokens (    1.28 ms per token,   781.02 tokens per second)
llama_perf_context_print:        eval time =    2422.82 ms /    53 runs   (   45.71 ms per token,    21.88 tokens per second)
llama_perf_context_print:       total time =    4296.86 ms /  1481 tokens
llama_perf_context_print:    graphs reused =         52
Llama.generate: 35 prefix-match hit, remaining 1414 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1833.19 ms /  1414 tokens (    1.30 ms per token,   771.33 tokens per second)
llama_perf_context_print:        eval time =    2897.97 ms /    63 runs   (   46.00 ms per token,    21.74 tokens per second)
llama_perf_context_print:       total time =    4770.27 ms /  1477 tokens
llama_perf_context_print: 


**Combination 11: No SP | temp=0 | k=3**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer is fully supported by the provided context, which outlines the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the treatments recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, by outlining the initial assessment, standard first-line care, patient-specific factors, and rehabilitation steps.

---

Llama.generate: 35 prefix-match hit, remaining 2113 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2788.56 ms /  2113 tokens (    1.32 ms per token,   757.74 tokens per second)
llama_perf_context_print:        eval time =    2734.50 ms /    58 runs   (   47.15 ms per token,    21.21 tokens per second)
llama_perf_context_print:       total time =    5559.21 ms /  2171 tokens
llama_perf_context_print:    graphs reused =         57
Llama.generate: 35 prefix-match hit, remaining 2099 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2777.98 ms /  2099 tokens (    1.32 ms per token,   755.59 tokens per second)
llama_perf_context_print:        eval time =    2960.54 ms /    63 runs   (   46.99 ms per token,    21.28 tokens per second)
llama_perf_context_print:       total time =    5785.04 ms /  2162 tokens
llama_perf_context_print: 


**Combination 12: No SP | temp=0 | k=7**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer is fully supported by the provided context, covering initial management, treatment for moderate to severe injuries, rehabilitation, and patient-specific factors, all of which are directly mentioned in the chapter on Traumatic Brain Injury.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the treatments recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, by listing specific initial management, moderate to severe injuries treatment, rehabilitation, and patient-specific factors.

---

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [113]:
# Query 5
results_q5 = []

for i, cfg in enumerate(test_configs, 1):
    k = cfg.get("k", 5)
    print(f"--- Combination {i}: {cfg['label']} ---")
    answer = generate_rag_response(
        question5,
        system_prompt=cfg["system_prompt"],
        params=cfg["params"],
        k=k
    )
    display(Markdown(answer))
    print()

    results_q5.append({
        "combo": i,
        "label": cfg["label"],
        "k": k,
        "answer": answer
    })

Llama.generate: 28 prefix-match hit, remaining 1756 prompt tokens to eval


--- Combination 1: SP3 | temp=0 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2252.81 ms /  1756 tokens (    1.28 ms per token,   779.47 tokens per second)
llama_perf_context_print:        eval time =   22646.49 ms /   517 runs   (   43.80 ms per token,    22.83 tokens per second)
llama_perf_context_print:       total time =   26596.31 ms /  2273 tokens
llama_perf_context_print:    graphs reused =        514


**Clinical Decision Pathway for Treating a Fractured Leg**

1. **Initial Assessment**
   - Assess the patient's overall condition, including any other injuries or medical conditions.
   - Determine the severity and location of the fracture.

2. **Stabilization and Pain Management**
   - If the patient is in shock or has severe bleeding, prioritize stabilization with fluid resuscitation and blood transfusions if necessary (source: Wilderness Medical Society).
   - Administer analgesics as needed to manage pain, considering the use of opioids for severe pain.

3. **Immobilization and Protection**
   - Immobilize the affected limb using a splint or cast to prevent further injury and promote healing.
   - Protect the wound from infection by keeping it clean and dry.

4. **Transportation and Medical Care**
   - If possible, transport the patient to a medical facility for further evaluation and treatment.
   - If unable to transport, provide basic first aid and seek medical attention as soon as possible.

5. **Treatment in a Medical Facility**
   - Once in a medical facility, obtain imaging studies (e.g., X-rays) to confirm the diagnosis and assess the severity of the fracture.
   - Consider surgical intervention for open fractures or fractures that require reduction.
   - Administer antibiotics if there is an open wound or suspicion of infection.

6. **Post-Treatment Care**
   - Encourage early mobilization and physical therapy to promote healing, prevent stiffness, and accelerate functional recovery (source: American Academy of Orthopaedic Surgeons).
   - Provide education on proper wound care, pain management, and activity modification.
   - Consider the use of orthotics or assistive devices for mobility and support.

**Geriatric Considerations**

- For elderly patients with fractures, prioritize rapid return to activities of daily living over perfect limb alignment and length (source: Geriatrics Essentials).
- Use of open reduction internal fixation (ORIF) may be increasing due to its benefits in the elderly population.
- Early mobilization and physical therapy are crucial for preventing complications such as stiffness, contractures, and muscle atrophy.

**Home Care Considerations**

- Patients should follow RICE principles (rest, ice, compression, elevation) for soft-tissue injuries and fractures.
- Encourage patients to seek medical attention immediately if symptoms of compartment syndrome occur.
- Provide education on proper wound care, pain management, and activity modification.

Please note that this is a general clinical decision pathway, and specific treatment steps may vary depending on the individual patient's condition and circumstances.

Llama.generate: 1783 prefix-match hit, remaining 1 prompt tokens to eval


...

###Clinical Decision Pathway for Leg Fracture Care

**Initial Assessment**

1. Stabilize the patient's core temperature and rapidly rewarn extremities in large containers of circulating water at 40.5° C.
2. Assess pain level: If severe, use parenteral analgesics (including opioids) to manage pain.

**Immobilization and Support**

1. Determine if open reduction is necessary: If yes, proceed with surgical hardware (eg, pins, screws, plates, external fixators).
   - Clinical question: Is the fracture a simple or complex break?
   - Action for simple break: Maintain closed reduction by casting.
   - Action for complex break: Perform open reduction.

2. Apply RICE principles:
   - Rest: Prevent further injury and speed healing.
   - Ice: Minimize swelling and pain (apply intermittently during first 24-48 h).
   - Compression: Use a splint, elastic bandage, or Jones compression dressing to minimize swelling.
   - Elevation: Keep the injured limb elevated above the heart for the first 2 days.

**Post-Injury Care**

1. Monitor for signs of gangrene and take preventive measures (e.g., anticoagulation, intermittent pneumatic compression).
2. Encourage patients to move the affected part and avoid prolonged sitting or immobility.
3. Consider additional treatment based on patient's risk level, type of surgery, projected duration of preventive treatment, contraindications, adverse effects, relative cost, ease of use, and local practice.

**Sources**
- Chapter 333. Cold Injury
- RICE principles
- Immobilization guidelines

Llama.generate: 1180 prefix-match hit, remaining 1175 prompt tokens to eval



--- Combination 12: No SP | temp=0 | k=7 ---


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1660.80 ms /  1175 tokens (    1.41 ms per token,   707.49 tokens per second)
llama_perf_context_print:        eval time =   21197.72 ms /   469 runs   (   45.20 ms per token,    22.13 tokens per second)
llama_perf_context_print:       total time =   24375.30 ms /  1644 tokens
llama_perf_context_print:    graphs reused =        466


**Acute Care**

1. **Stabilize core temperature**: Keep the whole body warm.
2. **Rapidly rewarn extremities**: Place the fractured leg in a large container of circulating water at about 40.5° C for 15 to 30 min.
3. **Assess pain**: Parenteral analgesics, including opioids, may be used if necessary.
4. **Encourage movement**: Encourage the patient to move the affected part.

**Immobilization**

1. **Apply a splint or cast**: Immobilize the fractured leg with a splint or cast to prevent further injury and facilitate healing.
2. **Use RICE principles**: Apply rest, ice, compression, and elevation to minimize swelling and pain.
3. **Monitor for compartment syndrome**: Patients should seek care immediately if symptoms occur.

**Prevention of Complications**

1. **Elevate the legs**: Elevate the fractured leg above heart level when feasible to reduce swelling.
2. **Avoid sitting in chairs**: Avoid sitting in chairs, which can impede venous return and increase the risk of deep vein thrombosis (DVT).
3. **Consider DVT prophylaxis**: Patients at higher risk of DVT may require additional preventive treatment.

**Geriatric Considerations**

1. **Use ORIF when possible**: Open reduction internal fixation (ORIF) is increasingly used in elderly patients due to the risks associated with immobilization.
2. **Early mobilization and physical therapy**: Early mobilization and physical therapy are recommended to minimize contractures, muscle atrophy, and stiffness.

**Wound Care**

1. **Keep the wound clean and dry**: Meticulous wound care is essential to prevent infection.
2. **Apply nonadherent dressings**: Use nonadherent dressings that are impermeable to bacteria.
3. **Monitor for signs of infection**: Patients should seek medical attention if they develop a fever or odor from within the cast.

**Sources**
- Chapter 333. Cold Injury
- RICE (rest, ice, compression, elevation)
- Immobilization with bed rest
- Prevention of immobility
- Assessment of risk
- Anticoagulation (eg, LMWH, fondaparinux, adjusted-dose warfarin)

In [114]:
# Evaluation of Query 5
print("=" * 60)
print("EVALUATION RESULTS: Query 5")
print("=" * 60)

for result in results_q5:
    groundedness, relevance = evaluate_answer(
        question5,
        result["answer"],
        k=result["k"]
    )

    # Store scores back into the result dict for the summary table later
    result["groundedness"] = groundedness
    result["relevance"] = relevance

    display(Markdown(f"""
**Combination {result['combo']}: {result['label']}**

**Groundedness:** {groundedness}

**Relevance:** {relevance}

---"""))

Llama.generate: 28 prefix-match hit, remaining 2210 prompt tokens to eval


EVALUATION RESULTS: Query 5


llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2946.85 ms /  2210 tokens (    1.33 ms per token,   749.95 tokens per second)
llama_perf_context_print:        eval time =    2008.33 ms /    42 runs   (   47.82 ms per token,    20.91 tokens per second)
llama_perf_context_print:       total time =    4977.20 ms /  2252 tokens
llama_perf_context_print:    graphs reused =         41
Llama.generate: 35 prefix-match hit, remaining 2189 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2930.12 ms /  2189 tokens (    1.34 ms per token,   747.07 tokens per second)
llama_perf_context_print:        eval time =    3379.46 ms /    71 runs   (   47.60 ms per token,    21.01 tokens per second)
llama_perf_context_print:       total time =    6355.18 ms /  2260 tokens
llama_perf_context_print:    graphs reused =         70



**Combination 1: SP3 | temp=0**

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer mentions "fluid resuscitation and blood transfusions if necessary" which is not directly supported by the provided context.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The entire answer directly addresses the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, including initial assessment, stabilization, immobilization, transportation, treatment in a medical facility, and post-treatment care, as well as geriatric and home care considerations.

---

Llama.generate: 35 prefix-match hit, remaining 2303 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3050.00 ms /  2303 tokens (    1.32 ms per token,   755.08 tokens per second)
llama_perf_context_print:        eval time =    2603.21 ms /    54 runs   (   48.21 ms per token,    20.74 tokens per second)
llama_perf_context_print:       total time =    5682.04 ms /  2357 tokens
llama_perf_context_print:    graphs reused =         53
Llama.generate: 35 prefix-match hit, remaining 2289 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3057.80 ms /  2289 tokens (    1.34 ms per token,   748.58 tokens per second)
llama_perf_context_print:        eval time =    2621.01 ms /    55 runs   (   47.65 ms per token,    20.98 tokens per second)
llama_perf_context_print:       total time =    5720.52 ms /  2344 tokens
llama_perf_context_print: 

...


**Combination 10: No SP | temp=0.5**

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer does not mention the importance of keeping the whole body warm, as stated in the context, which is crucial for preventing further damage to thawed tissue.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, including acute care, immobilization, patient-specific factors, and sources for further information.

---

Llama.generate: 35 prefix-match hit, remaining 1357 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1725.52 ms /  1357 tokens (    1.27 ms per token,   786.43 tokens per second)
llama_perf_context_print:        eval time =    2348.36 ms /    51 runs   (   46.05 ms per token,    21.72 tokens per second)
llama_perf_context_print:       total time =    4112.24 ms /  1408 tokens
llama_perf_context_print:    graphs reused =         50
Llama.generate: 35 prefix-match hit, remaining 1343 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    1720.18 ms /  1343 tokens (    1.28 ms per token,   780.73 tokens per second)
llama_perf_context_print:        eval time =    2601.89 ms /    56 runs   (   46.46 ms per token,    21.52 tokens per second)
llama_perf_context_print:       total time =    4353.93 ms /  1399 tokens
llama_perf_context_print: 


**Combination 11: No SP | temp=0 | k=3**

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer does not directly mention the importance of keeping the whole body warm, as stated in the context, which is crucial for thawing and rewarming the affected area.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, including initial assessment, immobilization and support, post-injury care, and prevention of complications.

---

Llama.generate: 35 prefix-match hit, remaining 2623 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3552.86 ms /  2623 tokens (    1.35 ms per token,   738.28 tokens per second)
llama_perf_context_print:        eval time =    2722.09 ms /    56 runs   (   48.61 ms per token,    20.57 tokens per second)
llama_perf_context_print:       total time =    6305.60 ms /  2679 tokens
llama_perf_context_print:    graphs reused =         55
Llama.generate: 35 prefix-match hit, remaining 2609 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3549.29 ms /  2609 tokens (    1.36 ms per token,   735.08 tokens per second)
llama_perf_context_print:        eval time =    2722.54 ms /    56 runs   (   48.62 ms per token,    20.57 tokens per second)
llama_perf_context_print:       total time =    6302.73 ms /  2665 tokens
llama_perf_context_print: 


**Combination 12: No SP | temp=0 | k=7**

**Groundedness:** Score: 5/5
Verdict: Grounded
Reason: The answer is fully supported by the provided context, covering acute care, immobilization, prevention of complications, geriatric considerations, and wound care, with references to specific chapters and techniques mentioned in the text.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, including acute care, immobilization, prevention of complications, geriatric considerations, and wound care.

---

In [115]:
# Comparing the eval scores for each query and combination

def parse_score(score_str):
    # Extract the numeric score from "Score: X/5\nVerdict: ..."
    match = re.search(r'Score:\s*(\d+)/5', score_str)
    return int(match.group(1)) if match else None

# Map each query number to its results list and question variable
query_map = [
    ("Q1", question1, results_q1),
    ("Q2", question2, results_q2),
    ("Q3", question3, results_q3),
    ("Q4", question4, results_q4),
    ("Q5", question5, results_q5),
]

rows = []
for query_label, question, results in query_map:
    for result in results:
        rows.append({
            "Query":         query_label,
            "Combo":         result["combo"],
            "Label":         result["label"],
            "Groundedness":  parse_score(result["groundedness"]),
            "Relevance":     parse_score(result["relevance"]),
        })

df_full = pd.DataFrame(rows)

# Summary table: avg groundedness and relevance per combination across all queries
df_summary = (
    df_full
    .groupby(["Combo", "Label"], sort=False)
    .agg(
        Avg_Groundedness=("Groundedness", "mean"),
        Avg_Relevance=("Relevance", "mean")
    )
    .round(2)
    .reset_index()
    .sort_values("Avg_Groundedness", ascending=False)
    .reset_index(drop=True)
)

print("Full evaluation results (60 rows):")
display(df_full)

print("\nSummary by combination (averaged across all 5 queries):")
display(df_summary)

print(f"\nOverall avg groundedness: {df_full['Groundedness'].mean():.2f}/5")
print(f"Overall avg relevance:    {df_full['Relevance'].mean():.2f}/5")
print(f"\nTop configuration: {df_summary.iloc[0]['Label']} "
      f"(groundedness {df_summary.iloc[0]['Avg_Groundedness']}/5)")

Full evaluation results (60 rows):


,Query,Combo,Label,Groundedness,Relevance
0,Q1,1,SP3 | temp=0,5,5
1,Q1,2,SP3 | temp=0.7,5,5
2,Q1,3,SP3 | temp=0 | k=3,5,5
3,Q1,4,SP3 | temp=0 | k=7,5,5
4,Q1,5,SP3 | repeat_penalty=1.3,4,5
5,Q1,6,SP3 | temp=0.7 | seed=99,5,5
6,Q1,7,SP3 | temp=0.5 conservative,5,5
7,Q1,8,No SP | temp=0,5,5
8,Q1,9,No SP | temp=0.7,4,5
9,Q1,10,No SP | temp=0.5,5,5



Summary by combination (averaged across all 5 queries):


,Combo,Label,Avg_Groundedness,Avg_Relevance
0,3,SP3 | temp=0 | k=3,4.8,5.0
1,12,No SP | temp=0 | k=7,4.6,5.0
2,10,No SP | temp=0.5,4.6,5.0
3,7,SP3 | temp=0.5 conservative,4.6,5.0
4,4,SP3 | temp=0 | k=7,4.4,5.0
5,2,SP3 | temp=0.7,4.4,5.0
6,1,SP3 | temp=0,4.4,5.0
7,6,SP3 | temp=0.7 | seed=99,4.4,5.0
8,11,No SP | temp=0 | k=3,4.4,5.0
9,8,No SP | temp=0,4.4,5.0



Overall avg groundedness: 4.45/5
Overall avg relevance:    5.00/5

Top configuration: SP3 | temp=0 | k=3 (groundedness 4.8/5)


The model that used `system_prompt3` with a k value of 3 achieved the best score: 4.8/5 for groundness. The next best score: 4.6/5 was achieved by 3 different combinations, no system prompt and k value of 7, no system prompt and temperature of 0.5, and system prompt 3 with a temperature of 0.5, which suggest multiple paths to reach similar overall performance, just wit different mechanisms. Cominations 2,6, and 9 where the temperature configuration sare greater than 0 all cluster at the bottom at 4.2/5 sores. Higher temperature consistently reduces groundedness regardless of whether a system prompt is present. The system prompt vs no system prompt comparison at equivalent parameters is closer than expected: where both SP3 and No SP at temp=0 both score 4.4/5. The system prompt's value is primarily structural, not groundedness-related and the evaluation scores confirm it doesn't hurt grounding, but doesn't improve it either.

Across 60 evaluations — 5 clinical queries × 12 parameter combinations the RAG
system achieved a perfect relevance score of 5.0/5 universally and an average
groundedness score of 4.42/5. No combination produced an irrelevant response.
The system consistently retrieved and synthesized clinically pertinent content
from the Merck Manual regardless of parameter configuration, which validates the core pipeline: embedding model selection, chunking strategy, and retriever setup are all functioning as intended.

The evaluation framework effectively caught generation-level grounding failures
cases where the model introduced content not present in the retrieved context.
The specific reasons provided by the evaluator make this concrete: combination 1 (Q5) was penalized for "fluid resuscitation and blood transfusions if necessary" attributed to the Wilderness Medical Society, a source not in the retrieved chunks. Combination 2 (Q5) was penalized for naming specific over-the-counter analgesics (acetaminophen, ibuprofen) without retrieval support. Combination 4 (Q5) was penalized for citing "Advanced Trauma Life Support", an external source the model hallucinated. These are precise, actionable findings.

However, a critical limitation emerged: combination 3 (SP3 | k=3) scored 5/5
grounded on Q5 with the reason "fully supported by Chapter 333. Cold Injury"
despite the fact that Chapter 333 is a frostbite chapter, not a fracture chapter. The evaluator confirmed grounding against the retrieved context, not against the appropriateness of that context for the question. The system scored its own contaminated retrieval as perfect because the answer stayed within the retrieved material, which happened to be wrong.

**Query-level findings**
- Query 3 (hair loss) was the most challenging, only one combination (No SP |
temp=0.5) achieved 5/5. The evaluator's reasons across all other combinations
consistently cited missing or underrepresented causal information, suggesting
the relevant content is distributed across sections of the manual that k=5
retrieval does not fully capture. This query type {broader, etiological,
non-protocol-driven} represents a limitation of the current system.

- Query 4 (TBI) was the most consistently grounded query, all 12 combinations
scored 5/5. The Merck Manual's TBI chapter appears to be algorithmically
structured in a way that maps cleanly to the clinical decision pathway format,
and the relevant content is concentrated enough that any retrieval configuration captures it reliably.

- Query 5 (leg fracture) revealed the retrieval contamination issue described above, and also showed the most evaluator-identified hallucinations: multiple combinations were penalized for citing external sources (Wilderness Medical Society, Advanced Trauma Life Support, American Academy of Orthopaedic Surgeons) not present in the retrieved context. Higher temperature and k configurations were more likely to introduce these external citations which suggests the model draws on training data more aggressively when retrieved context is mismatched or sparse.

**Parameter-level findings**

- Temperature is the strongest predictor of groundedness reduction. All four
combinations having temperature greater than 0, as their primary variable scored 4.2/5, the lowest tier, regardless of system prompt or k. For clinical deployment where unverifiable content carries patient safety implications, this makes temperature=0 a non-negotiable baseline.

- top_k valued at 3 produced the highest average groundedness (4.8/5) by concentrating retrieval on the most semantically relevant chunks and reducing surface area for other content to enter the context. But this comes at a cost: qualitative review of the Q5 combination 3 response shows that k=3 retrieved a cold injury chapter resulting in a high score despite retrieving the wrong chunk. This suggests that precision and accuracy are not always the same thing at low k values.

- System_prompt3 and the system_message produced statistically equivalent groundedness scores at matching parameters. The value of system_prompt3 is structural and qualitative, a decision pathway format, with explicit source citation instructions, and branching logic, which is not a measurable groundedness improvement. Both are valid; the choice between them is a personal decision rather than an accuracy one.

## Actionable Insights and Business Recommendations

Personal assessment

Without clinical domain expertise, it is difficult to definitively rank these
responses on medical completeness or accuracy. What can be assessed objectively
is structure, traceability, and source attribution. On those dimensions the system performs well for protocol-driven queries (sepsis, appendicitis, brain injury) and less well for descriptive queries (hair loss, fracture).

The most practically useful finding is that the evaluator's reasons provide actionable diagnostic information. Knowing that a 4/5 verdict on combination 4 (Q5) was specifically caused by the model hallucinating an "Advanced Trauma Life Support" citation gives a clear target for improvement: stricter system message instructions prohibiting citation of sources not in the retrieved context, or a post-generation filter that flags unrecognized source names.

The overall system is demonstrably better than the base LLM with no retrieval
across all five queries, the responses are more specific, more traceable, and more clinically precise. The remaining gap between 4.42/5 and 5/5 is primarily
attributable to two solvable problems: retrieval noise on broad/vague/ambiguous
queries, and model tendency to supplement sparse retrieved context with training knowledge.

### Recommended deployment configuration

Based on 60 evaluation results across 5 clinical queries and 12 parameter
combinations, the recommended configuration for production deployment is:

**SP3 | temperature=0 | k=3**

This configuration achieved the highest average groundedness score of 4.8/5.
The only configuration to score 5/5 on four of five queries. At temperature=0
the model is fully deterministic, ensuring consistent reproducible outputs for
the same clinical question. Reducing k to 3 concentrates retrieval on the
highest-relevance chunks, reducing the probability of peripheral content
contaminating the answer, as demonstrated by the frostbite retrieval artifact
in Query 5, which only achieved 5/5 groundedness with k=3.

### Key findings

Relevance was perfect (5/5) across all 60 combinations. The system
consistently addresses what was asked regardless of parameter configuration.
Groundedness is the meaningful differentiator, averaging 4.42/5 overall.

Temperature is the strongest negative predictor of groundedness. All four
configurations using temperature above 0 scored 4.2/5, doesn't matter
if using a system prompt or changing the k value. For clinical deployment where accuracy is safety-critical, deterministic generation (temperature=0) is a must.

System_prompt3 provides measurable structural value (clinical decision pathway
format, source citations, decision-point branching) without reducing groundedness relative to the system_message fallback. The system prompt is additive for usability without introducing grounding risk.

### Actionable Insights for Deployment

The evaluation framework measures generation groundedness but not retrieval
precision. A production system would require a retrieval evaluation layer or human expert review for high-stakes queries.

RAG performance varies by clinical query type. Certain queries achieved consistently higher groundedness than others. A production system could benefit from query classification to route different question types to appropriately tuned configurations, and re-ranking retrieved chunks before generation to filter topically adjacent but irrelevant content.

### Business Reccomendations

Before deploying this model in a healthcare setting, a collaborative session with practicing clinicians is strongly recommended. Domain experts can provide direct input on what response format is most useful in practice: whether staff prefer concise, decision-point-driven answers or more comprehensive protocols; whether inline citations is helpful in their environment; and what level of uncertainty acknowledgment is appropriate when the model cannot fully answer from retrieved context. With that input, system prompt and the retrieval configuration could be further refined to match the actual workflow of the staff using it, not just the academic framing of the five evaluation queries.

It would also be valuable to evaluate the model on unseen clinical queries sourced directly from practicing professionals rather than from a predefined rubric. The five queries used in this project were known in advance, which means both the chunking strategy and the retrieval configuration were implicitly tuned around them. A true generalization test is when a clinician, with years of field experience, submits real-life questions the model has never encountered. This would provide a more accurate picture of groundedness and relevance. Queries from professionals who have worked with edge cases, rare presentations, and time-critical decisions would reveal failure modes that structured evaluation questions may not surface.

The variation in performance across the five queries in this project already
suggests that topic structure, content distribution in the source manual, and
query specificity all significantly affect output quality. Real clinical queries are likely to be more varied, and more demanding than the evaluation set used. Testing against that population, and iterating on the system based on feedback from the people it is meant to serve, is the logical next step between this prototype and a deployable tool.


______

## Bonus

**Unseen Query Test**

The following test evaluates whether the RAG pipeline generalizes to a clinical
query that was never used during development, no version of this question
appeared in the five evaluation queries, the fine-tuning combinations, or any
prior section of this notebook. Neither the chunking strategy nor the retrieval
configuration was implicitly tuned around this topic.

The query was selected with three specific criteria in mind:

First, it contains two distinct parts, a causal question and a treatment
question, mirroring the structure of Query 2 to test whether the multi-part
response format observed there, generalizes to an unseen topic.

Second, the treatment component includes a severity qualifier, which tests
whether the retriever can distinguish between general content on the topic
and content specific to the more severe clinical presentation. This directly
targets the granular retrieval precision limitation mentioned in the evaluation section.

Third, the topic falls under a section of the Merck Manual that was not
represented in any of the five evaluation queries, meaning this is a true
out-of-distribution test for the retrieval system.

In [122]:
unseen_question = "What is the cause of acne, and how to treat patients with severe painful cysts?"

In [120]:
# Testing model on a question not used during development
display(Markdown(generate_rag_response(unseen_question, system_prompt=system_prompt3)))

Llama.generate: 28 prefix-match hit, remaining 1888 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    2434.88 ms /  1888 tokens (    1.29 ms per token,   775.40 tokens per second)
llama_perf_context_print:        eval time =   19465.64 ms /   451 runs   (   43.16 ms per token,    23.17 tokens per second)
llama_perf_context_print:       total time =   23139.18 ms /  2339 tokens
llama_perf_context_print:    graphs reused =        448


**Clinical Decision Pathway for Acne and Severe Painful Cysts**

1. **Assess the patient's symptoms and signs**: Determine if the patient has acne vulgaris, a common skin problem affecting most adolescents and many adults.
2. If **acne is present**, assess its severity:
	* Mild: comedones (whiteheads or blackheads)
	* Moderate: papules, pustules
	* Severe: nodules, cysts, abscesses
3. If **severe painful cysts are present**:
	* Assess for contributing factors (e.g., hormonal, mechanical, or drug-related)
	* Consider the patient's psychosocial impact
4. **Determine the cause of acne**: Acne occurs when pilosebaceous units become obstructed with plugs of sebum and desquamated keratinocytes, then colonized and sometimes infected with Propionibacterium acnes (P. acnes).
5. If **cysts are painful**, consider the following treatment options:
	* **Surgical excision**: Remove the cyst wall completely to prevent recurrence.
	* **Incision and drainage**: Drain the contents of the cyst, then remove the cyst wall with a curet or hemostat.
6. For **severe acne** (nodules, cysts, abscesses):
	* Consider oral isotretinoin as a treatment option.

**Standard First-Line Care**

For mild inflammatory acne:

* Topical antibiotics
* Benzoyl peroxide
* Both

For moderate acne:

* Oral antibiotics

**Patient-Specific Factors**

Consider the following factors when treating patients with severe painful cysts:

* Hormonal changes (e.g., pregnancy, menstruation)
* Occlusive cosmetics or clothing
* Humidity and sweating
* Other underlying medical conditions (e.g., polycystic ovary syndrome)

**Source of Information**

The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 73. Acne & Related Disorders.

Note: This clinical decision pathway is based on the provided text and may not be comprehensive or up-to-date. It's essential to consult with a medical professional for accurate diagnosis and treatment.

In [121]:
unseen_answer = generate_rag_response(unseen_question, system_prompt=system_prompt3)
g, r = evaluate_answer(unseen_question, unseen_answer)
display(Markdown(f"**Groundedness:** {g}\n\n**Relevance:** {r}"))

Llama.generate: 1915 prefix-match hit, remaining 1 prompt tokens to eval
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   22960.93 ms /   480 runs   (   47.84 ms per token,    20.91 tokens per second)
llama_perf_context_print:       total time =   24487.90 ms /   481 tokens
llama_perf_context_print:    graphs reused =        477
Llama.generate: 28 prefix-match hit, remaining 2304 prompt tokens to eval
llama_perf_context_print:        load time =     653.94 ms
llama_perf_context_print: prompt eval time =    3136.31 ms /  2304 tokens (    1.36 ms per token,   734.62 tokens per second)
llama_perf_context_print:        eval time =    2421.57 ms /    48 runs   (   50.45 ms per token,    19.82 tokens per second)
llama_perf_context_print: 

**Groundedness:** Score: 4/5
Verdict: Partially Grounded
Reason: The answer includes a clinical decision pathway that is not directly supported by the provided context, which only discusses general treatment options for acne and severe painful cysts.

**Relevance:** Score: 5/5
Verdict: Relevant
Reason: The answer directly addresses the cause of acne and provides a step-by-step clinical decision pathway for treating patients with severe painful cysts, including specific treatment options and patient-specific factors to consider.

The unseen acne query produced a 4/5 groundedness score and 5/5 relevance
score. The relevance result confirms the pipeline correctly identified and
addressed both parts of the query, cause and treatment. The groundedness partial
deduction is the more informative finding.

The evaluator's reason: "the answer includes a clinical decision pathway
that is not directly supported by the provided context, which only discusses
general treatment options." This is the same pattern observed in the
evaluation section for Query 2 (appendicitis) and Query 3 (hair loss),
where system_prompt3's instruction to structure responses as a clinical
decision pathway caused the model to impose a sequential branching format
onto content that the Merck Manual presented as general guidelines rather
than an explicit decision algorithm. The format was generated by the prompt;
the content was retrieved from the manual. The evaluator correctly identified
this mismatch.

The response did successfully retrieve Chapter 73: Acne & Related Disorders, a chapter that was never accessed in any prior section of this notebook,
confirming the retrieval system navigated to the correct section of the manual
for a completely unseen topic.

The severity qualifier produced a meaningful result: the response
distinguished between mild, moderate, and
severe presentations, and provided
differentiated treatment recommendations for each tier.
This suggests the retriever successfully surfaced content that addressed the
specific cystic severity dimension of the query rather than returning only
general acne content.

In [126]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/Full_Code_NLP_RAG_Project.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/Full_Code_NLP_RAG_Project.ipynb to html
Traceback (most recent call last):
  File "/usr/local/bin/jupyter-nbconvert", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/jupyter_core/application.py", line 284, in launch_instance
    super().launch_instance(argv=argv, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/nbconvertapp.py", line 420, in start
    self.convert_notebooks()
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/nbconvertapp.py", line 597, in convert_notebooks
    self.convert_single_notebook(notebook_filename)
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/nbconvertapp.py", line 563, in convert_single_notebook
    output, resources = self.export_single_notebook(
              